# זיהוי פעילות כלי נגינה בתוך מיקס פוליפוני — Legacy PANNs CNN6

הגוף הקונבולוציוני משחזר את Cnn6LinearSpec הישן: BatchNorm נפרד לכל bin תדר,
ארבעה בלוקי 5×5 עם BatchNorm ו־average pooling, ואיגום זמן מקומי max+mean.
אין LSTM; הראש הפוליפוני מפיק ON/OFF לכל כלי בכל 0.5 שנייה.
הקטע שנבחר הוא ולידציה טבעית דטרמיניסטית.


## ICASSP 2027: הרצה לפי כלי ורוחב פס

עותק עבודה נקי של המחברת שבדרייב. בחרו בתא ההגדרות `SELECTED_INSTRUMENTS` לכלי אחד, `VAL_SONG_ID` לקטע שבו הכלי אכן מופיע, ו־`AUDIO_MODE` מתוך `44k`, `96k_full`, `96k_lp22`. עדכנו יחד את `CONTROLLED_VALIDATION_BACKGROUND_INSTRUMENTS`, `ISOLATED_PRETRAIN_NEGATIVE_INSTRUMENTS` ו־`CONFUSER_HARD_NEGATIVE_MAPPING` לכלי הנבחר. אפשר לכבות כל סוג hard negative באמצעות הדגל שלו.

קטע 2.2 מוחרג. משפחת הקטע שנבחרה לוולידציה אינה באימון. ההרצה היא 50 אפוקים ללא early stopping, והתוצאה המדווחת משתמשת במשקלים הסופיים בלבד; אין שמירת צ׳קפוינט מיטבי לפי אותו קטע. יש להריץ בנפרד עבור כל כלי וכל אחד משלושת רוחבי הפס, בשלושה זרעים 42, 43, 44, ולבדוק שיש לכלי דוגמאות ON ו־OFF בוולידציה. פלטי ההרצות ייכתבו לתיקיות חדשות ב־Drive, ללא שינוי במקורות.

שים לב: `selected-only` ו־`controlled` הם ניתוחי ולידציה נוספים; עבור התוצאה העיקרית השתמשו במיקס הטבעי בלבד.


In [ ]:
# Colab already includes most packages; this keeps the environment reproducible.
%pip -q install soundfile scipy scikit-learn pandas matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. בחירת הניסוי — זה התא היחיד שעורכים בין הרצות

בחר כאן את כלי הנגינה, תדר הדגימה, רצועת הוולידציה ושני סוגי ה־hard negative.

- `TARGET_OFF`: מלמד שהכלי הנבחן יכול להיות כבוי בתוך מיקס פעיל.
- `CONFUSER`: מכריח כלי אחר ודומה להופיע כשהכלי הנבחן כבוי.

אם מנגנון אינו דרוש, משאירים אותו `False` ואת הרשימה/המיפוי שלו ריקים.


In [ ]:
# ===================== בחירות הניסוי =====================
# גוף הרשת הישן Cnn6LinearSpec, מותאם לפלט פוליפוני בכל 0.5 שנייה.
# זהו מזהה ניסוי חדש; צ'קפוינטים של גרסת GroupNorm אינם תואמים.
MODEL_ARCHITECTURE = "legacy_cnn6_linear_spec_framewise_v1"

RUN_SEED = 42  # הריצו גם 43 ו-44 בכל תנאי רוחב פס.
AUDIO_MODE = "44k"  # אפשרויות: "44k", "96k_full", "96k_lp22", "16k"

# שמות התיקיות המדויקים בדרייב. Duble_bass מאוית כך בדאטה הקיים.
SELECTED_INSTRUMENTS = (
    "Clarinet",

)

# משפחת קטע הוולידציה שנבחר מוחזקת כולה מחוץ לאימון.
VAL_SONG_ID = 5
# ולידציה קלה נוספת: מיקס דטרמיניסטי של כלי הפלט בלבד, באותו ציר זמן
# ובאותן תוויות של קטע הוולידציה. הדיווח יתבסס על סוף האימון בלבד.
USE_SELECTED_ONLY_VALIDATION = True
# ולידציה מבוקרת: תווית הפלט נשארת של הכלי שנבחר בלבד, אך האודיו כולל
# את שתי הגיטרות המבלבלות הבאות מאותו קטע ובאותו ציר זמן.
CONTROLLED_VALIDATION_BACKGROUND_INSTRUMENTS = (
    "Saxophone",
    "Flute",
)

# False = ניסוי חדש מאפוק 1. אחרי ניתוק בלבד משנים ל-True ומריצים שוב מלמעלה.
AUTO_RESUME = False

RUN_ALIGNMENT_DIAGNOSTIC = False
RUN_DRY_RUN_BEFORE_TRAINING = False
RUN_FULL_TRAINING = True

DIAGNOSTIC_SONG_ID = VAL_SONG_ID
DIAGNOSTIC_TARGET_INSTRUMENT = "Clarinet"
DIAGNOSTIC_CONFUSER_INSTRUMENTS = ("Saxophone", "Flute")
DIAGNOSTIC_MAX_LAG_SECONDS = 10.0

# ---------- מקורות האימון ----------
REAL_MIX_RATIO = 0.50
ISOLATED_STEM_RATIO = 0.15
# רצפה חיובית לכל כלי. אם בריכת ה-15% נותנת פחות, האפוק גדל בלי
# לגרוע דוגמאות מהמיקס הטבעי או מהמיקס הסינתטי. None מבטל את הרצפה.
ISOLATED_POSITIVE_SAMPLES_PER_INSTRUMENT = 160
# ב-stem מבודד ה-loss מחושב עבור הכלי הנשמע בלבד; OFF לכלים האחרים
# נלמד מהמיקסים הטבעיים והסינתטיים ולא מזהות מקור מלאכותית.
ISOLATED_STEM_TARGET_ONLY_LOSS = True

# קדם-אימון של CNN6 בלבד על זהות הכלי ב-stems מבודדים. כל דוגמה היא
# בדיוק חלון אחד של 0.5 שנייה, ורק חלון שה-RMS הגולמי שלו גבוה מהסף.
# אין השלמת פערים ואין חלונות 8 שניות בקדם-האימון. משפחת ולידציה 5 מוחרגת.
USE_ISOLATED_STEM_PRETRAINING = True
ACTIVITY_THRESHOLD_DBFS = -70.0
ISOLATED_PRETRAIN_CLIP_SECONDS = 0.5
ISOLATED_PRETRAIN_EPOCHS = 10
ISOLATED_PRETRAIN_SAMPLES_PER_INSTRUMENT = 256
ISOLATED_PRETRAIN_BATCH_SIZE = 16
ISOLATED_PRETRAIN_LEARNING_RATE = 2e-4
ISOLATED_PRETRAIN_RANDOM_GAIN_DB_MIN = -6.0
ISOLATED_PRETRAIN_RANDOM_GAIN_DB_MAX = 6.0
ISOLATED_PRETRAIN_REUSE_CHECKPOINT = True
ISOLATED_PRETRAIN_FORCE_REBUILD = False
# בכלי יחיד הקדם-אימון הופך לבינארי: הכלי הנבחר מול קטעי רקע פעילים
# משלושת כלי הרקע. כך אין Softmax בעל מחלקה אחת ואין loss=0 טריוויאלי.
ISOLATED_PRETRAIN_NEGATIVE_INSTRUMENTS = CONTROLLED_VALIDATION_BACKGROUND_INSTRUMENTS
PRETRAINED_ENCODER_FREEZE_EPOCHS = 2
SYNTHETIC_MIX_RATIO = 0.35
SYNTHETIC_BACKGROUND_PROBABILITY = 0.50
SYNTHETIC_MIN_TOTAL_INSTRUMENTS = 4

# כל target stem במיקס סינתטי מנורמל לפי ה-active RMS שלו, לפני החיבור.
# לאחר הנרמול נוסף gain אקראי קטן, כך שהמודל אינו לומד עוצמה קבועה אך אף
# כלי אינו נעלם רק מפני שהוקלט חלש יותר מהתופים.
BALANCE_SYNTHETIC_TARGET_ACTIVE_RMS = True
SYNTHETIC_TARGET_REFERENCE_DBFS = -26.0
SYNTHETIC_TARGET_RANDOM_GAIN_DB_MIN = -4.0
SYNTHETIC_TARGET_RANDOM_GAIN_DB_MAX = 4.0
SYNTHETIC_TARGET_MAX_ADJUSTMENT_DB = 24.0

# השמטה נגדית פעילה רק לכלים הפעילים מעל 70% ברצועת אימון נתונה.
USE_COUNTERFACTUAL_STEM_DROPOUT = False

# חלוקת האימון: 50% מיקס טבעי, 15% stem מבודד ו-35% סינתטי.
# target-OFF נשאר כבוי בניסוי זה; היחסים הבאים שמורים רק אם יופעל בעתיד.
USE_TARGET_OFF_HARD_NEGATIVES = False
# תמיד עוקב אחרי כלי הפלט שנבחרו כדי שלא תישאר הגדרה ישנה מהרצה קודמת.
TARGET_OFF_INSTRUMENTS = SELECTED_INSTRUMENTS
TARGET_OFF_BACKGROUND_PROBABILITY = 1.0
TARGET_OFF_MIN_SECONDS = 4.0
TARGET_OFF_FULLY_OFF_RATIO = 0.10
TARGET_OFF_PARTIALLY_OFF_RATIO = 0.10
TARGET_OFF_REGULAR_RATIO = 0.15
USE_CONFUSER_HARD_NEGATIVES = True
CONFUSER_HARD_NEGATIVE_MAPPING = {
    "Clarinet": ("Saxophone", "Flute"),
}

# ---------- יציבות ומהירות ----------
# נשארים עם batch 16 למהירות. הסמפלר מאזן מקור ושיר בלבד;
# ON/OFF מאוזנים בתוך ה-loss כדי לא לקשור OFF לזהות של שיר מסוים.
TRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
TRAIN_SAMPLES_PER_EPOCH = 2200
DATA_LOADER_WORKERS = 6
DATA_LOADER_PREFETCH = 4
# Batch 16 נשאר. קצבי הלמידה נשמרים עד אפוק 30 ורק אז דועכים בהדרגה.
ENCODER_LEARNING_RATE = 4e-5
NEW_LAYERS_LEARNING_RATE = 2e-4
WARMUP_EPOCHS = 3
LR_HOLD_EPOCHS = 30
USE_EMA = True
EMA_DECAY = 0.995
USE_CLASS_BALANCE_WEIGHTS = True
MAX_CLASS_BALANCE_WEIGHT = 2.0

print("Experiment choices:")

print(f"  architecture:     {MODEL_ARCHITECTURE}")

print(f"  audio mode:       {AUDIO_MODE}")
print(f"  instruments:      {SELECTED_INSTRUMENTS}")
print(f"  validation song:  {VAL_SONG_ID}")
print(
    f"  controlled val:  enabled={USE_SELECTED_ONLY_VALIDATION}, "
    f"backgrounds={CONTROLLED_VALIDATION_BACKGROUND_INSTRUMENTS}"
)
print(f"  auto resume:      {AUTO_RESUME}")
print(f"  run diagnostic:   {RUN_ALIGNMENT_DIAGNOSTIC}")
print(f"  run dry-run:      {RUN_DRY_RUN_BEFORE_TRAINING}")
print(f"  run training:     {RUN_FULL_TRAINING}")
print(
    f"  source ratios:    real={REAL_MIX_RATIO:.2f}, "
    f"isolated={ISOLATED_STEM_RATIO:.2f}, synthetic={SYNTHETIC_MIX_RATIO:.2f}"
)
print(
    f"  isolated quota:  {ISOLATED_POSITIVE_SAMPLES_PER_INSTRUMENT} positive sequences/instrument, "
    f"target-only loss={ISOLATED_STEM_TARGET_ONLY_LOSS}"
)
print(
    f"  CNN6 pretraining: enabled={USE_ISOLATED_STEM_PRETRAINING}, "
    f"epochs={ISOLATED_PRETRAIN_EPOCHS}, "
    f"samples/instrument={ISOLATED_PRETRAIN_SAMPLES_PER_INSTRUMENT}, "
    f"clip={ISOLATED_PRETRAIN_CLIP_SECONDS:.1f}s, "
    f"strict RMS>{ACTIVITY_THRESHOLD_DBFS:.1f} dBFS, "
    f"freeze={PRETRAINED_ENCODER_FREEZE_EPOCHS} epochs, "
    f"single-class negatives={ISOLATED_PRETRAIN_NEGATIVE_INSTRUMENTS}"
)
print(
    f"  synthetic RMS:   balanced={BALANCE_SYNTHETIC_TARGET_ACTIVE_RMS}, "
    f"reference={SYNTHETIC_TARGET_REFERENCE_DBFS:.1f} dBFS, "
    f"jitter=[{SYNTHETIC_TARGET_RANDOM_GAIN_DB_MIN:.1f}, "
    f"{SYNTHETIC_TARGET_RANDOM_GAIN_DB_MAX:.1f}] dB"
)
print(
    f"  runtime:          batch={TRAIN_BATCH_SIZE}, accumulation={GRADIENT_ACCUMULATION_STEPS}, "
    f"workers={DATA_LOADER_WORKERS}, samples/epoch={TRAIN_SAMPLES_PER_EPOCH}"
)
print(
    f"  target-OFF:       instruments={TARGET_OFF_INSTRUMENTS}, "
    f"regular/full/partial={TARGET_OFF_REGULAR_RATIO:.2f}/"
    f"{TARGET_OFF_FULLY_OFF_RATIO:.2f}/{TARGET_OFF_PARTIALLY_OFF_RATIO:.2f}"
)
print(
    f"  optimization:     lr={ENCODER_LEARNING_RATE:.1e}/{NEW_LAYERS_LEARNING_RATE:.1e}, "
    f"warmup={WARMUP_EPOCHS}, hold-through={LR_HOLD_EPOCHS}, "
    f"EMA={USE_EMA} ({EMA_DECAY})"
)
print(
    f"  loss:             source/song sampler, class-balanced weights capped at "
    f"{MAX_CLASS_BALANCE_WEIGHT:.1f}"
)


## 2. הגדרות ופונקציות פנימיות

התא הבא מכיל את הפייפליין. **אין לערוך בו את בחירות הניסוי** — הן מגיעות
מהתא היחיד שבתחילת המחברת.


In [ ]:
"""
Instrument activity recognition inside polyphonic music mixes.

Designed for Google Colab + Google Drive.

Main experiment:
- Linear-frequency log-power spectrogram (no Mel filter bank)
- Historical Cnn6LinearSpec body: per-frequency BatchNorm + four 5x5 CNN blocks

- Local max+mean temporal pooling preserves the old clipwise aggregation in every 0.5 s bin

- CNN-only isolated-stem pretraining on strict active 0.5-second clips

- Direct multi-label CNN output every 0.5 s during polyphonic fine-tuning
- Automatic labels from isolated stems using a 0.5 s trailing RMS window
- Training on isolated stems + real mixes + same-song counterfactual synthetic mixes
- Half of synthetic examples are clean; half use a loudness-controlled non-target background bus
- Optional RMS-guided hard negatives force confusable instruments into eligible OFF windows
- Natural validation plus matched clean and fully polyphonic controlled stem-dropout validation
- Audio modes: 44.1 kHz, 96 kHz full-band, 96 kHz low-pass at 22.05 kHz,
  and 16 kHz downsampled from the 96 kHz master data

The only section that normally needs editing is CONFIGURATION.
"""

from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, MutableMapping, Optional, Sequence, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import signal
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm import tqdm as _text_tqdm


def tqdm(*args, **kwargs):
    """Text-only progress bars: avoid hundreds of persisted Colab widgets."""
    kwargs.setdefault("mininterval", 30.0)
    kwargs.setdefault("maxinterval", 60.0)
    kwargs.setdefault("dynamic_ncols", False)
    return _text_tqdm(*args, **kwargs)




In [ ]:
from __future__ import annotations

# =============================================================================
# CONFIGURATION -- EDIT THIS SECTION IN COLAB
# =============================================================================


@dataclass
class ExperimentConfig:
    # Google Drive folders. Each source folder contains root-level mix files
    # (1.wav, 2.wav, ...) and one subfolder per instrument.
    drive_44_root: str = "/content/drive/MyDrive/train_44"
    drive_96_root: str = "/content/drive/MyDrive/train_96"
    drive_output_root: str = "/content/drive/MyDrive/instrument_mix_runs"

    # Local Colab disk cache. Audio is read from here during training.
    local_cache_root: str = "/content/instrument_mix_audio_cache"
    use_local_audio_cache: bool = True
    force_rebuild_audio_cache: bool = False

    # Set explicitly in the single run-configuration cell below.
    # The 16 kHz condition is always downsampled from the 96 kHz masters.
    audio_mode: str = ""

    # Exact or case-insensitive folder names. Only these classes receive model
    # outputs. Other instruments remain naturally present in the real mix.
    # Neutral internal default. The user-facing cell supplies the instruments.
    selected_instruments: Tuple[str, ...] = ()

    # Validation can be any complete song. All other detected songs train.
    val_song_id: Union[int, float] = 4
    # A second, easier diagnostic mix made only from the selected validation stems.
    # It never replaces natural validation or controls checkpoint selection.
    use_selected_only_validation: bool = False
    # Optional deterministic non-target stems added to the controlled validation audio.
    # They remain background only and never become output labels.
    controlled_validation_background_instruments: Tuple[str, ...] = ()
    # Decimal IDs such as 2.2 are supported. IDs from the same base family
    # (2, 2.2, 2.3) always stay in the same train/validation split.
    song_ids: Optional[Tuple[Union[int, float], ...]] = None  # None = auto-detect all mixes.

    # Training sources.
    use_real_mix: bool = True
    use_stems: bool = True
    use_synthetic_mix: bool = False
    # Source ratios. The code uses the first pair when synthetic mixing is off
    # and the second triplet when it is on.
    real_mix_ratio_no_synthetic: float = 0.65
    stem_ratio_no_synthetic: float = 0.35
    real_mix_ratio_with_synthetic: float = 0.20
    stem_ratio_with_synthetic: float = 0.30
    synthetic_mix_ratio: float = 0.50
    # When target-OFF hard negatives are enabled, the 50% synthetic share is
    # split into 30% regular, 10% fully target-OFF, and 10% partially target-OFF.
    target_off_regular_synthetic_ratio: float = 0.30
    target_off_fully_off_ratio: float = 0.10
    target_off_partially_off_ratio: float = 0.10

    # Synthetic mixes use only same-song, same-time stems. With a dropout
    # plan, every available selected stem is included unless its planned
    # interval is muted. Half of the synthetic examples stay clean; the other
    # half receive 4-6 non-target stems combined into one loudness-controlled
    # background bus. Background stems never receive target labels.
    synthetic_inclusion_probability: float = 1.0
    use_synthetic_background_instruments: bool = True
    synthetic_background_probability: float = 0.50
    synthetic_background_min_instruments: int = 4
    synthetic_background_max_instruments: Optional[int] = 6
    # Minimum total polyphony, counting both selected targets and background stems.
    synthetic_min_total_instruments: int = 4
    # Independently normalize every included target over its active half-second
    # steps. A small deterministic-per-epoch gain jitter preserves loudness
    # diversity without letting recording level identify or hide an instrument.
    balance_synthetic_target_active_rms: bool = False
    synthetic_target_reference_dbfs: float = -26.0
    synthetic_target_random_gain_db_min: float = -4.0
    synthetic_target_random_gain_db_max: float = 4.0
    synthetic_target_max_adjustment_db: float = 24.0

    # Optional hard-negative contexts. These instruments remain background
    # only: they never receive model-output labels. In an eligible sequence,
    # the target is OFF according to the original 96 kHz RMS labels while a
    # configured confuser is ON. The background bus is still used with the
    # normal probability, preserving the clean/contextual 50/50 split.
    use_hard_negative_backgrounds: bool = False
    # Neutral internal default. Optional confusers come from the run cell.
    hard_negative_confusers: Dict[str, Tuple[str, ...]] = field(default_factory=dict)
    hard_negative_probability: float = 0.80
    hard_negative_min_overlap_seconds: float = 0.50

    # Contextual target-OFF hard negatives. Dedicated synthetic pools contain
    # either a fully OFF target or at least the requested contiguous OFF
    # duration. Their 4-6-instrument background is forced independently of the
    # regular synthetic background probability. Per-frame labels stay unchanged.
    use_target_off_hard_negatives: bool = False
    target_off_hard_negative_instruments: Tuple[str, ...] = ()
    target_off_hard_negative_probability: float = 0.80
    target_off_hard_negative_min_seconds: float = 1.0

    # Clean and fully polyphonic controlled validation share the exact
    # same deterministic target-dropout plan. Only their background context differs.
    use_controlled_polyphonic_validation: bool = False
    synthetic_background_relative_db_min: float = -6.0
    synthetic_background_relative_db_max: float = 0.0
    synthetic_peak_limit: float = 0.99

    # Counterfactual stem dropout. For instruments that are naturally active
    # nearly all the time, create several long OFF intervals until their
    # activity in the synthetic version is at most the requested fraction.
    # The held-out real mix is never overwritten: controlled validation is a
    # second, deterministic diagnostic set.
    use_counterfactual_stem_dropout: bool = True
    counterfactual_min_original_activity_fraction: float = 0.70
    counterfactual_target_activity_fraction: float = 0.65
    counterfactual_min_interval_seconds: float = 5.0
    counterfactual_max_interval_seconds: float = 60.0
    counterfactual_min_intervals: int = 3
    counterfactual_fade_seconds: float = 0.10
    counterfactual_max_attempts: int = 500
    use_controlled_dropout_validation: bool = False
    controlled_validation_seed_offset: int = 10000
    controlled_validation_selection_weight: float = 0.75
    worst_class_selection_weight: float = 0.10
    final_checkpoint_metric: str = "natural"

    # Temporal design.
    sequence_seconds: float = 8.0
    sequence_hop_seconds: float = 4.0
    val_sequence_hop_seconds: float = 4.0
    label_hop_seconds: float = 0.5

    # Automatic activity labels. Every half-second label uses the RMS energy in
    # the preceding 0.5 s (truncated only at the beginning of the song).
    label_window_seconds: float = 0.5
    activity_threshold_dbfs: float = -70.0
    max_internal_gap_seconds: float = 0.5
    minimum_event_seconds: float = 0.5
    force_rebuild_labels: bool = False

    # Linear STFT. Physical durations stay constant across all sample rates.
    stft_window_ms: float = 32.0
    stft_hop_ms: float = 8.0
    stft_center: bool = True
    log_epsilon: float = 1e-10

    # 96 kHz low-pass control condition.
    lowpass_cutoff_hz: float = 22050.0
    lowpass_numtaps: int = 2049
    lowpass_kaiser_beta: float = 8.6

    # Historical PANNs Cnn6LinearSpec body, with no recurrent layer.

    # Input normalization is fixed to BatchNorm2d(freq_bins), exactly as in

    # the isolated-instrument network; each frequency bin has its own state.

    model_architecture: str = "legacy_cnn6_linear_spec_framewise_v1"

    embedding_size: int = 512

    cnn_dropout: float = 0.20

    embedding_dropout: float = 0.50

    # Mild SpecAugment is optional and disabled by default for the bandwidth
    # comparison. Widths are in linear STFT frames/bins.
    use_spec_augment: bool = False
    time_mask_width: int = 16
    time_mask_count: int = 2
    freq_mask_width: int = 8
    freq_mask_count: int = 2

    # Optional automatic pretraining of this exact CNN6 encoder on isolated
    # stem identity. Every example is one strict active-RMS half-second frame.
    # Only training-song families are used; validation never leaks.
    use_isolated_stem_pretraining: bool = False
    isolated_pretrain_clip_seconds: float = 0.5
    isolated_pretrain_epochs: int = 10
    isolated_pretrain_samples_per_instrument: int = 256
    isolated_pretrain_batch_size: int = 16
    isolated_pretrain_learning_rate: float = 2e-4
    isolated_pretrain_random_gain_db_min: float = -6.0
    isolated_pretrain_random_gain_db_max: float = 6.0
    isolated_pretrain_reuse_checkpoint: bool = True
    isolated_pretrain_force_rebuild: bool = False
    # With one selected output, active clips from these non-target stems provide
    # the negative class for a genuine binary presence/absence pretraining task.
    isolated_pretrain_negative_instruments: Tuple[str, ...] = ()

    # A manual checkpoint remains available as an alternative to automatic
    # pretraining. Only compatible CNN6/fc1 tensors are transferred.
    pretrained_stem_checkpoint: Optional[str] = None
    transfer_fc1: bool = True
    freeze_pretrained_encoder_epochs: int = 2

    # Optimization. A100 runs use true mini-batches; accumulation is not a speed tool.
    epochs: int = 50
    batch_size: int = 8
    gradient_accumulation_steps: int = 1
    # Baseline source quotas are computed from this value. A fixed isolated
    # positive floor may enlarge the epoch while preserving every real/synthetic quota.
    train_samples_per_epoch: Optional[int] = 2200
    isolated_positive_samples_per_instrument: Optional[int] = None
    # On an isolated stem, supervise only the instrument that is actually heard.
    # Natural and synthetic mixes remain fully multi-label supervised.
    isolated_stem_target_only_loss: bool = False
    # Exact source and per-class focus quotas. Every epoch is reproducible.
    deterministic_epoch_sampling: bool = True
    class_aware_sampling: bool = True
    class_aware_off_fraction: float = 0.50
    encoder_learning_rate: float = 2e-5
    new_layers_learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    max_grad_norm: float = 5.0
    use_early_stopping: bool = False
    early_stopping_patience: int = 10
    # Warm up, hold the full LR long enough for batch-16 optimization,
    # then apply validation-independent cosine decay.
    use_lr_scheduler: bool = True
    warmup_epochs: int = 3
    lr_hold_epochs: int = 30
    min_encoder_learning_rate: float = 1e-6
    min_new_layers_learning_rate: float = 5e-6
    # EMA is evaluated and checkpointed; raw weights are retained for exact resume.
    use_ema: bool = True
    ema_decay: float = 0.995
    use_amp: bool = True
    num_workers: int = 4
    pin_memory: bool = True
    prefetch_factor: int = 4
    # Keep False with epoch-dependent deterministic synthesis: workers restart
    # after dataset.set_epoch(), so every worker receives the correct epoch.
    persistent_workers: bool = False

    # Multi-label objective and decisions. ON and OFF receive separate
    # per-instrument weights computed from the actual sampled source mixture,
    # including counterfactual dropout labels.
    use_class_balance_weights: bool = True
    max_class_balance_weight: float = 10.0
    probability_threshold: float = 0.50

    # Reproducibility and diagnostics.
    seed: int = 0
    dry_run: bool = False
    save_last_checkpoint: bool = True
    save_best_checkpoint: bool = True
    resume_checkpoint: Optional[str] = None




In [ ]:
from __future__ import annotations

# =============================================================================
# GENERAL UTILITIES
# =============================================================================


@dataclass(frozen=True)
class AudioModeSpec:
    name: str
    source_kind: str  # "44" or "96"
    target_sr: int
    processing: str  # "copy", "lowpass", or "downsample16"


AUDIO_MODES: Dict[str, AudioModeSpec] = {
    "44k": AudioModeSpec("44k", "44", 44100, "copy"),
    "96k_full": AudioModeSpec("96k_full", "96", 96000, "copy"),
    "96k_lp22": AudioModeSpec("96k_lp22", "96", 96000, "lowpass"),
    "16k": AudioModeSpec("16k", "96", 16000, "downsample16"),
}


def mount_google_drive(mount_point: str = "/content/drive") -> None:
    """Mount Google Drive when running inside Colab."""
    try:
        from google.colab import drive  # type: ignore
    except ImportError as exc:
        raise RuntimeError(
            "google.colab is unavailable. Run this notebook inside Google Colab "
            "or mount/provide the data paths manually."
        ) from exc
    drive.mount(mount_point)


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Reproducibility is more important than autotuned-kernel speed in this
    # experiment: bandwidth comparisons must not inherit cuDNN kernel noise.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def normalize_name(value: str) -> str:
    value = unicodedata.normalize("NFKC", value).casefold()
    return "".join(ch for ch in value if ch.isalnum())


def safe_slug(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    ascii_value = value.encode("ascii", "ignore").decode("ascii")
    slug = re.sub(r"[^A-Za-z0-9]+", "_", ascii_value).strip("_").lower()
    return slug or hashlib.sha1(value.encode("utf-8")).hexdigest()[:10]


SongId = Union[int, float]


def canonical_song_id(value: Any) -> SongId:
    number = float(value)
    if not math.isfinite(number) or number < 0:
        raise ValueError(f"Invalid song ID: {value!r}")
    return int(number) if number.is_integer() else number


def extract_song_id(filename: str) -> Optional[SongId]:
    # The final numeric token may be a decimal family member. Drive files
    # can be named either mix_2.2 / piano_2.2 or with an explicit .wav.
    # Strip only a known audio extension: Path.stem would incorrectly
    # interpret the final .2 in an extensionless name as a file suffix.
    name = Path(filename).name.strip()
    known_audio_extensions = {".wav", ".wave", ".flac", ".aif", ".aiff"}
    suffix = Path(name).suffix.casefold()
    if suffix in known_audio_extensions:
        name = name[: -len(suffix)]

    # In a right-to-left Drive interface, an intended name such as
    # duble_bass_2.2 can be stored as 2.duble_bass_2. Interpret that mirrored
    # form as <base>.<derivative>: 2.duble_bass_2 -> song 2.2.
    mirrored_decimal = re.fullmatch(r"(\d+)\.(.+?)[_-](\d+)", name)
    if mirrored_decimal and any(character.isalpha() for character in mirrored_decimal.group(2)):
        derivative = mirrored_decimal.group(1)
        base = mirrored_decimal.group(3)
        return canonical_song_id(f"{base}.{derivative}")

    matches = re.findall(r"(\d+(?:\.\d+)?)", name)
    return canonical_song_id(matches[-1]) if matches else None


def song_family_id(song_id: SongId) -> int:
    return int(math.floor(float(canonical_song_id(song_id))))


def song_id_sort_key(song_id: SongId) -> Tuple[float, str]:
    canonical = canonical_song_id(song_id)
    return float(canonical), str(canonical)


def song_id_seed(song_id: SongId) -> int:
    canonical = str(canonical_song_id(song_id))
    return int(hashlib.sha1(canonical.encode("utf-8")).hexdigest()[:8], 16)


def is_readable_audio(path: Path) -> bool:
    try:
        sf.info(str(path))
        return True
    except Exception:
        return False


def audio_duration(path: Path) -> float:
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate)


def json_ready(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [json_ready(v) for v in value]
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


def save_json(path: Path, payload: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(dict(payload)), handle, ensure_ascii=False, indent=2)


def active_hard_negative_mapping(
    cfg: ExperimentConfig,
) -> Dict[str, Tuple[str, ...]]:
    """Return only mappings whose target is one of the model outputs."""
    selected_by_normalized = {
        normalize_name(instrument): instrument
        for instrument in cfg.selected_instruments
    }
    active: Dict[str, Tuple[str, ...]] = {}
    for configured_target, configured_confusers in cfg.hard_negative_confusers.items():
        target = selected_by_normalized.get(normalize_name(configured_target))
        if target is None:
            continue
        unique_confusers: List[str] = []
        seen: set[str] = set()
        for confuser in configured_confusers:
            normalized = normalize_name(confuser)
            if normalized and normalized not in seen:
                unique_confusers.append(confuser)
                seen.add(normalized)
        if unique_confusers:
            active[target] = tuple(unique_confusers)
    return active


def configured_hard_negative_instruments(cfg: ExperimentConfig) -> Tuple[str, ...]:
    """Unique confuser folders required for the active model classes."""
    instruments: List[str] = []
    seen: set[str] = set()
    for confusers in active_hard_negative_mapping(cfg).values():
        for instrument in confusers:
            normalized = normalize_name(instrument)
            if normalized not in seen:
                instruments.append(instrument)
                seen.add(normalized)
    return tuple(instruments)


def validate_config(cfg: ExperimentConfig) -> None:
    if cfg.audio_mode not in AUDIO_MODES:
        raise ValueError(f"Unknown audio_mode={cfg.audio_mode!r}. Choose {list(AUDIO_MODES)}")
    if not cfg.selected_instruments:
        raise ValueError("selected_instruments must contain at least one instrument.")
    if len({normalize_name(x) for x in cfg.selected_instruments}) != len(cfg.selected_instruments):
        raise ValueError("selected_instruments contains duplicate normalized names.")
    if cfg.sequence_seconds <= 0 or cfg.label_hop_seconds <= 0:
        raise ValueError("sequence_seconds and label_hop_seconds must be positive.")
    steps = cfg.sequence_seconds / cfg.label_hop_seconds
    if not math.isclose(steps, round(steps), rel_tol=0, abs_tol=1e-8):
        raise ValueError("sequence_seconds must be an integer multiple of label_hop_seconds.")
    if cfg.sequence_hop_seconds % cfg.label_hop_seconds > 1e-8:
        raise ValueError("sequence_hop_seconds must align to label_hop_seconds.")
    if cfg.val_sequence_hop_seconds % cfg.label_hop_seconds > 1e-8:
        raise ValueError("val_sequence_hop_seconds must align to label_hop_seconds.")
    if cfg.label_window_seconds < cfg.label_hop_seconds:
        raise ValueError("label_window_seconds should be >= label_hop_seconds.")
    if cfg.lowpass_numtaps % 2 == 0:
        raise ValueError("lowpass_numtaps must be odd for a centered linear-phase FIR.")
    if not (0.0 < cfg.probability_threshold < 1.0):
        raise ValueError("probability_threshold must be between 0 and 1.")
    if cfg.max_class_balance_weight < 1.0:
        raise ValueError("max_class_balance_weight must be >= 1.")
    if cfg.batch_size < 1 or cfg.gradient_accumulation_steps < 1:
        raise ValueError("batch_size and gradient_accumulation_steps must be >= 1.")
    isolated_floor = cfg.isolated_positive_samples_per_instrument
    if isolated_floor is not None:
        if isinstance(isolated_floor, bool) or int(isolated_floor) != isolated_floor or isolated_floor < 1:
            raise ValueError("isolated_positive_samples_per_instrument must be a positive integer or None.")
        if not cfg.use_stems:
            raise ValueError("An isolated positive quota requires use_stems=True.")
    if not (cfg.use_real_mix or cfg.use_stems or cfg.use_synthetic_mix):
        raise ValueError("At least one training source must be enabled.")
    if not (0.0 < cfg.synthetic_inclusion_probability <= 1.0):
        raise ValueError("synthetic_inclusion_probability must be in (0, 1].")
    if not (0.0 <= cfg.synthetic_background_probability <= 1.0):
        raise ValueError("synthetic_background_probability must be in [0, 1].")
    if cfg.synthetic_background_min_instruments < 0:
        raise ValueError("synthetic_background_min_instruments must be >= 0.")
    if (
        cfg.synthetic_background_max_instruments is not None
        and cfg.synthetic_background_max_instruments
        < cfg.synthetic_background_min_instruments
    ):
        raise ValueError(
            "synthetic_background_max_instruments must be >= the minimum."
        )
    if cfg.synthetic_background_relative_db_max < cfg.synthetic_background_relative_db_min:
        raise ValueError("synthetic_background_relative_db_max must be >= the minimum.")
    if cfg.synthetic_target_random_gain_db_max < cfg.synthetic_target_random_gain_db_min:
        raise ValueError("synthetic_target_random_gain_db_max must be >= the minimum.")
    if cfg.synthetic_target_max_adjustment_db <= 0:
        raise ValueError("synthetic_target_max_adjustment_db must be positive.")
    if cfg.synthetic_target_reference_dbfs >= 0:
        raise ValueError("synthetic_target_reference_dbfs must be below 0 dBFS.")
    if cfg.use_isolated_stem_pretraining and cfg.pretrained_stem_checkpoint:
        raise ValueError(
            "Choose automatic isolated-stem pretraining or a manual checkpoint, not both."
        )
    if cfg.isolated_pretrain_epochs < 1:
        raise ValueError("isolated_pretrain_epochs must be positive.")
    if cfg.model_architecture != "legacy_cnn6_linear_spec_framewise_v1":

        raise ValueError(

            "This notebook implements only "

            "model_architecture='legacy_cnn6_linear_spec_framewise_v1'."

        )
    if cfg.isolated_pretrain_samples_per_instrument < 1:
        raise ValueError("isolated_pretrain_samples_per_instrument must be positive.")
    if not math.isclose(
        cfg.isolated_pretrain_clip_seconds, 0.5, rel_tol=0.0, abs_tol=1e-8
    ):
        raise ValueError("Isolated CNN6 pretraining clips must be exactly 0.5 seconds.")
    if not math.isclose(
        cfg.isolated_pretrain_clip_seconds,
        cfg.label_hop_seconds,
        rel_tol=0.0,
        abs_tol=1e-8,
    ) or not math.isclose(
        cfg.isolated_pretrain_clip_seconds,
        cfg.label_window_seconds,
        rel_tol=0.0,
        abs_tol=1e-8,
    ):
        raise ValueError(
            "The isolated pretraining clip, label hop, and RMS label window must match."
        )
    if cfg.isolated_pretrain_batch_size < 1 or cfg.isolated_pretrain_learning_rate <= 0:
        raise ValueError("Isolated pretraining batch size / learning rate is invalid.")
    if cfg.isolated_pretrain_random_gain_db_max < cfg.isolated_pretrain_random_gain_db_min:
        raise ValueError("isolated_pretrain_random_gain_db_max must be >= the minimum.")
    if not (0.0 <= cfg.hard_negative_probability <= 1.0):
        raise ValueError("hard_negative_probability must be in [0, 1].")
    if cfg.hard_negative_min_overlap_seconds <= 0:
        raise ValueError("hard_negative_min_overlap_seconds must be positive.")
    if cfg.use_hard_negative_backgrounds:
        if not (cfg.use_synthetic_mix and cfg.use_synthetic_background_instruments):
            raise ValueError(
                "Hard negatives require synthetic mixing and synthetic backgrounds."
            )
        active_mapping = active_hard_negative_mapping(cfg)
        if not active_mapping:
            warnings.warn(
                "Hard negatives are enabled, but no configured target is selected; "
                "the option will have no effect."
            )
        selected_normalized = {normalize_name(name) for name in cfg.selected_instruments}
        for target, confusers in active_mapping.items():
            if normalize_name(target) in {normalize_name(name) for name in confusers}:
                raise ValueError(f"Hard-negative target {target!r} cannot confuse itself.")
            overlap = selected_normalized.intersection(
                normalize_name(name) for name in confusers
            )
            if overlap:
                raise ValueError(
                    f"Hard-negative confusers for {target!r} must not be model outputs."
                )
        maximum = cfg.synthetic_background_max_instruments
        maximum_forced = len(
            {
                normalize_name(confuser)
                for confusers in active_mapping.values()
                for confuser in confusers
            }
        )
        if maximum is not None and maximum < maximum_forced:
            raise ValueError(
                "synthetic_background_max_instruments is smaller than the number "
                "of hard-negative confusers that may be forced together."
            )
    if not (0.0 <= cfg.target_off_hard_negative_probability <= 1.0):
        raise ValueError("target_off_hard_negative_probability must be in [0, 1].")
    if cfg.target_off_hard_negative_min_seconds <= 0:
        raise ValueError("target_off_hard_negative_min_seconds must be positive.")
    if cfg.use_target_off_hard_negatives:
        if not (cfg.use_synthetic_mix and cfg.use_synthetic_background_instruments):
            raise ValueError(
                "Target-OFF hard negatives require synthetic mixing and backgrounds."
            )
        if not cfg.target_off_hard_negative_instruments:
            raise ValueError(
                "target_off_hard_negative_instruments must not be empty when enabled."
            )
        selected_by_normalized = {
            normalize_name(name): name for name in cfg.selected_instruments
        }
        configured_normalized = [
            normalize_name(name)
            for name in cfg.target_off_hard_negative_instruments
        ]
        if any(not name for name in configured_normalized):
            raise ValueError("Target-OFF hard-negative instrument names must not be empty.")
        if len(set(configured_normalized)) != len(configured_normalized):
            raise ValueError(
                "target_off_hard_negative_instruments contains duplicate names."
            )
        unknown = [
            name
            for name in cfg.target_off_hard_negative_instruments
            if normalize_name(name) not in selected_by_normalized
        ]
        if unknown:
            raise ValueError(
                "Target-OFF hard-negative instruments must be selected model outputs: "
                + ", ".join(unknown)
            )
        target_off_ratios = (
            cfg.target_off_regular_synthetic_ratio,
            cfg.target_off_fully_off_ratio,
            cfg.target_off_partially_off_ratio,
        )
        if any(ratio <= 0.0 for ratio in target_off_ratios):
            raise ValueError(
                "All regular/full/partial target-OFF source ratios must be positive."
            )
        if not math.isclose(
            sum(target_off_ratios),
            cfg.synthetic_mix_ratio,
            rel_tol=0.0,
            abs_tol=1e-8,
        ):
            raise ValueError(
                "The regular/full/partial target-OFF ratios must sum to "
                "synthetic_mix_ratio."
            )
        if cfg.target_off_hard_negative_min_seconds > cfg.sequence_seconds:
            raise ValueError(
                "target_off_hard_negative_min_seconds cannot exceed sequence_seconds."
            )
        target_off_steps = (
            cfg.target_off_hard_negative_min_seconds / cfg.label_hop_seconds
        )
        if not math.isclose(
            target_off_steps, round(target_off_steps), rel_tol=0.0, abs_tol=1e-8
        ):
            raise ValueError(
                "target_off_hard_negative_min_seconds must align to label_hop_seconds."
            )
    if not (0.0 < cfg.counterfactual_target_activity_fraction < 1.0):
        raise ValueError("counterfactual_target_activity_fraction must be in (0, 1).")
    if not (
        cfg.counterfactual_target_activity_fraction
        < cfg.counterfactual_min_original_activity_fraction
        <= 1.0
    ):
        raise ValueError(
            "counterfactual_min_original_activity_fraction must be greater than "
            "the target fraction and no greater than 1."
        )
    if cfg.counterfactual_min_interval_seconds <= 0:
        raise ValueError("counterfactual_min_interval_seconds must be positive.")
    if cfg.counterfactual_max_interval_seconds < cfg.counterfactual_min_interval_seconds:
        raise ValueError("counterfactual_max_interval_seconds must be >= the minimum.")
    if cfg.counterfactual_min_intervals < 1 or cfg.counterfactual_max_attempts < 1:
        raise ValueError("counterfactual interval/attempt counts must be >= 1.")
    if not (0.0 <= cfg.counterfactual_fade_seconds <= cfg.label_hop_seconds):
        raise ValueError("counterfactual_fade_seconds must be between 0 and label_hop_seconds.")
    if not (0.0 <= cfg.controlled_validation_selection_weight <= 1.0):
        raise ValueError("controlled_validation_selection_weight must be in [0, 1].")
    if not (0.0 <= cfg.worst_class_selection_weight <= 1.0):
        raise ValueError("worst_class_selection_weight must be in [0, 1].")
    if cfg.early_stopping_patience < 1:
        raise ValueError("early_stopping_patience must be positive.")
    if not isinstance(cfg.use_early_stopping, bool):
        raise ValueError("use_early_stopping must be a boolean.")
    if not isinstance(cfg.deterministic_epoch_sampling, bool):
        raise ValueError("deterministic_epoch_sampling must be a boolean.")
    if not isinstance(cfg.class_aware_sampling, bool):
        raise ValueError("class_aware_sampling must be a boolean.")
    if cfg.class_aware_sampling:
        raise ValueError(
            "class_aware_sampling must be False; ON/OFF is balanced in the loss."
        )
    if cfg.train_samples_per_epoch is not None and cfg.train_samples_per_epoch < 1:
        raise ValueError("train_samples_per_epoch must be positive when provided.")
    if not (0 <= cfg.warmup_epochs <= cfg.lr_hold_epochs < cfg.epochs):
        raise ValueError(
            "Require 0 <= warmup_epochs <= lr_hold_epochs < epochs."
        )
    if not isinstance(cfg.use_ema, bool) or not (0.0 < cfg.ema_decay < 1.0):
        raise ValueError("EMA configuration is invalid.")
    if cfg.num_workers < 0 or cfg.prefetch_factor < 1:
        raise ValueError("Data-loader worker/prefetch settings are invalid.")
    if cfg.persistent_workers and cfg.num_workers == 0:
        raise ValueError("persistent_workers requires num_workers > 0.")
    if cfg.persistent_workers and cfg.deterministic_epoch_sampling:
        raise ValueError(
            "persistent_workers must be False so epoch-dependent synthesis reaches workers."
        )
    if cfg.synthetic_min_total_instruments < 2:
        raise ValueError("synthetic_min_total_instruments must be at least 2.")
    if cfg.min_encoder_learning_rate < 0 or cfg.min_new_layers_learning_rate < 0:
        raise ValueError("Minimum learning rates must be non-negative.")
    if cfg.final_checkpoint_metric not in {"natural", "controlled", "combined"}:
        raise ValueError(
            "final_checkpoint_metric must be 'natural', 'controlled', or 'combined'."
        )




In [ ]:
from __future__ import annotations

# =============================================================================
# DRIVE CATALOG AND LOCAL AUDIO CACHE
# =============================================================================


@dataclass
class SourceCatalog:
    root: Path
    mixes: Dict[SongId, Path]
    stems: Dict[str, Dict[SongId, Path]]
    resolved_instrument_dirs: Dict[str, str]


@dataclass
class LocalCatalog:
    root: Path
    sample_rate: int
    mixes: Dict[SongId, Path]
    stems: Dict[str, Dict[SongId, Path]]


def _resolve_instrument_directories(root: Path, instruments: Sequence[str]) -> Dict[str, Path]:
    available_dirs = [p for p in root.iterdir() if p.is_dir()]
    normalized_to_paths: Dict[str, List[Path]] = defaultdict(list)
    for path in available_dirs:
        normalized_to_paths[normalize_name(path.name)].append(path)

    resolved: Dict[str, Path] = {}
    for requested in instruments:
        candidates = normalized_to_paths.get(normalize_name(requested), [])
        if len(candidates) == 1:
            resolved[requested] = candidates[0]
        elif len(candidates) > 1:
            raise RuntimeError(
                f"Ambiguous instrument folder for {requested!r}: {[str(p) for p in candidates]}"
            )
        else:
            available = ", ".join(sorted(p.name for p in available_dirs))
            raise FileNotFoundError(
                f"Instrument folder {requested!r} was not found under {root}. "
                f"Available folders: {available}"
            )
    return resolved


def scan_source_catalog(
    root: Path,
    instruments: Sequence[str],
    include_background_instruments: bool = False,
) -> SourceCatalog:
    if not root.exists():
        raise FileNotFoundError(f"Data root does not exist: {root}")

    mixes: Dict[SongId, Path] = {}
    for path in sorted(root.iterdir()):
        if not path.is_file() or not is_readable_audio(path):
            continue
        song_id = extract_song_id(path.name)
        if song_id is None:
            continue
        if song_id in mixes:
            raise RuntimeError(
                f"Multiple root-level mix files map to song {song_id}: "
                f"{mixes[song_id]} and {path}"
            )
        mixes[song_id] = path

    if not mixes:
        raise FileNotFoundError(f"No readable root-level mix files were found under {root}")

    resolved_dirs = _resolve_instrument_directories(root, instruments)
    if include_background_instruments:
        selected_paths = {path.resolve() for path in resolved_dirs.values()}
        normalized_names = {normalize_name(name) for name in resolved_dirs}
        for directory in sorted(path for path in root.iterdir() if path.is_dir()):
            if directory.resolve() in selected_paths:
                continue
            normalized = normalize_name(directory.name)
            if normalized in normalized_names:
                warnings.warn(
                    f"Skipping ambiguous background instrument directory: {directory}"
                )
                continue
            has_audio = any(
                path.is_file()
                and extract_song_id(path.name) is not None
                and is_readable_audio(path)
                for path in directory.iterdir()
            )
            if has_audio:
                resolved_dirs[directory.name] = directory
                normalized_names.add(normalized)

    stems: Dict[str, Dict[SongId, Path]] = {
        instrument: {} for instrument in resolved_dirs
    }
    for instrument, directory in resolved_dirs.items():
        for path in sorted(directory.iterdir()):
            if not path.is_file() or not is_readable_audio(path):
                continue
            song_id = extract_song_id(path.name)
            if song_id is None:
                warnings.warn(f"Could not extract song ID from stem file: {path}")
                continue
            if song_id in stems[instrument]:
                raise RuntimeError(
                    f"Multiple stems for {instrument!r}, song {song_id}: "
                    f"{stems[instrument][song_id]} and {path}"
                )
            stems[instrument][song_id] = path

    return SourceCatalog(
        root=root,
        mixes=mixes,
        stems=stems,
        resolved_instrument_dirs={k: v.name for k, v in resolved_dirs.items()},
    )


def print_source_availability(
    catalog: SourceCatalog,
    instruments: Sequence[str],
    song_ids: Sequence[SongId],
    title: str,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for instrument in instruments:
        row: Dict[str, Any] = {"Instrument": instrument}
        for song_id in song_ids:
            row[f"Song {song_id}"] = "Found" if song_id in catalog.stems[instrument] else "Absent"
        rows.append(row)
    frame = pd.DataFrame(rows)
    print(f"\n{title}")
    print(frame.to_string(index=False))
    return frame


def _source_signature(path: Path, mode: AudioModeSpec, cfg: ExperimentConfig) -> Dict[str, Any]:
    stat = path.stat()
    return {
        "source": str(path),
        "source_size": int(stat.st_size),
        "source_mtime_ns": int(stat.st_mtime_ns),
        "mode": mode.name,
        "target_sr": mode.target_sr,
        "processing": mode.processing,
        "lowpass_cutoff_hz": cfg.lowpass_cutoff_hz if mode.processing == "lowpass" else None,
        "lowpass_numtaps": cfg.lowpass_numtaps if mode.processing == "lowpass" else None,
        "lowpass_kaiser_beta": cfg.lowpass_kaiser_beta if mode.processing == "lowpass" else None,
    }


def _cache_is_current(destination: Path, signature: Mapping[str, Any]) -> bool:
    meta_path = destination.with_suffix(destination.suffix + ".meta.json")
    if not destination.exists() or not meta_path.exists():
        return False
    try:
        with meta_path.open("r", encoding="utf-8") as handle:
            previous = json.load(handle)
        if previous != json_ready(dict(signature)):
            return False
        sf.info(str(destination))
        return True
    except Exception:
        return False


def _read_full_mono(path: Path) -> Tuple[np.ndarray, int]:
    audio, sr = sf.read(str(path), dtype="float32", always_2d=True)
    mono = np.mean(audio, axis=1, dtype=np.float32)
    del audio
    return np.ascontiguousarray(mono, dtype=np.float32), int(sr)


def _process_audio_to_cache(
    source: Path,
    destination: Path,
    mode: AudioModeSpec,
    cfg: ExperimentConfig,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    signature = _source_signature(source, mode, cfg)

    if not cfg.force_rebuild_audio_cache and _cache_is_current(destination, signature):
        return

    temp_path = destination.with_suffix(destination.suffix + ".tmp.wav")
    if temp_path.exists():
        temp_path.unlink()

    source_info = sf.info(str(source))

    if mode.processing == "copy":
        if int(source_info.samplerate) != mode.target_sr:
            raise ValueError(
                f"Expected {mode.target_sr} Hz in {source}, found {source_info.samplerate} Hz."
            )
        shutil.copy2(source, temp_path)

    elif mode.processing == "lowpass":
        mono, source_sr = _read_full_mono(source)
        if source_sr != 96000:
            raise ValueError(f"96k_lp22 requires a 96 kHz source, found {source_sr} in {source}")
        taps = signal.firwin(
            numtaps=cfg.lowpass_numtaps,
            cutoff=cfg.lowpass_cutoff_hz,
            fs=source_sr,
            window=("kaiser", cfg.lowpass_kaiser_beta),
            pass_zero="lowpass",
        ).astype(np.float32)
        filtered = signal.oaconvolve(mono, taps, mode="same").astype(np.float32, copy=False)
        sf.write(str(temp_path), filtered, source_sr, subtype="FLOAT")
        del mono, filtered, taps

    elif mode.processing == "downsample16":
        mono, source_sr = _read_full_mono(source)
        if source_sr != 96000:
            raise ValueError(f"16k mode requires a 96 kHz source, found {source_sr} in {source}")
        downsampled = signal.resample_poly(mono, up=1, down=6, axis=0).astype(np.float32, copy=False)
        sf.write(str(temp_path), downsampled, 16000, subtype="FLOAT")
        del mono, downsampled

    else:
        raise ValueError(f"Unsupported cache processing mode: {mode.processing}")

    os.replace(temp_path, destination)
    save_json(destination.with_suffix(destination.suffix + ".meta.json"), signature)
    gc.collect()


def prepare_local_catalog(
    source_catalog: SourceCatalog,
    song_ids: Sequence[SongId],
    mode: AudioModeSpec,
    cfg: ExperimentConfig,
) -> LocalCatalog:
    """Copy/process mixes and every discovered target/background stem locally."""
    catalog_instruments = tuple(source_catalog.stems)
    if not cfg.use_local_audio_cache:
        if mode.processing != "copy":
            raise ValueError(
                "96k_lp22 and 16k require a processed cache. Set use_local_audio_cache=True."
            )
        return LocalCatalog(
            root=source_catalog.root,
            sample_rate=mode.target_sr,
            mixes={song_id: source_catalog.mixes[song_id] for song_id in song_ids},
            stems={
                instrument: {
                    song_id: path
                    for song_id, path in source_catalog.stems[instrument].items()
                    if song_id in song_ids
                }
                for instrument in catalog_instruments
            },
        )

    cache_root = Path(cfg.local_cache_root) / mode.name
    mix_root = cache_root / "mixes"
    stem_root = cache_root / "stems"
    cache_root.mkdir(parents=True, exist_ok=True)

    jobs: List[Tuple[str, Optional[str], SongId, Path, Path]] = []
    for song_id in song_ids:
        source = source_catalog.mixes[song_id]
        destination = mix_root / f"{song_id}.wav"
        jobs.append(("mix", None, song_id, source, destination))

    for instrument in catalog_instruments:
        folder = stem_root / safe_slug(instrument)
        for song_id, source in source_catalog.stems[instrument].items():
            if song_id not in song_ids:
                continue
            destination = folder / f"{song_id}.wav"
            jobs.append(("stem", instrument, song_id, source, destination))

    print(f"\nPreparing local audio cache for mode {mode.name!r}: {cache_root}")
    for _, _, _, source, destination in tqdm(jobs, desc="Audio cache"):
        _process_audio_to_cache(source, destination, mode, cfg)

    mixes = {song_id: mix_root / f"{song_id}.wav" for song_id in song_ids}
    stems: Dict[str, Dict[SongId, Path]] = {
        instrument: {} for instrument in catalog_instruments
    }
    for instrument in catalog_instruments:
        folder = stem_root / safe_slug(instrument)
        for song_id in song_ids:
            path = folder / f"{song_id}.wav"
            if path.exists():
                stems[instrument][song_id] = path

    return LocalCatalog(
        root=cache_root,
        sample_rate=mode.target_sr,
        mixes=mixes,
        stems=stems,
    )


def validate_synthetic_background_availability(
    local_catalog: LocalCatalog,
    song_ids: Sequence[SongId],
    selected_instruments: Sequence[str],
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    selected = {normalize_name(name) for name in selected_instruments}
    rows: List[Dict[str, Any]] = []
    for song_id in song_ids:
        available = sorted(
            instrument
            for instrument, paths in local_catalog.stems.items()
            if normalize_name(instrument) not in selected and song_id in paths
        )
        rows.append(
            {
                "song_id": song_id,
                "background_instrument_count": len(available),
                "background_instruments": ", ".join(available),
            }
        )
        background_stems_required = (
            (cfg.use_synthetic_mix and cfg.use_synthetic_background_instruments)
            or (
                cfg.use_controlled_dropout_validation
                and cfg.use_controlled_polyphonic_validation
            )
        )
        if background_stems_required and len(available) < cfg.synthetic_background_min_instruments:
            raise RuntimeError(
                f"Song {song_id} has only {len(available)} non-target stems; "
                f"at least {cfg.synthetic_background_min_instruments} are required."
            )
    frame = pd.DataFrame(rows)
    print("\nNon-target stems available for synthetic backgrounds:")
    print(frame.to_string(index=False))
    return frame




In [ ]:
from __future__ import annotations

# =============================================================================
# AUTOMATIC ACTIVITY LABELS
# =============================================================================


@dataclass
class LabelStore:
    instruments: Tuple[str, ...]
    class_to_index: Dict[str, int]
    song_labels: Dict[SongId, np.ndarray]  # [half-second steps, classes]
    song_rms_dbfs: Dict[SongId, np.ndarray]  # raw RMS dBFS before gap/event smoothing
    song_durations: Dict[SongId, float]
    label_hop_seconds: float

    def get_sequence(self, song_id: SongId, start_step: int, steps: int) -> np.ndarray:
        full = self.song_labels[song_id]
        result = np.zeros((steps, full.shape[1]), dtype=np.float32)
        if start_step >= len(full):
            return result
        available = min(steps, len(full) - start_step)
        result[:available] = full[start_step : start_step + available]
        return result


@dataclass
class CounterfactualDropoutPlan:
    name: str
    audio_keep_masks: Dict[SongId, np.ndarray]  # [steps, classes], boolean
    song_labels: Dict[SongId, np.ndarray]  # labels after dropout
    summary: pd.DataFrame

    @staticmethod
    def _slice_with_padding(full: np.ndarray, start_step: int, steps: int, fill: float) -> np.ndarray:
        result = np.full((steps, full.shape[1]), fill, dtype=full.dtype)
        if start_step >= len(full):
            return result
        available = min(steps, len(full) - start_step)
        result[:available] = full[start_step : start_step + available]
        return result

    def get_target_sequence(self, song_id: SongId, start_step: int, steps: int) -> np.ndarray:
        return self._slice_with_padding(
            self.song_labels[song_id], start_step, steps, fill=0.0
        ).astype(np.float32, copy=False)

    def get_keep_sequence(self, song_id: SongId, start_step: int, steps: int) -> np.ndarray:
        return self._slice_with_padding(
            self.audio_keep_masks[song_id], start_step, steps, fill=True
        ).astype(bool, copy=False)

    def previous_keep(self, song_id: SongId, start_step: int) -> np.ndarray:
        full = self.audio_keep_masks[song_id]
        if start_step <= 0 or len(full) == 0:
            return np.ones(full.shape[1], dtype=bool)
        return full[min(start_step - 1, len(full) - 1)].astype(bool, copy=False)


def _fill_short_false_runs(values: np.ndarray, max_gap_steps: int) -> np.ndarray:
    result = values.astype(bool, copy=True)
    if max_gap_steps <= 0 or len(result) == 0:
        return result
    index = 0
    while index < len(result):
        if result[index]:
            index += 1
            continue
        start = index
        while index < len(result) and not result[index]:
            index += 1
        end = index
        bounded = start > 0 and end < len(result) and result[start - 1] and result[end]
        if bounded and (end - start) <= max_gap_steps:
            result[start:end] = True
    return result


def _remove_short_true_runs(values: np.ndarray, min_event_steps: int) -> np.ndarray:
    result = values.astype(bool, copy=True)
    if min_event_steps <= 1 or len(result) == 0:
        return result
    index = 0
    while index < len(result):
        if not result[index]:
            index += 1
            continue
        start = index
        while index < len(result) and result[index]:
            index += 1
        end = index
        if (end - start) < min_event_steps:
            result[start:end] = False
    return result


def compute_activity_labels(
    stem_path: Path,
    target_duration_seconds: float,
    cfg: ExperimentConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute one binary activity label every label_hop_seconds.

    Label i represents the half-second interval ending at (i+1)*hop and uses
    the RMS energy in the preceding label_window_seconds. With the default
    0.5-second window this gives 16 aligned labels for an 8-second sequence.
    """
    mono, sr = _read_full_mono(stem_path)
    target_samples = int(round(target_duration_seconds * sr))
    hop_samples = int(round(cfg.label_hop_seconds * sr))
    window_samples = int(round(cfg.label_window_seconds * sr))
    num_steps = int(math.ceil(target_duration_seconds / cfg.label_hop_seconds))

    # Cumulative squared energy. We do not peak-normalize: dBFS must retain the
    # original studio level and its large silence-to-playing separation.
    energy = mono.astype(np.float64, copy=False)
    energy = np.square(energy, dtype=np.float64)
    cumulative = np.empty(len(energy) + 1, dtype=np.float64)
    cumulative[0] = 0.0
    np.cumsum(energy, out=cumulative[1:])

    rms_db = np.empty(num_steps, dtype=np.float32)
    available_samples = len(mono)

    for step in range(num_steps):
        end = min((step + 1) * hop_samples, target_samples)
        start = max(0, end - window_samples)
        denominator = max(1, end - start)

        clamped_start = min(start, available_samples)
        clamped_end = min(end, available_samples)
        sum_squares = cumulative[clamped_end] - cumulative[clamped_start]
        rms = math.sqrt(max(0.0, float(sum_squares)) / float(denominator) + 1e-20)
        rms_db[step] = 20.0 * math.log10(rms + 1e-12)

    active = rms_db > cfg.activity_threshold_dbfs
    max_gap_steps = int(round(cfg.max_internal_gap_seconds / cfg.label_hop_seconds))
    min_event_steps = max(1, int(math.ceil(cfg.minimum_event_seconds / cfg.label_hop_seconds)))
    active = _fill_short_false_runs(active, max_gap_steps=max_gap_steps)
    active = _remove_short_true_runs(active, min_event_steps=min_event_steps)

    del mono, energy, cumulative
    gc.collect()
    return active.astype(np.uint8), rms_db


def _label_cache_metadata(
    source: Path,
    instrument: str,
    song_id: SongId,
    duration: float,
    cfg: ExperimentConfig,
) -> Dict[str, Any]:
    stat = source.stat()
    return {
        "source": str(source),
        "source_size": int(stat.st_size),
        "source_mtime_ns": int(stat.st_mtime_ns),
        "instrument": instrument,
        "song_id": song_id,
        "target_duration_seconds": float(duration),
        "label_window_seconds": cfg.label_window_seconds,
        "label_hop_seconds": cfg.label_hop_seconds,
        "activity_threshold_dbfs": cfg.activity_threshold_dbfs,
        "max_internal_gap_seconds": cfg.max_internal_gap_seconds,
        "minimum_event_seconds": cfg.minimum_event_seconds,
    }


def load_or_create_activity_labels(
    source: Path,
    instrument: str,
    song_id: SongId,
    duration: float,
    cache_root: Path,
    cfg: ExperimentConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    cache_dir = cache_root / safe_slug(instrument)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"song_{song_id}.npz"
    metadata = _label_cache_metadata(source, instrument, song_id, duration, cfg)

    if cache_path.exists() and not cfg.force_rebuild_labels:
        try:
            with np.load(cache_path, allow_pickle=False) as content:
                cached_meta = json.loads(str(content["metadata"].item()))
                if cached_meta == json_ready(metadata):
                    return content["labels"].astype(np.uint8), content["rms_dbfs"].astype(np.float32)
        except Exception:
            pass

    labels, rms_db = compute_activity_labels(source, duration, cfg)
    np.savez_compressed(
        cache_path,
        labels=labels,
        rms_dbfs=rms_db,
        metadata=np.array(json.dumps(json_ready(metadata), ensure_ascii=False)),
    )
    return labels, rms_db


def build_label_store(
    label_source_catalog: SourceCatalog,
    local_catalog: LocalCatalog,
    song_ids: Sequence[SongId],
    instruments: Sequence[str],
    label_cache_root: Path,
    cfg: ExperimentConfig,
) -> Tuple[LabelStore, pd.DataFrame]:
    class_to_index = {instrument: index for index, instrument in enumerate(instruments)}
    song_durations = {song_id: audio_duration(local_catalog.mixes[song_id]) for song_id in song_ids}
    song_labels: Dict[SongId, np.ndarray] = {}
    song_rms_dbfs: Dict[SongId, np.ndarray] = {}
    summary_rows: List[Dict[str, Any]] = []

    print("\nBuilding/loading automatic activity labels from 96 kHz master stems...")
    for song_id in song_ids:
        duration = song_durations[song_id]
        num_steps = int(math.ceil(duration / cfg.label_hop_seconds))
        matrix = np.zeros((num_steps, len(instruments)), dtype=np.float32)
        rms_matrix = np.full(
            (num_steps, len(instruments)), -np.inf, dtype=np.float32
        )

        for instrument in instruments:
            class_index = class_to_index[instrument]
            source = label_source_catalog.stems[instrument].get(song_id)
            if source is None:
                labels = np.zeros(num_steps, dtype=np.uint8)
                rms_db = np.full(num_steps, -np.inf, dtype=np.float32)
                status = "Absent"
            else:
                labels, rms_db = load_or_create_activity_labels(
                    source=source,
                    instrument=instrument,
                    song_id=song_id,
                    duration=duration,
                    cache_root=label_cache_root,
                    cfg=cfg,
                )
                if len(labels) < num_steps:
                    labels = np.pad(labels, (0, num_steps - len(labels)))
                    rms_db = np.pad(rms_db, (0, num_steps - len(rms_db)), constant_values=-np.inf)
                labels = labels[:num_steps]
                rms_db = rms_db[:num_steps]
                status = "Found"

            matrix[:, class_index] = labels.astype(np.float32)
            rms_matrix[:, class_index] = rms_db.astype(np.float32)
            finite_rms = rms_db[np.isfinite(rms_db)]
            summary_rows.append(
                {
                    "song_id": song_id,
                    "instrument": instrument,
                    "stem": status,
                    "activity_percent": float(labels.mean() * 100.0) if len(labels) else 0.0,
                    "minimum_rms_dbfs": float(np.min(finite_rms)) if len(finite_rms) else np.nan,
                    "maximum_rms_dbfs": float(np.max(finite_rms)) if len(finite_rms) else np.nan,
                }
            )

        song_labels[song_id] = matrix
        song_rms_dbfs[song_id] = rms_matrix

    summary = pd.DataFrame(summary_rows)
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    return (
        LabelStore(
            instruments=tuple(instruments),
            class_to_index=class_to_index,
            song_labels=song_labels,
            song_rms_dbfs=song_rms_dbfs,
            song_durations=song_durations,
            label_hop_seconds=cfg.label_hop_seconds,
        ),
        summary,
    )



def _labels_after_audio_dropout(
    original_labels: np.ndarray,
    audio_keep: np.ndarray,
    label_window_steps: int,
) -> np.ndarray:
    """Apply the trailing label window to a planned audio keep/mute mask."""
    original = original_labels.astype(bool, copy=False)
    keep = audio_keep.astype(bool, copy=False)
    recent_keep = np.zeros(len(keep), dtype=bool)
    cumulative = np.concatenate(([0], np.cumsum(keep.astype(np.int32))))
    for step in range(len(keep)):
        start = max(0, step - label_window_steps + 1)
        recent_keep[step] = (cumulative[step + 1] - cumulative[start]) > 0
    return (original & recent_keep).astype(np.float32)


def build_counterfactual_dropout_plan(
    label_store: LabelStore,
    local_catalog: LocalCatalog,
    song_ids: Sequence[SongId],
    cfg: ExperimentConfig,
    *,
    name: str,
    seed_offset: int,
) -> CounterfactualDropoutPlan:
    """
    Build a deterministic, full-song dropout plan.

    Several non-overlapping long intervals are selected for any instrument whose
    original activity exceeds the configured threshold. Intervals are added until
    the counterfactual label activity is at or below the target fraction. Because
    the plan lives on the full song, central 8-second windows and transition
    windows are both seen by the model.
    """
    min_steps = max(1, int(round(cfg.counterfactual_min_interval_seconds / cfg.label_hop_seconds)))
    max_steps = max(min_steps, int(round(cfg.counterfactual_max_interval_seconds / cfg.label_hop_seconds)))
    label_window_steps = max(1, int(round(cfg.label_window_seconds / cfg.label_hop_seconds)))

    keep_by_song: Dict[SongId, np.ndarray] = {}
    labels_by_song: Dict[SongId, np.ndarray] = {}
    rows: List[Dict[str, Any]] = []

    for song_id in song_ids:
        original_matrix = label_store.song_labels[song_id].astype(np.float32, copy=False)
        keep_matrix = np.ones_like(original_matrix, dtype=bool)
        planned_matrix = original_matrix.copy()

        for instrument, class_index in label_store.class_to_index.items():
            original = original_matrix[:, class_index] > 0.5
            keep = np.ones(len(original), dtype=bool)
            original_fraction = float(original.mean()) if len(original) else 0.0
            intervals: List[Tuple[int, int]] = []

            can_dropout = (
                song_id in local_catalog.stems[instrument]
                and original_fraction >= cfg.counterfactual_min_original_activity_fraction
                and np.any(original)
            )

            if can_dropout:
                rng = np.random.default_rng(
                    int(
                        cfg.seed
                        + seed_offset
                        + song_id_seed(song_id)
                        + class_index * 9176
                    ) % (2**32)
                )
                attempts = 0
                effective = _labels_after_audio_dropout(original, keep, label_window_steps)

                while attempts < cfg.counterfactual_max_attempts:
                    need_more_intervals = len(intervals) < cfg.counterfactual_min_intervals
                    need_lower_activity = float(effective.mean()) > cfg.counterfactual_target_activity_fraction
                    if not (need_more_intervals or need_lower_activity):
                        break

                    attempts += 1
                    duration_steps = int(rng.integers(min_steps, max_steps + 1))
                    duration_steps = min(duration_steps, len(original))
                    if duration_steps <= 0:
                        break

                    possible_starts = np.arange(0, len(original) - duration_steps + 1, dtype=int)
                    if len(possible_starts) == 0:
                        break

                    # Prefer intervals that remove currently retained positive
                    # activity while still randomizing among strong candidates.
                    scores = np.array(
                        [
                            np.count_nonzero(original[start : start + duration_steps] & keep[start : start + duration_steps])
                            for start in possible_starts
                        ],
                        dtype=np.int32,
                    )
                    best = int(scores.max()) if len(scores) else 0
                    if best <= 0:
                        break
                    threshold = max(1, int(math.ceil(0.75 * best)))
                    strong = possible_starts[scores >= threshold]
                    start = int(rng.choice(strong))
                    end = start + duration_steps

                    # Avoid counting a fully covered interval twice.
                    if not np.any(keep[start:end]):
                        continue
                    keep[start:end] = False
                    intervals.append((start, end))
                    effective = _labels_after_audio_dropout(original, keep, label_window_steps)

                if float(effective.mean()) > cfg.counterfactual_target_activity_fraction + 1e-6:
                    warnings.warn(
                        f"{name}: could not reduce {instrument}, song {song_id}, to the requested "
                        f"activity fraction after {attempts} attempts."
                    )
            else:
                effective = original.astype(np.float32)

            keep_matrix[:, class_index] = keep
            planned_matrix[:, class_index] = effective
            rows.append(
                {
                    "plan": name,
                    "song_id": song_id,
                    "instrument": instrument,
                    "original_activity_percent": 100.0 * original_fraction,
                    "counterfactual_activity_percent": 100.0 * float(effective.mean()) if len(effective) else 0.0,
                    "dropout_interval_count": len(intervals),
                    "muted_seconds": float(np.count_nonzero(~keep) * cfg.label_hop_seconds),
                    "intervals_seconds": [
                        [
                            float(start * cfg.label_hop_seconds),
                            float(end * cfg.label_hop_seconds),
                        ]
                        for start, end in intervals
                    ],
                }
            )

        keep_by_song[song_id] = keep_matrix
        labels_by_song[song_id] = planned_matrix.astype(np.float32, copy=False)

    summary = pd.DataFrame(rows)
    print(f"\n{name} counterfactual dropout plan:")
    display_columns = [
        "song_id",
        "instrument",
        "original_activity_percent",
        "counterfactual_activity_percent",
        "dropout_interval_count",
        "muted_seconds",
    ]
    print(summary[display_columns].to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    return CounterfactualDropoutPlan(
        name=name,
        audio_keep_masks=keep_by_song,
        song_labels=labels_by_song,
        summary=summary,
    )




In [ ]:
from __future__ import annotations

# =============================================================================
# DATASET DESCRIPTORS AND AUDIO LOADING
# =============================================================================


@dataclass(frozen=True)
class SequenceDescriptor:
    source_type: str
    song_id: SongId
    start_step: int
    instrument_index: int = -1


SYNTHETIC_REGULAR = "synthetic"
SYNTHETIC_TARGET_OFF_FULL = "synthetic_target_off_full"
SYNTHETIC_TARGET_OFF_PARTIAL = "synthetic_target_off_partial"
SYNTHETIC_SOURCE_TYPES = frozenset(
    {
        SYNTHETIC_REGULAR,
        SYNTHETIC_TARGET_OFF_FULL,
        SYNTHETIC_TARGET_OFF_PARTIAL,
    }
)
TARGET_OFF_SOURCE_TYPES = frozenset(
    {SYNTHETIC_TARGET_OFF_FULL, SYNTHETIC_TARGET_OFF_PARTIAL}
)


def longest_true_run(values: np.ndarray) -> int:
    longest = 0
    current = 0
    for value in np.asarray(values, dtype=bool).reshape(-1):
        if bool(value):
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest


def _sequence_start_steps(total_steps: int, sequence_steps: int, hop_steps: int) -> List[int]:
    if total_steps <= sequence_steps:
        return [0]
    last_start = total_steps - sequence_steps
    starts = list(range(0, last_start + 1, hop_steps))
    if starts[-1] != last_start:
        starts.append(last_start)
    return starts


def build_descriptors(
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    train_song_ids: Sequence[SongId],
    val_song_ids: Sequence[SongId],
    cfg: ExperimentConfig,
) -> Tuple[List[SequenceDescriptor], List[SequenceDescriptor], pd.DataFrame]:
    sequence_steps = int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
    train_hop_steps = int(round(cfg.sequence_hop_seconds / cfg.label_hop_seconds))
    val_hop_steps = int(round(cfg.val_sequence_hop_seconds / cfg.label_hop_seconds))

    train_descriptors: List[SequenceDescriptor] = []
    val_descriptors: List[SequenceDescriptor] = []

    for song_id in train_song_ids:
        labels = label_store.song_labels[song_id]
        starts = _sequence_start_steps(len(labels), sequence_steps, train_hop_steps)

        if cfg.use_real_mix:
            train_descriptors.extend(
                SequenceDescriptor("real_mix", song_id, start_step) for start_step in starts
            )

        if cfg.use_stems:
            for instrument, class_index in label_store.class_to_index.items():
                if song_id not in local_catalog.stems[instrument]:
                    continue
                for start_step in starts:
                    target = labels[start_step : start_step + sequence_steps, class_index]
                    if np.any(target > 0.5):
                        train_descriptors.append(
                            SequenceDescriptor("stem", song_id, start_step, class_index)
                        )

        if cfg.use_synthetic_mix:
            for start_step in starts:
                target = labels[start_step : start_step + sequence_steps]
                has_available_positive = False
                for instrument, class_index in label_store.class_to_index.items():
                    if song_id in local_catalog.stems[instrument] and np.any(target[:, class_index] > 0.5):
                        has_available_positive = True
                        break
                if has_available_positive:
                    train_descriptors.append(SequenceDescriptor("synthetic", song_id, start_step))

    # Primary validation is always the untouched real polyphonic mix. A
    # separate controlled counterfactual loader is built later when enabled.
    for song_id in val_song_ids:
        labels = label_store.song_labels[song_id]
        starts = _sequence_start_steps(len(labels), sequence_steps, val_hop_steps)
        val_descriptors.extend(
            SequenceDescriptor("real_mix", song_id, start_step) for start_step in starts
        )

    counts = Counter(descriptor.source_type for descriptor in train_descriptors)
    summary = pd.DataFrame(
        [{"source_type": source_type, "sequence_count": count} for source_type, count in counts.items()]
    )
    print("\nTraining sequence descriptors:")
    print(summary.to_string(index=False) if len(summary) else "No training descriptors")
    print(f"Validation real-mix sequences: {len(val_descriptors)}")

    if not train_descriptors:
        raise RuntimeError("No training descriptors were created. Check paths, classes, and labels.")
    if not val_descriptors:
        raise RuntimeError("No validation descriptors were created.")
    return train_descriptors, val_descriptors, summary


def read_audio_segment(
    path: Path,
    start_seconds: float,
    duration_seconds: float,
    expected_sr: int,
) -> np.ndarray:
    info = sf.info(str(path))
    if int(info.samplerate) != int(expected_sr):
        raise ValueError(f"Expected {expected_sr} Hz in {path}, found {info.samplerate} Hz")

    segment_samples = int(round(duration_seconds * expected_sr))
    start_sample = int(round(start_seconds * expected_sr))
    stop_sample = start_sample + segment_samples

    audio, _ = sf.read(
        str(path),
        start=max(0, start_sample),
        stop=max(0, stop_sample),
        dtype="float32",
        always_2d=True,
    )
    mono = np.mean(audio, axis=1, dtype=np.float32)
    del audio

    if len(mono) < segment_samples:
        mono = np.pad(mono, (0, segment_samples - len(mono)))
    elif len(mono) > segment_samples:
        mono = mono[:segment_samples]

    mono = np.nan_to_num(mono, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
    return np.ascontiguousarray(mono)



def hard_negative_overlap_steps(
    song_id: SongId,
    start_step: int,
    sequence_steps: int,
    target_label_store: LabelStore,
    hard_negative_label_store: LabelStore,
    mapping: Mapping[str, Sequence[str]],
    target_sequence: Optional[np.ndarray] = None,
) -> Dict[Tuple[str, str], int]:
    """Count 0.5-second target-OFF/confuser-ON overlaps in one sequence."""
    if target_sequence is None:
        target_sequence = target_label_store.get_sequence(
            song_id, start_step, sequence_steps
        )
    confuser_sequence = hard_negative_label_store.get_sequence(
        song_id, start_step, sequence_steps
    )
    overlaps: Dict[Tuple[str, str], int] = {}
    for target, confusers in mapping.items():
        target_index = target_label_store.class_to_index[target]
        target_off = target_sequence[:, target_index] <= 0.5
        for confuser in confusers:
            confuser_index = hard_negative_label_store.class_to_index[confuser]
            confuser_on = confuser_sequence[:, confuser_index] > 0.5
            overlaps[(target, confuser)] = int(
                np.count_nonzero(target_off & confuser_on)
            )
    return overlaps


def summarize_hard_negative_eligibility(
    descriptors: Sequence[SequenceDescriptor],
    local_catalog: LocalCatalog,
    target_label_store: LabelStore,
    hard_negative_label_store: Optional[LabelStore],
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    mapping = active_hard_negative_mapping(cfg)
    if not cfg.use_hard_negative_backgrounds or not mapping:
        return pd.DataFrame()
    if hard_negative_label_store is None:
        raise RuntimeError("Hard-negative labels were not built.")

    minimum_steps = max(
        1,
        int(math.ceil(
            cfg.hard_negative_min_overlap_seconds / cfg.label_hop_seconds
        )),
    )
    sequence_steps = int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
    stats: Dict[Tuple[str, str], Dict[str, int]] = {
        (target, confuser): {
            "available_sequences": 0,
            "eligible_sequences": 0,
            "overlap_steps": 0,
        }
        for target, confusers in mapping.items()
        for confuser in confusers
    }
    for descriptor in descriptors:
        if descriptor.source_type != "synthetic":
            continue
        overlaps = hard_negative_overlap_steps(
            descriptor.song_id,
            descriptor.start_step,
            sequence_steps,
            target_label_store,
            hard_negative_label_store,
            mapping,
        )
        local_by_normalized = {
            normalize_name(instrument): instrument
            for instrument, paths in local_catalog.stems.items()
            if descriptor.song_id in paths
        }
        for pair, overlap_count in overlaps.items():
            _, confuser = pair
            if normalize_name(confuser) not in local_by_normalized:
                continue
            pair_stats = stats[pair]
            pair_stats["available_sequences"] += 1
            pair_stats["overlap_steps"] += overlap_count
            if overlap_count >= minimum_steps:
                pair_stats["eligible_sequences"] += 1

    rows: List[Dict[str, Any]] = []
    for (target, confuser), pair_stats in stats.items():
        available = pair_stats["available_sequences"]
        eligible = pair_stats["eligible_sequences"]
        rows.append(
            {
                "target": target,
                "confuser": confuser,
                "available_synthetic_sequences": available,
                "eligible_synthetic_sequences": eligible,
                "eligible_sequence_percent": (
                    100.0 * eligible / available if available else 0.0
                ),
                "target_OFF_confuser_ON_seconds": (
                    pair_stats["overlap_steps"] * cfg.label_hop_seconds
                ),
                "force_probability_when_eligible": (
                    cfg.synthetic_background_probability
                    * cfg.hard_negative_probability
                ),
            }
        )
    summary = pd.DataFrame(rows)
    print("\nHard-negative eligibility from original 96 kHz RMS labels:")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    if int(summary["eligible_synthetic_sequences"].sum()) == 0:
        raise RuntimeError(
            "Hard negatives are enabled, but no training sequence contains "
            "the required target-OFF/confuser-ON overlap."
        )
    return summary


def _audio_keep_envelope(
    keep_steps: np.ndarray,
    previous_keep: bool,
    total_samples: int,
    hop_samples: int,
    fade_samples: int,
) -> np.ndarray:
    """Convert half-second keep/mute decisions to a sample-level smooth envelope."""
    keep = np.asarray(keep_steps, dtype=bool)
    envelope = np.repeat(keep.astype(np.float32), hop_samples)
    if len(envelope) < total_samples:
        pad_value = float(keep[-1]) if len(keep) else 1.0
        envelope = np.pad(envelope, (0, total_samples - len(envelope)), constant_values=pad_value)
    envelope = envelope[:total_samples].astype(np.float32, copy=False)

    if fade_samples <= 0 or total_samples == 0:
        return envelope

    extended = np.concatenate(([bool(previous_keep)], keep))
    transitions = np.flatnonzero(extended[1:] != extended[:-1])
    for step_index in transitions:
        boundary = int(step_index * hop_samples)
        old_value = bool(extended[step_index])
        new_value = bool(extended[step_index + 1])
        if old_value and not new_value:
            left = max(0, boundary - fade_samples)
            width = boundary - left
            if width > 0:
                envelope[left:boundary] = np.minimum(
                    envelope[left:boundary],
                    np.linspace(1.0, 0.0, width, endpoint=False, dtype=np.float32),
                )
        elif not old_value and new_value:
            right = min(total_samples, boundary + fade_samples)
            width = right - boundary
            if width > 0:
                envelope[boundary:right] = np.linspace(
                    0.0, 1.0, width, endpoint=False, dtype=np.float32
                )
    return envelope


class InstrumentSequenceDataset(Dataset):
    def __init__(
        self,
        descriptors: Sequence[SequenceDescriptor],
        local_catalog: LocalCatalog,
        label_store: LabelStore,
        cfg: ExperimentConfig,
        dropout_plan: Optional[CounterfactualDropoutPlan] = None,
        deterministic_synthetic: bool = False,
        background_mode: str = "sampled",
        fallback_to_real_mix_when_no_target: bool = True,
        hard_negative_label_store: Optional[LabelStore] = None,
        fixed_background_instruments: Optional[Sequence[str]] = None,
    ) -> None:
        self.descriptors = list(descriptors)
        self.local_catalog = local_catalog
        self.label_store = label_store
        self.cfg = cfg
        self.dropout_plan = dropout_plan
        self.deterministic_synthetic = deterministic_synthetic
        self.current_epoch = 0
        if background_mode not in {"none", "sampled", "all"}:
            raise ValueError("background_mode must be 'none', 'sampled', or 'all'.")
        self.background_mode = background_mode
        self.fallback_to_real_mix_when_no_target = fallback_to_real_mix_when_no_target
        self.hard_negative_label_store = hard_negative_label_store
        self.fixed_background_instruments = (
            tuple(fixed_background_instruments)
            if fixed_background_instruments is not None
            else None
        )
        self.hard_negative_mapping = active_hard_negative_mapping(cfg)
        if (
            cfg.use_hard_negative_backgrounds
            and self.hard_negative_mapping
            and hard_negative_label_store is None
            and background_mode == "sampled"
            and any(
                descriptor.source_type == "synthetic"
                for descriptor in self.descriptors
            )
        ):
            raise ValueError("Training hard negatives require a confuser label store.")
        self.sequence_steps = int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
        self.index_to_instrument = {index: name for name, index in label_store.class_to_index.items()}
        configured_target_off = {
            normalize_name(name)
            for name in cfg.target_off_hard_negative_instruments
        }
        self.target_off_hard_negative_indices = tuple(
            class_index
            for instrument, class_index in label_store.class_to_index.items()
            if normalize_name(instrument) in configured_target_off
        )

    def __len__(self) -> int:
        return len(self.descriptors)

    def set_epoch(self, epoch: int) -> None:
        self.current_epoch = int(epoch)

    def _uses_deterministic_synthetic_rng(self) -> bool:
        return bool(
            self.deterministic_synthetic
            or (
                self.cfg.deterministic_epoch_sampling
                and self.background_mode == "sampled"
            )
        )

    def _load_real_mix(self, descriptor: SequenceDescriptor) -> Tuple[np.ndarray, np.ndarray]:
        start_seconds = descriptor.start_step * self.cfg.label_hop_seconds
        audio = read_audio_segment(
            self.local_catalog.mixes[descriptor.song_id],
            start_seconds,
            self.cfg.sequence_seconds,
            self.local_catalog.sample_rate,
        )
        target = self.label_store.get_sequence(
            descriptor.song_id, descriptor.start_step, self.sequence_steps
        )
        return audio, target

    def _load_stem(self, descriptor: SequenceDescriptor) -> Tuple[np.ndarray, np.ndarray]:
        instrument = self.index_to_instrument[descriptor.instrument_index]
        path = self.local_catalog.stems[instrument][descriptor.song_id]
        start_seconds = descriptor.start_step * self.cfg.label_hop_seconds
        audio = read_audio_segment(
            path,
            start_seconds,
            self.cfg.sequence_seconds,
            self.local_catalog.sample_rate,
        )
        full_target = self.label_store.get_sequence(
            descriptor.song_id, descriptor.start_step, self.sequence_steps
        )
        target = np.zeros_like(full_target, dtype=np.float32)
        target[:, descriptor.instrument_index] = full_target[:, descriptor.instrument_index]
        return audio, target

    def _deterministic_synthetic_rng(
        self,
        descriptor: SequenceDescriptor,
        token: str,
    ) -> np.random.Generator:
        digest = int(
            hashlib.sha1(
                f"{descriptor.source_type}|{descriptor.song_id}|"
                f"{descriptor.start_step}|{descriptor.instrument_index}|{token}".encode(
                    "utf-8"
                )
            ).hexdigest()[:8],
            16,
        )
        if self.deterministic_synthetic:
            epoch_component = self.cfg.controlled_validation_seed_offset
        else:
            epoch_component = (self.current_epoch + 1) * 1_000_003
        seed = (self.cfg.seed + epoch_component + digest) % (2**32)
        return np.random.default_rng(seed)

    def _random_uniform(
        self,
        descriptor: SequenceDescriptor,
        token: str,
        minimum: float,
        maximum: float,
    ) -> float:
        if self._uses_deterministic_synthetic_rng():
            rng = self._deterministic_synthetic_rng(descriptor, token)
            return float(rng.uniform(minimum, maximum))
        return float(np.random.uniform(minimum, maximum))

    def _active_rms(
        self,
        audio: np.ndarray,
        active_steps: Optional[np.ndarray] = None,
    ) -> float:
        """RMS over active half-second frames."""
        hop_samples = int(round(
            self.cfg.label_hop_seconds * self.local_catalog.sample_rate
        ))
        if hop_samples <= 0 or len(audio) == 0:
            return 0.0
        frame_count = int(math.ceil(len(audio) / hop_samples))
        padded = np.pad(audio, (0, frame_count * hop_samples - len(audio)))
        frames = padded.reshape(frame_count, hop_samples)
        frame_rms = np.sqrt(np.mean(np.square(frames, dtype=np.float64), axis=1))
        if active_steps is None:
            threshold = 10.0 ** (self.cfg.activity_threshold_dbfs / 20.0)
            active = frame_rms > max(threshold, 1e-8)
        else:
            active = np.zeros(frame_count, dtype=bool)
            supplied = np.asarray(active_steps, dtype=bool).reshape(-1)
            active[: min(frame_count, len(supplied))] = supplied[:frame_count]
        if not np.any(active):
            return 0.0
        return float(np.sqrt(np.mean(np.square(frames[active], dtype=np.float64))))

    def _balance_target_active_rms(
        self,
        descriptor: SequenceDescriptor,
        class_index: int,
        audio: np.ndarray,
        active_steps: np.ndarray,
    ) -> np.ndarray:
        if not self.cfg.balance_synthetic_target_active_rms:
            return audio
        active_rms = self._active_rms(audio, active_steps)
        if active_rms <= 1e-8:
            return audio
        instrument = self.index_to_instrument[class_index]
        jitter_db = self._random_uniform(
            descriptor,
            f"target_active_rms_jitter_{safe_slug(instrument)}",
            self.cfg.synthetic_target_random_gain_db_min,
            self.cfg.synthetic_target_random_gain_db_max,
        )
        desired_dbfs = self.cfg.synthetic_target_reference_dbfs + jitter_db
        desired_rms = 10.0 ** (desired_dbfs / 20.0)
        requested_gain_db = 20.0 * math.log10(desired_rms / active_rms)
        applied_gain_db = float(np.clip(
            requested_gain_db,
            -self.cfg.synthetic_target_max_adjustment_db,
            self.cfg.synthetic_target_max_adjustment_db,
        ))
        gain = 10.0 ** (applied_gain_db / 20.0)
        return (audio * gain).astype(np.float32, copy=False)

    def _background_candidates(self, descriptor: SequenceDescriptor) -> List[str]:
        selected = {normalize_name(name) for name in self.label_store.instruments}
        fixed = (
            None
            if self.fixed_background_instruments is None
            else {normalize_name(name) for name in self.fixed_background_instruments}
        )
        return sorted(
            instrument
            for instrument, paths in self.local_catalog.stems.items()
            if normalize_name(instrument) not in selected
            and descriptor.song_id in paths
            and (fixed is None or normalize_name(instrument) in fixed)
        )

    def _eligible_hard_negative_backgrounds(
        self,
        descriptor: SequenceDescriptor,
        original_target: np.ndarray,
        candidates: Sequence[str],
    ) -> List[str]:
        if (
            not self.cfg.use_hard_negative_backgrounds
            or not self.hard_negative_mapping
            or self.hard_negative_label_store is None
        ):
            return []
        minimum_steps = max(
            1,
            int(math.ceil(
                self.cfg.hard_negative_min_overlap_seconds
                / self.cfg.label_hop_seconds
            )),
        )
        overlaps = hard_negative_overlap_steps(
            descriptor.song_id,
            descriptor.start_step,
            self.sequence_steps,
            self.label_store,
            self.hard_negative_label_store,
            self.hard_negative_mapping,
            target_sequence=original_target,
        )
        candidate_by_normalized = {
            normalize_name(instrument): instrument for instrument in candidates
        }
        eligible: List[str] = []
        seen: set[str] = set()
        for (_, configured_confuser), overlap_count in overlaps.items():
            if overlap_count < minimum_steps:
                continue
            normalized = normalize_name(configured_confuser)
            actual_instrument = candidate_by_normalized.get(normalized)
            if actual_instrument is not None and normalized not in seen:
                eligible.append(actual_instrument)
                seen.add(normalized)
        return eligible

    def _target_off_hard_negative_eligible(
        self,
        emitted_target: np.ndarray,
    ) -> bool:
        if (
            self.background_mode != "sampled"
            or not self.cfg.use_target_off_hard_negatives
            or not self.target_off_hard_negative_indices
        ):
            return False
        minimum_steps = max(
            1,
            int(math.ceil(
                self.cfg.target_off_hard_negative_min_seconds
                / self.cfg.label_hop_seconds
            )),
        )
        return any(
            longest_true_run(emitted_target[:, class_index] <= 0.5)
            >= minimum_steps
            for class_index in self.target_off_hard_negative_indices
        )

    def _choose_background_instruments(
        self,
        descriptor: SequenceDescriptor,
        original_target: np.ndarray,
        emitted_target: np.ndarray,
        included_target_count: int = 0,
    ) -> List[str]:
        if (
            self.background_mode == "none"
            or not self.cfg.use_synthetic_background_instruments
        ):
            return []
        candidates = self._background_candidates(descriptor)
        if self.background_mode == "all":
            return candidates

        dedicated_target_off = descriptor.source_type in TARGET_OFF_SOURCE_TYPES
        force_target_off_context = dedicated_target_off and (
            self._random_uniform(
                descriptor, "target_off_hard_negative_usage", 0.0, 1.0
            ) < self.cfg.target_off_hard_negative_probability
        )
        regular_background = self._random_uniform(
            descriptor, "background_usage", 0.0, 1.0
        ) < self.cfg.synthetic_background_probability
        if not (force_target_off_context or regular_background):
            return []

        minimum = max(
            self.cfg.synthetic_background_min_instruments,
            self.cfg.synthetic_min_total_instruments - int(included_target_count),
        )
        minimum = max(1, minimum)
        if len(candidates) < minimum:
            raise RuntimeError(
                f"Song {descriptor.song_id} has {included_target_count} included target stems "
                f"and only {len(candidates)} background stems; at least {minimum} backgrounds "
                f"are required to reach {self.cfg.synthetic_min_total_instruments} total instruments."
            )
        configured_maximum = self.cfg.synthetic_background_max_instruments
        maximum = len(candidates) if configured_maximum is None else min(
            len(candidates), configured_maximum
        )

        eligible_hard_negatives = self._eligible_hard_negative_backgrounds(
            descriptor, original_target, candidates
        )
        force_hard_negative = bool(eligible_hard_negatives) and (
            self._random_uniform(
                descriptor, "hard_negative_usage", 0.0, 1.0
            ) < self.cfg.hard_negative_probability
        )
        forced = eligible_hard_negatives if force_hard_negative else []
        if len(forced) > maximum:
            raise RuntimeError(
                "More hard-negative confusers were selected than the configured "
                "background maximum."
            )

        if self._uses_deterministic_synthetic_rng():
            rng = self._deterministic_synthetic_rng(
                descriptor, "background_selection"
            )
            count = int(rng.integers(max(minimum, len(forced)), maximum + 1))
        else:
            count = int(np.random.randint(max(minimum, len(forced)), maximum + 1))

        forced_normalized = {normalize_name(value) for value in forced}
        remaining = [
            value for value in candidates
            if normalize_name(value) not in forced_normalized
        ]
        extra_count = count - len(forced)
        if extra_count > 0:
            if self._uses_deterministic_synthetic_rng():
                extras = rng.choice(
                    np.asarray(remaining, dtype=object), size=extra_count, replace=False
                )
            else:
                extras = np.random.choice(
                    np.asarray(remaining, dtype=object), size=extra_count, replace=False
                )
            return forced + [str(value) for value in extras]
        return list(forced)

    def _load_synthetic_mix(self, descriptor: SequenceDescriptor) -> Tuple[np.ndarray, np.ndarray]:
        start_seconds = descriptor.start_step * self.cfg.label_hop_seconds
        original_target = self.label_store.get_sequence(
            descriptor.song_id, descriptor.start_step, self.sequence_steps
        )
        if self.dropout_plan is None:
            full_target = original_target
        else:
            full_target = self.dropout_plan.get_target_sequence(
                descriptor.song_id, descriptor.start_step, self.sequence_steps
            )

        candidates: List[int] = []
        for instrument, class_index in self.label_store.class_to_index.items():
            if descriptor.song_id not in self.local_catalog.stems[instrument]:
                continue
            if np.any(original_target[:, class_index] > 0.5):
                candidates.append(class_index)

        if not candidates and self.fallback_to_real_mix_when_no_target:
            return self._load_real_mix(
                SequenceDescriptor("real_mix", descriptor.song_id, descriptor.start_step)
            )

        if self.dropout_plan is not None:
            included = candidates
        else:
            if self._uses_deterministic_synthetic_rng():
                inclusion_rng = self._deterministic_synthetic_rng(
                    descriptor, "target_inclusion"
                )
                included = [
                    index
                    for index in candidates
                    if inclusion_rng.random() < self.cfg.synthetic_inclusion_probability
                ]
                if not included and candidates:
                    included = [int(inclusion_rng.choice(candidates))]
            else:
                included = [
                    index
                    for index in candidates
                    if np.random.random() < self.cfg.synthetic_inclusion_probability
                ]
                if not included and candidates:
                    included = [int(np.random.choice(candidates))]

        segment_samples = int(round(self.cfg.sequence_seconds * self.local_catalog.sample_rate))
        hop_samples = int(round(self.cfg.label_hop_seconds * self.local_catalog.sample_rate))
        fade_samples = int(round(self.cfg.counterfactual_fade_seconds * self.local_catalog.sample_rate))
        target_bus = np.zeros(segment_samples, dtype=np.float32)
        target_reference_bus = np.zeros(segment_samples, dtype=np.float32)
        target = np.zeros_like(full_target, dtype=np.float32)

        keep_steps = None
        previous_keep = None
        if self.dropout_plan is not None:
            keep_steps = self.dropout_plan.get_keep_sequence(
                descriptor.song_id, descriptor.start_step, self.sequence_steps
            )
            previous_keep = self.dropout_plan.previous_keep(descriptor.song_id, descriptor.start_step)

        for class_index in included:
            instrument = self.index_to_instrument[class_index]
            raw_stem = read_audio_segment(
                self.local_catalog.stems[instrument][descriptor.song_id],
                start_seconds,
                self.cfg.sequence_seconds,
                self.local_catalog.sample_rate,
            )
            # Normalize every target independently over the frames in which
            # that instrument is active. This removes recording-level identity and
            # prevents loud drums from erasing quieter harmonic instruments.
            balanced_stem = self._balance_target_active_rms(
                descriptor,
                class_index,
                raw_stem,
                original_target[:, class_index] > 0.5,
            )
            # The reference bus is never muted and keeps background loudness
            # independent of counterfactual dropout state.
            target_reference_bus += balanced_stem
            stem = balanced_stem
            if keep_steps is not None and previous_keep is not None:
                envelope = _audio_keep_envelope(
                    keep_steps[:, class_index],
                    bool(previous_keep[class_index]),
                    segment_samples,
                    hop_samples,
                    fade_samples,
                )
                stem = stem * envelope

            target[:, class_index] = full_target[:, class_index]
            target_bus += stem

        # Preserve the recorded relative levels of non-target stems. They are
        # summed first and only the complete background bus is scaled.
        background_bus = np.zeros(segment_samples, dtype=np.float32)
        for instrument in self._choose_background_instruments(
            descriptor,
            original_target,
            target,
            included_target_count=len(included),
        ):
            background_bus += read_audio_segment(
                self.local_catalog.stems[instrument][descriptor.song_id],
                start_seconds,
                self.cfg.sequence_seconds,
                self.local_catalog.sample_rate,
            )

        if np.any(background_bus):
            reference_active = np.any(original_target > 0.5, axis=1)
            target_reference_rms = self._active_rms(
                target_reference_bus, reference_active
            )
            background_rms = self._active_rms(background_bus)
            if target_reference_rms > 1e-8 and background_rms > 1e-8:
                relative_db = self._random_uniform(
                    descriptor,
                    "background_relative_db",
                    self.cfg.synthetic_background_relative_db_min,
                    self.cfg.synthetic_background_relative_db_max,
                )
                desired_background_rms = (
                    target_reference_rms * 10.0 ** (relative_db / 20.0)
                )
                background_bus *= desired_background_rms / background_rms

        synthetic = target_bus + background_bus
        peak = float(np.max(np.abs(synthetic))) if len(synthetic) else 0.0
        if peak > self.cfg.synthetic_peak_limit > 0:
            synthetic *= self.cfg.synthetic_peak_limit / peak

        return synthetic.astype(np.float32, copy=False), target

    def __getitem__(self, index: int) -> Dict[str, Any]:
        descriptor = self.descriptors[index]
        if descriptor.source_type == "real_mix":
            audio, target = self._load_real_mix(descriptor)
        elif descriptor.source_type == "stem":
            audio, target = self._load_stem(descriptor)
        elif descriptor.source_type in SYNTHETIC_SOURCE_TYPES:
            audio, target = self._load_synthetic_mix(descriptor)
        else:
            raise ValueError(f"Unknown source type: {descriptor.source_type}")

        loss_mask = descriptor_expected_loss_mask(
            descriptor, self.label_store, self.cfg
        )
        return {
            "audio": torch.from_numpy(audio),
            "target": torch.from_numpy(target.astype(np.float32, copy=False)),
            "loss_mask": torch.from_numpy(loss_mask),
            # A float tensor preserves decimal IDs through default DataLoader collation.
            "song_id": float(descriptor.song_id),
            "start_step": int(descriptor.start_step),
            "source_type": descriptor.source_type,
        }


def normalized_source_ratios(
    descriptors: Sequence[SequenceDescriptor],
    cfg: ExperimentConfig,
) -> Tuple[Dict[str, float], Counter]:
    """Return the exact source distribution used by the weighted sampler."""
    counts = Counter(descriptor.source_type for descriptor in descriptors)
    if cfg.use_synthetic_mix:
        requested = {
            "real_mix": cfg.real_mix_ratio_with_synthetic if cfg.use_real_mix else 0.0,
            "stem": cfg.stem_ratio_with_synthetic if cfg.use_stems else 0.0,
        }
        if cfg.use_target_off_hard_negatives:
            requested.update(
                {
                    SYNTHETIC_REGULAR: cfg.target_off_regular_synthetic_ratio,
                    SYNTHETIC_TARGET_OFF_FULL: cfg.target_off_fully_off_ratio,
                    SYNTHETIC_TARGET_OFF_PARTIAL: cfg.target_off_partially_off_ratio,
                }
            )
        else:
            requested[SYNTHETIC_REGULAR] = cfg.synthetic_mix_ratio
    else:
        requested = {
            "real_mix": cfg.real_mix_ratio_no_synthetic if cfg.use_real_mix else 0.0,
            "stem": cfg.stem_ratio_no_synthetic if cfg.use_stems else 0.0,
            "synthetic": 0.0,
        }
    requested = {
        source: weight
        for source, weight in requested.items()
        if weight > 0 and counts.get(source, 0) > 0
    }
    if not requested:
        raise RuntimeError("No enabled source type has any descriptors.")
    ratio_sum = sum(requested.values())
    return (
        {source: weight / ratio_sum for source, weight in requested.items()},
        counts,
    )


def _exact_integer_quotas(
    weights: Mapping[str, float],
    total: int,
) -> Dict[str, int]:
    """Largest-remainder allocation whose integer quotas sum exactly to total."""
    if total < 0:
        raise ValueError("total must be non-negative")
    positive = {key: float(value) for key, value in weights.items() if value > 0}
    if not positive:
        return {}
    weight_sum = sum(positive.values())
    raw = {key: total * value / weight_sum for key, value in positive.items()}
    quotas = {key: int(math.floor(value)) for key, value in raw.items()}
    remainder = total - sum(quotas.values())
    ranked = sorted(
        positive,
        key=lambda key: (-(raw[key] - quotas[key]), key),
    )
    for key in ranked[:remainder]:
        quotas[key] += 1
    return quotas


class DeterministicSourceSongQuotaSampler(Sampler[int]):
    """Exact source quotas with deterministic song-balanced sampling.

    Real and synthetic sources are balanced across songs. Isolated-stem
    descriptors are balanced across output instruments and then across songs.
    ON/OFF balance is deliberately handled by the frame-wise loss, not by
    selecting whole recordings as class-specific OFF examples.
    """

    def __init__(
        self,
        descriptors: Sequence[SequenceDescriptor],
        class_names: Sequence[str],
        source_ratios: Mapping[str, float],
        num_samples: int,
        seed: int,
        isolated_positive_samples_per_instrument: Optional[int] = None,
    ) -> None:
        self.descriptors = list(descriptors)
        self.class_names = tuple(class_names)
        self.source_ratios = dict(source_ratios)
        self.base_num_samples = int(num_samples)
        self.seed = int(seed)
        self.epoch = 0
        self.source_quotas = _exact_integer_quotas(
            self.source_ratios, self.base_num_samples
        )

        self.song_pools: Dict[Tuple[str, SongId], List[int]] = defaultdict(list)
        self.stem_pools: Dict[Tuple[int, SongId], List[int]] = defaultdict(list)
        for index, descriptor in enumerate(self.descriptors):
            if descriptor.source_type == "stem":
                self.stem_pools[(descriptor.instrument_index, descriptor.song_id)].append(index)
            elif descriptor.source_type in self.source_quotas:
                self.song_pools[(descriptor.source_type, descriptor.song_id)].append(index)

        if isolated_positive_samples_per_instrument is not None:
            available_classes = {class_index for class_index, _ in self.stem_pools}
            missing_classes = [
                self.class_names[class_index]
                for class_index in range(len(self.class_names))
                if class_index not in available_classes
            ]
            if missing_classes:
                raise RuntimeError(
                    "No isolated positive descriptors are available for: "
                    + ", ".join(missing_classes)
                )
            required_stems = (
                int(isolated_positive_samples_per_instrument) * len(self.class_names)
            )
            self.source_quotas["stem"] = max(
                self.source_quotas.get("stem", 0), required_stems
            )
        self.num_samples = sum(self.source_quotas.values())

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __len__(self) -> int:
        return self.num_samples

    def _rng(self, token: str) -> np.random.Generator:
        digest = int(hashlib.sha1(token.encode("utf-8")).hexdigest()[:8], 16)
        epoch_seed = (
            self.seed + (self.epoch + 1) * 1_000_003 + digest
        ) % (2**32)
        return np.random.default_rng(epoch_seed)

    def _sample_pool(
        self,
        indices: Sequence[int],
        count: int,
        token: str,
    ) -> List[int]:
        if count <= 0:
            return []
        if not indices:
            raise RuntimeError(f"Empty sampler pool for requested quota: {token}")
        rng = self._rng(token)
        selected: List[int] = []
        values = np.asarray(indices, dtype=np.int64)
        while len(selected) < count:
            selected.extend(int(value) for value in rng.permutation(values).tolist())
        return selected[:count]

    def quota_rows(self) -> List[Dict[str, Any]]:
        rows: List[Dict[str, Any]] = []
        for source_type, source_quota in self.source_quotas.items():
            if source_type == "stem":
                available_classes = sorted(
                    {class_index for class_index, _ in self.stem_pools}
                )
                if not available_classes:
                    raise RuntimeError("No isolated-stem pools are available.")
                class_quotas = _exact_integer_quotas(
                    {str(index): 1.0 for index in available_classes}, source_quota
                )
                for class_token, class_quota in class_quotas.items():
                    class_index = int(class_token)
                    songs = sorted(
                        {
                            song_id
                            for candidate_class, song_id in self.stem_pools
                            if candidate_class == class_index
                        },
                        key=song_id_sort_key,
                    )
                    song_quotas = _exact_integer_quotas(
                        {str(song_id): 1.0 for song_id in songs}, class_quota
                    )
                    by_token = {str(song_id): song_id for song_id in songs}
                    for song_token, quota in song_quotas.items():
                        song_id = by_token[song_token]
                        pool = self.stem_pools[(class_index, song_id)]
                        rows.append(
                            {
                                "source_type": source_type,
                                "scope": "instrument_song",
                                "instrument": self.class_names[class_index],
                                "song_id": song_id,
                                "quota": quota,
                                "pool_size": len(pool),
                            }
                        )
            else:
                songs = sorted(
                    {
                        song_id
                        for candidate_source, song_id in self.song_pools
                        if candidate_source == source_type
                    },
                    key=song_id_sort_key,
                )
                if not songs:
                    raise RuntimeError(f"No song pools exist for source {source_type}.")
                song_quotas = _exact_integer_quotas(
                    {str(song_id): 1.0 for song_id in songs}, source_quota
                )
                by_token = {str(song_id): song_id for song_id in songs}
                for song_token, quota in song_quotas.items():
                    song_id = by_token[song_token]
                    pool = self.song_pools[(source_type, song_id)]
                    rows.append(
                        {
                            "source_type": source_type,
                            "scope": "song",
                            "instrument": "ALL",
                            "song_id": song_id,
                            "quota": quota,
                            "pool_size": len(pool),
                        }
                    )
        return rows

    def __iter__(self) -> Iterable[int]:
        selected: List[int] = []
        for row in self.quota_rows():
            source_type = str(row["source_type"])
            song_id = canonical_song_id(row["song_id"])
            if source_type == "stem":
                class_index = self.class_names.index(str(row["instrument"]))
                pool = self.stem_pools[(class_index, song_id)]
            else:
                pool = self.song_pools[(source_type, song_id)]
            selected.extend(
                self._sample_pool(
                    pool,
                    int(row["quota"]),
                    f"{source_type}|{row['instrument']}|{song_id}",
                )
            )
        if len(selected) != self.num_samples:
            raise RuntimeError(
                f"Sampler created {len(selected)} indices; expected {self.num_samples}."
            )
        self._rng("complete_epoch_order").shuffle(selected)
        return iter(selected)


def build_epoch_quota_sampler(
    train_dataset: InstrumentSequenceDataset,
    cfg: ExperimentConfig,
) -> DeterministicSourceSongQuotaSampler:
    descriptors = train_dataset.descriptors
    normalized, counts = normalized_source_ratios(descriptors, cfg)
    num_samples = int(cfg.train_samples_per_epoch or len(descriptors))
    sampler = DeterministicSourceSongQuotaSampler(
        descriptors=descriptors,
        class_names=train_dataset.label_store.instruments,
        source_ratios=normalized,
        num_samples=num_samples,
        seed=cfg.seed,
        isolated_positive_samples_per_instrument=(
            cfg.isolated_positive_samples_per_instrument
        ),
    )

    print("\nExact source/song-balanced quotas per training epoch:")
    print(f"  baseline samples/epoch: {num_samples}")
    if sampler.num_samples != num_samples:
        print(
            f"  actual samples/epoch:   {sampler.num_samples} "
            "(enlarged to satisfy the per-instrument isolated-positive floor)"
        )
    for source, quota in sampler.source_quotas.items():
        print(
            f"  {source:28s}: quota={quota:4d}, pool={counts[source]:4d}, "
            f"share={quota / sampler.num_samples:.3f}"
        )
    quota_frame = pd.DataFrame(sampler.quota_rows())
    print("\nPer-song quotas (isolated stems have an exact positive quota per instrument):")
    print(quota_frame.to_string(index=False))
    print("epoch seed rule: CFG.seed + (epoch + 1) * 1,000,003")
    return sampler


def descriptor_expected_target(
    descriptor: SequenceDescriptor,
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    dropout_plan: Optional[CounterfactualDropoutPlan],
    cfg: ExperimentConfig,
) -> np.ndarray:
    """Expected labels emitted for one sampled descriptor, without loading audio."""
    sequence_steps = int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
    original = label_store.get_sequence(
        descriptor.song_id, descriptor.start_step, sequence_steps
    )
    if descriptor.source_type == "real_mix":
        return original
    if descriptor.source_type == "stem":
        target = np.zeros_like(original, dtype=np.float32)
        target[:, descriptor.instrument_index] = original[:, descriptor.instrument_index]
        return target
    if descriptor.source_type not in SYNTHETIC_SOURCE_TYPES:
        raise ValueError(f"Unknown source type: {descriptor.source_type}")

    available = np.asarray(
        [
            descriptor.song_id in local_catalog.stems[instrument]
            for instrument in label_store.instruments
        ],
        dtype=np.float32,
    )
    if dropout_plan is not None:
        planned = dropout_plan.get_target_sequence(
            descriptor.song_id, descriptor.start_step, sequence_steps
        )
        return planned * available[None, :]

    # Legacy random inclusion mode: use its expected target. The default
    # counterfactual configuration above uses a deterministic dropout plan.
    return original * available[None, :] * cfg.synthetic_inclusion_probability


def descriptor_expected_loss_mask(
    descriptor: SequenceDescriptor,
    label_store: LabelStore,
    cfg: ExperimentConfig,
) -> np.ndarray:
    sequence_steps = int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
    mask = np.ones((sequence_steps, len(label_store.instruments)), dtype=np.float32)
    if descriptor.source_type == "stem" and cfg.isolated_stem_target_only_loss:
        mask.fill(0.0)
        mask[:, descriptor.instrument_index] = 1.0
    return mask


def add_target_off_training_descriptors(
    descriptors: Sequence[SequenceDescriptor],
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    dropout_plan: Optional[CounterfactualDropoutPlan],
    cfg: ExperimentConfig,
) -> List[SequenceDescriptor]:
    """Add dedicated full-OFF and partial-OFF views of eligible synthetic windows."""
    base_descriptors = list(descriptors)
    if not cfg.use_target_off_hard_negatives:
        return base_descriptors

    selected_by_normalized = {
        normalize_name(name): name for name in label_store.instruments
    }
    target_indices = tuple(
        label_store.class_to_index[
            selected_by_normalized[normalize_name(requested)]
        ]
        for requested in cfg.target_off_hard_negative_instruments
    )
    minimum_run_steps = max(
        1,
        int(round(
            cfg.target_off_hard_negative_min_seconds / cfg.label_hop_seconds
        )),
    )
    fully_off: List[SequenceDescriptor] = []
    partially_off: List[SequenceDescriptor] = []

    for descriptor in base_descriptors:
        if descriptor.source_type != SYNTHETIC_REGULAR:
            continue
        target = descriptor_expected_target(
            descriptor, local_catalog, label_store, dropout_plan, cfg
        )
        off_masks = [
            target[:, class_index] <= 0.5 for class_index in target_indices
        ]
        is_fully_off = any(bool(np.all(mask)) for mask in off_masks)
        has_partial_long_off = any(
            bool(np.any(~mask)) and longest_true_run(mask) >= minimum_run_steps
            for mask in off_masks
        )
        if is_fully_off:
            fully_off.append(
                SequenceDescriptor(
                    SYNTHETIC_TARGET_OFF_FULL,
                    descriptor.song_id,
                    descriptor.start_step,
                )
            )
        elif has_partial_long_off:
            partially_off.append(
                SequenceDescriptor(
                    SYNTHETIC_TARGET_OFF_PARTIAL,
                    descriptor.song_id,
                    descriptor.start_step,
                )
            )

    if not fully_off:
        raise RuntimeError(
            "No fully target-OFF synthetic sequences were found for the 10% pool."
        )
    if not partially_off:
        raise RuntimeError(
            "No partially target-OFF sequences with the required contiguous "
            "duration were found for the 10% pool."
        )

    print("\nDedicated target-OFF descriptor pools:")
    print(f"  regular synthetic:    {sum(d.source_type == SYNTHETIC_REGULAR for d in base_descriptors)}")
    print(f"  fully target-OFF:     {len(fully_off)}")
    print(f"  partially target-OFF: {len(partially_off)}")
    print(
        "  requested epoch shares: "
        f"{cfg.target_off_regular_synthetic_ratio:.0%} regular, "
        f"{cfg.target_off_fully_off_ratio:.0%} fully OFF, "
        f"{cfg.target_off_partially_off_ratio:.0%} partially OFF"
    )
    return base_descriptors + fully_off + partially_off


def summarize_target_off_hard_negative_eligibility(
    descriptors: Sequence[SequenceDescriptor],
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    dropout_plan: Optional[CounterfactualDropoutPlan],
    cfg: ExperimentConfig,
) -> pd.DataFrame:
    """Report how many synthetic sequences can teach a target-specific OFF state."""
    if not cfg.use_target_off_hard_negatives:
        return pd.DataFrame()

    minimum_steps = max(
        1,
        int(math.ceil(
            cfg.target_off_hard_negative_min_seconds / cfg.label_hop_seconds
        )),
    )
    selected_by_normalized = {
        normalize_name(name): name for name in label_store.instruments
    }
    synthetic_descriptors = [
        descriptor
        for descriptor in descriptors
        if descriptor.source_type == "synthetic"
    ]
    contextual_probability_when_eligible = 1.0 - (
        (1.0 - cfg.target_off_hard_negative_probability)
        * (1.0 - cfg.synthetic_background_probability)
    )
    rows: List[Dict[str, Any]] = []
    for requested in cfg.target_off_hard_negative_instruments:
        instrument = selected_by_normalized[normalize_name(requested)]
        class_index = label_store.class_to_index[instrument]
        eligible_sequences = 0
        fully_off_sequences = 0
        partially_off_sequences = 0
        always_on_sequences = 0
        off_steps = 0

        for descriptor in synthetic_descriptors:
            target = descriptor_expected_target(
                descriptor, local_catalog, label_store, dropout_plan, cfg
            )
            off_mask = target[:, class_index] <= 0.5
            instrument_off_steps = int(np.count_nonzero(off_mask))
            contiguous_off_steps = longest_true_run(off_mask)
            off_steps += instrument_off_steps
            if instrument_off_steps == len(target):
                fully_off_sequences += 1
            elif instrument_off_steps > 0:
                partially_off_sequences += 1
            else:
                always_on_sequences += 1
            if contiguous_off_steps >= minimum_steps:
                eligible_sequences += 1

        rows.append(
            {
                "instrument": instrument,
                "synthetic_sequences": len(synthetic_descriptors),
                "eligible_sequences": eligible_sequences,
                "eligible_percent": (
                    100.0 * eligible_sequences / len(synthetic_descriptors)
                    if synthetic_descriptors else 0.0
                ),
                "fully_OFF_sequences": fully_off_sequences,
                "partially_OFF_sequences": partially_off_sequences,
                "always_ON_sequences": always_on_sequences,
                "OFF_steps_0p5s": off_steps,
                "OFF_seconds": off_steps * cfg.label_hop_seconds,
                "force_probability": cfg.target_off_hard_negative_probability,
                "context_probability_when_eligible": contextual_probability_when_eligible,
            }
        )

    frame = pd.DataFrame(rows)
    print("\nTarget-OFF hard-negative eligibility after dropout:")
    print(frame.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    missing = frame.loc[frame["eligible_sequences"] <= 0, "instrument"].tolist()
    if missing:
        raise RuntimeError(
            "No eligible synthetic target-OFF sequences for: " + ", ".join(missing)
        )
    return frame


def calculate_class_balance_weights(
    descriptors: Sequence[SequenceDescriptor],
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    dropout_plan: Optional[CounterfactualDropoutPlan],
    cfg: ExperimentConfig,
    sampler: Optional[Sampler[int]] = None,
) -> Tuple[torch.Tensor, torch.Tensor, pd.DataFrame]:
    """
    Balance ON and OFF per instrument using the distribution the sampler emits.

    This is important because a 40% real-mix source share does not imply 40%
    positives: an instrument may be ON throughout nearly every real-mix window.
    """
    expected_on = np.zeros(len(label_store.instruments), dtype=np.float64)
    expected_steps = np.zeros(len(label_store.instruments), dtype=np.float64)

    if sampler is not None:
        selected_indices = [int(index) for index in iter(sampler)]
        weighted_descriptors = [
            (descriptors[index], 1.0 / max(1, len(selected_indices)))
            for index in selected_indices
        ]
        balance_label = "exact source/song-balanced epoch-1 sampler"
    else:
        normalized, counts = normalized_source_ratios(descriptors, cfg)
        weighted_descriptors = [
            (
                descriptor,
                normalized[descriptor.source_type] / counts[descriptor.source_type],
            )
            for descriptor in descriptors
            if descriptor.source_type in normalized
        ]
        balance_label = "source-only expectation"

    for descriptor, descriptor_probability in weighted_descriptors:
        target = descriptor_expected_target(
            descriptor, local_catalog, label_store, dropout_plan, cfg
        )
        loss_mask = descriptor_expected_loss_mask(descriptor, label_store, cfg)
        expected_on += descriptor_probability * (target * loss_mask).sum(axis=0)
        expected_steps += descriptor_probability * loss_mask.sum(axis=0)

    if np.any(expected_steps <= 0):
        missing = [
            label_store.instruments[index]
            for index, value in enumerate(expected_steps)
            if value <= 0
        ]
        raise RuntimeError(
            "Could not estimate sampled supervised labels for: " + ", ".join(missing)
        )
    on_fraction = expected_on / expected_steps
    off_fraction = 1.0 - on_fraction
    epsilon = 1e-8

    missing_on = [
        label_store.instruments[index]
        for index, value in enumerate(on_fraction)
        if value <= epsilon
    ]
    missing_off = [
        label_store.instruments[index]
        for index, value in enumerate(off_fraction)
        if value <= epsilon
    ]
    if missing_on:
        raise RuntimeError("No expected ON training labels for: " + ", ".join(missing_on))
    if missing_off:
        raise RuntimeError(
            "No expected OFF training labels for: "
            + ", ".join(missing_off)
            + ". Increase counterfactual dropout or add inactive material."
        )

    if cfg.use_class_balance_weights:
        on_weight = np.minimum(
            0.5 / np.maximum(on_fraction, epsilon), cfg.max_class_balance_weight
        )
        off_weight = np.minimum(
            0.5 / np.maximum(off_fraction, epsilon), cfg.max_class_balance_weight
        )
    else:
        on_weight = np.ones_like(on_fraction)
        off_weight = np.ones_like(off_fraction)

    frame = pd.DataFrame(
        {
            "instrument": label_store.instruments,
            "expected_ON_fraction": on_fraction,
            "expected_OFF_fraction": off_fraction,
            "ON_loss_weight": on_weight,
            "OFF_loss_weight": off_weight,
        }
    )
    print(f"\nExpected training balance from {balance_label}:")
    print(frame.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    return (
        torch.tensor(on_weight, dtype=torch.float32),
        torch.tensor(off_weight, dtype=torch.float32),
        frame,
    )




In [ ]:
from __future__ import annotations

# =============================================================================
# PANNs CNN6 LINEAR SPECTROGRAM — FRAMEWISE, NO LSTM
# =============================================================================


def init_layer(layer: nn.Module) -> None:
    if hasattr(layer, "weight") and getattr(layer, "weight") is not None:
        nn.init.xavier_uniform_(getattr(layer, "weight"))
    if hasattr(layer, "bias") and getattr(layer, "bias") is not None:
        nn.init.zeros_(getattr(layer, "bias"))


def init_bn(layer: nn.Module) -> None:
    if hasattr(layer, "bias") and getattr(layer, "bias") is not None:
        nn.init.zeros_(getattr(layer, "bias"))
    if hasattr(layer, "weight") and getattr(layer, "weight") is not None:
        nn.init.ones_(getattr(layer, "weight"))


class ConvBlock5x5(nn.Module):
    """The original one-convolution PANNs CNN6 block."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=(5, 5),
            stride=(1, 1),
            padding=(2, 2),
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        init_layer(self.conv1)
        init_bn(self.bn1)

    def forward(
        self,
        x: torch.Tensor,
        pool_size: Tuple[int, int] = (2, 2),
        pool_type: str = "avg",
    ) -> torch.Tensor:
        x = F.relu_(self.bn1(self.conv1(x)))
        if pool_type == "avg":
            return F.avg_pool2d(x, kernel_size=pool_size)
        if pool_type == "max":
            return F.max_pool2d(x, kernel_size=pool_size)
        if pool_type == "avg+max":
            return F.avg_pool2d(x, pool_size) + F.max_pool2d(x, pool_size)
        raise ValueError(f"Unsupported pool_type={pool_type!r}")


class LinearSpecAugment(nn.Module):
    def __init__(
        self,
        time_mask_width: int,
        time_mask_count: int,
        freq_mask_width: int,
        freq_mask_count: int,
    ) -> None:
        super().__init__()
        self.time_mask_width = time_mask_width
        self.time_mask_count = time_mask_count
        self.freq_mask_width = freq_mask_width
        self.freq_mask_count = freq_mask_count

    @staticmethod
    def _mask_axis(x: torch.Tensor, axis: int, max_width: int, count: int) -> torch.Tensor:
        axis_length = x.shape[axis]
        if max_width <= 0 or count <= 0 or axis_length <= 1:
            return x
        for _ in range(count):
            width = int(torch.randint(0, min(max_width, axis_length - 1) + 1, (1,), device=x.device))
            if width <= 0:
                continue
            start = int(torch.randint(0, axis_length - width + 1, (1,), device=x.device))
            slices = [slice(None)] * x.ndim
            slices[axis] = slice(start, start + width)
            x[tuple(slices)] = 0.0
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        x = self._mask_axis(x, axis=2, max_width=self.time_mask_width, count=self.time_mask_count)
        x = self._mask_axis(x, axis=3, max_width=self.freq_mask_width, count=self.freq_mask_count)
        return x


class Cnn6LinearSpecFramewise(nn.Module):
    """Historical Cnn6LinearSpec adapted to framewise multi-label output.

    Everything before the output head matches the old isolated-instrument CNN6:
    linear log-power STFT, one BatchNorm channel per frequency bin, four 5x5
    convolutional blocks with BatchNorm and 2x2 average pooling, global frequency
    averaging, max+mean temporal aggregation, and a 512-unit fully connected layer.
    The sole adaptation is that temporal max+mean pooling is performed separately
    inside every requested output bin instead of once across the whole clip.
    """

    def __init__(
        self,
        sample_rate: int,
        window_size: int,
        hop_size: int,
        classes_num: int,
        output_steps: int,
        cfg: ExperimentConfig,
    ) -> None:
        super().__init__()
        self.sample_rate = int(sample_rate)
        self.window_size = int(window_size)
        self.hop_size = int(hop_size)
        self.freq_bins = self.window_size // 2 + 1
        self.output_steps = int(output_steps)
        self.log_epsilon = float(cfg.log_epsilon)
        self.stft_center = bool(cfg.stft_center)
        self.register_buffer(
            "hann_window", torch.hann_window(self.window_size), persistent=False
        )

        # Exact old input normalization: [B, F, T, 1] and one BN channel per
        # physical linear-frequency bin. This intentionally makes 44/96 kHz
        # models separate architectures, as they were in the historical runs.
        self.bn0 = nn.BatchNorm2d(self.freq_bins)
        init_bn(self.bn0)

        self.spec_augmenter = LinearSpecAugment(
            cfg.time_mask_width,
            cfg.time_mask_count,
            cfg.freq_mask_width,
            cfg.freq_mask_count,
        )
        self.use_spec_augment = bool(cfg.use_spec_augment)

        self.conv_block1 = ConvBlock5x5(1, 64)
        self.conv_block2 = ConvBlock5x5(64, 128)
        self.conv_block3 = ConvBlock5x5(128, 256)
        self.conv_block4 = ConvBlock5x5(256, 512)
        self.fc1 = nn.Linear(512, cfg.embedding_size, bias=True)
        self.fc_out = nn.Linear(cfg.embedding_size, classes_num, bias=True)
        init_layer(self.fc1)
        init_layer(self.fc_out)

        self.cnn_dropout = float(cfg.cnn_dropout)
        self.embedding_dropout = float(cfg.embedding_dropout)

    def extract_log_linear_spectrogram(self, waveform: torch.Tensor) -> torch.Tensor:
        # Equivalent settings to the frozen torchlibrosa Spectrogram used by the
        # old notebook, while retaining the current dependency-free torch.stft.
        device_type = waveform.device.type
        with torch.autocast(device_type=device_type, enabled=False):
            complex_spec = torch.stft(
                waveform.float(),
                n_fft=self.window_size,
                hop_length=self.hop_size,
                win_length=self.window_size,
                window=self.hann_window.float(),
                center=self.stft_center,
                pad_mode="reflect",
                normalized=False,
                onesided=True,
                return_complex=True,
            )  # [B, F, T]
            power = complex_spec.abs().square()
            x = torch.log(power + self.log_epsilon)
            x = x.transpose(1, 2).unsqueeze(1)  # [B, 1, T, F]
        return x

    def encoder_modules(self) -> Tuple[nn.Module, ...]:
        return (
            self.bn0,
            self.conv_block1,
            self.conv_block2,
            self.conv_block3,
            self.conv_block4,
            self.fc1,
        )

    def set_encoder_trainable(self, trainable: bool) -> None:
        for module in self.encoder_modules():
            for parameter in module.parameters():
                parameter.requires_grad = trainable

    def keep_frozen_encoder_in_eval_mode(self) -> None:
        for module in self.encoder_modules():
            module.eval()

    def _extract_framewise_hidden(
        self,
        waveform: torch.Tensor,
        output_steps: Optional[int] = None,
    ) -> torch.Tensor:
        x = self.extract_log_linear_spectrogram(waveform)

        # Old per-frequency BatchNorm convention.
        x = x.transpose(1, 3)  # [B, F, T, 1]
        x = self.bn0(x)
        x = x.transpose(1, 3)  # [B, 1, T, F]
        if self.training and self.use_spec_augment:
            x = self.spec_augmenter(x)

        x = self.conv_block1(x, pool_size=(2, 2), pool_type="avg")
        x = F.dropout(x, p=self.cnn_dropout, training=self.training)
        x = self.conv_block2(x, pool_size=(2, 2), pool_type="avg")
        x = F.dropout(x, p=self.cnn_dropout, training=self.training)
        x = self.conv_block3(x, pool_size=(2, 2), pool_type="avg")
        x = F.dropout(x, p=self.cnn_dropout, training=self.training)
        x = self.conv_block4(x, pool_size=(2, 2), pool_type="avg")
        x = F.dropout(x, p=self.cnn_dropout, training=self.training)

        x = torch.mean(x, dim=3)  # [B, 512, T_cnn], old global frequency mean
        pooled_steps = self.output_steps if output_steps is None else int(output_steps)

        # Exact old x_max + x_mean idea, applied locally in every half-second
        # output bin so temporal localization is retained.
        x_max = F.adaptive_max_pool1d(x, output_size=pooled_steps)
        x_mean = F.adaptive_avg_pool1d(x, output_size=pooled_steps)
        x = x_max + x_mean
        x = x.transpose(1, 2)  # [B, steps, 512]

        # The old model drops the pooled representation before fc1.
        x = F.dropout(x, p=self.embedding_dropout, training=self.training)
        return F.relu_(self.fc1(x))

    def extract_cnn_embeddings(
        self,
        waveform: torch.Tensor,
        output_steps: Optional[int] = None,
    ) -> torch.Tensor:
        """Old-style embedding used by isolated pretraining."""
        hidden = self._extract_framewise_hidden(waveform, output_steps=output_steps)
        return F.dropout(
            hidden, p=self.embedding_dropout, training=self.training
        )

    def forward(self, waveform: torch.Tensor) -> Dict[str, torch.Tensor]:
        # Historical code applies fc_out to the undropped fc1 activation while
        # exposing a separately dropped embedding. Preserve that behavior.
        hidden = self._extract_framewise_hidden(waveform)
        embeddings = F.dropout(
            hidden, p=self.embedding_dropout, training=self.training
        )
        logits = self.fc_out(hidden)
        return {
            "segmentwise_logits": logits,
            "embedding": embeddings,
            "framewise_embedding": embeddings,
        }


def make_model(
    cfg: ExperimentConfig,
    target_sr: int,
    classes_num: int,
    output_steps: Optional[int] = None,
    stage_name: str = "polyphonic fine-tuning",
) -> Cnn6LinearSpecFramewise:
    window_size = int(round(target_sr * cfg.stft_window_ms / 1000.0))
    hop_size = int(round(target_sr * cfg.stft_hop_ms / 1000.0))
    resolved_output_steps = (
        int(round(cfg.sequence_seconds / cfg.label_hop_seconds))
        if output_steps is None
        else int(output_steps)
    )
    if resolved_output_steps < 1:
        raise ValueError("output_steps must be positive.")
    print(f"\nModel/STFT configuration ({stage_name}):")
    print("  architecture:      legacy Cnn6LinearSpec, framewise (no LSTM)")

    print("  input norm:        BatchNorm per linear-frequency bin")

    print("  temporal pooling:  local max + mean per output step")
    print(f"  sample rate:       {target_sr}")
    print(f"  STFT window:       {window_size} samples ({cfg.stft_window_ms:.1f} ms)")
    print(f"  STFT hop:          {hop_size} samples ({cfg.stft_hop_ms:.1f} ms)")
    print(f"  frequency bins:    {window_size // 2 + 1}")
    print(f"  frequency spacing: {target_sr / window_size:.3f} Hz")
    print(f"  output steps:      {resolved_output_steps}")
    return Cnn6LinearSpecFramewise(
        sample_rate=target_sr,
        window_size=window_size,
        hop_size=hop_size,
        classes_num=classes_num,
        output_steps=resolved_output_steps,
        cfg=cfg,
    )


@dataclass(frozen=True)
class IsolatedPretrainingDescriptor:
    song_id: SongId
    start_step: int
    instrument: str
    target_index: int  # selected-class index; -1 denotes a non-target negative


class IsolatedStemPretrainingDataset(Dataset):
    def __init__(
        self,
        descriptors: Sequence[IsolatedPretrainingDescriptor],
        local_catalog: LocalCatalog,
        cfg: ExperimentConfig,
        binary_mode: bool,
    ) -> None:
        self.descriptors = list(descriptors)
        self.local_catalog = local_catalog
        self.cfg = cfg
        self.binary_mode = bool(binary_mode)
        self.current_epoch = 0

    def __len__(self) -> int:
        return len(self.descriptors)

    def set_epoch(self, epoch: int) -> None:
        self.current_epoch = int(epoch)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        descriptor = self.descriptors[index]
        start_seconds = descriptor.start_step * self.cfg.label_hop_seconds
        audio = read_audio_segment(
            self.local_catalog.stems[descriptor.instrument][descriptor.song_id],
            start_seconds,
            self.cfg.isolated_pretrain_clip_seconds,
            self.local_catalog.sample_rate,
        )
        digest = int(hashlib.sha1(
            f"isolated_pretrain|{self.current_epoch}|{descriptor.song_id}|"
            f"{descriptor.start_step}|{descriptor.instrument}|{descriptor.target_index}".encode("utf-8")
        ).hexdigest()[:8], 16)
        rng = np.random.default_rng(
            (self.cfg.seed + (self.current_epoch + 1) * 1_000_003 + digest) % (2**32)
        )
        gain_db = float(rng.uniform(
            self.cfg.isolated_pretrain_random_gain_db_min,
            self.cfg.isolated_pretrain_random_gain_db_max,
        ))
        audio = audio * (10.0 ** (gain_db / 20.0))
        peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
        if peak > self.cfg.synthetic_peak_limit > 0:
            audio *= self.cfg.synthetic_peak_limit / peak
        return {
            "audio": torch.from_numpy(audio.astype(np.float32, copy=False)),
            "target_index": int(
                descriptor.target_index if not self.binary_mode
                else (1 if descriptor.target_index >= 0 else 0)
            ),
        }


class BalancedIsolatedStemSampler(Sampler[int]):
    def __init__(
        self,
        descriptors: Sequence[IsolatedPretrainingDescriptor],
        group_values: Sequence[int],
        samples_per_group: int,
        seed: int,
    ) -> None:
        self.by_group: Dict[int, List[int]] = defaultdict(list)
        for index, descriptor in enumerate(descriptors):
            self.by_group[int(descriptor.target_index)].append(index)
        self.group_values = tuple(int(value) for value in group_values)
        missing = [value for value in self.group_values if not self.by_group[value]]
        if missing:
            raise RuntimeError(f"No eligible isolated pretraining sequences for groups: {missing}")
        self.samples_per_group = int(samples_per_group)
        self.seed = int(seed)
        self.current_epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.current_epoch = int(epoch)

    def __len__(self) -> int:
        return len(self.group_values) * self.samples_per_group

    def __iter__(self):
        rng = np.random.default_rng(
            (self.seed + (self.current_epoch + 1) * 2_000_003) % (2**32)
        )
        selected: List[int] = []
        for group_value in self.group_values:
            pool = np.asarray(self.by_group[group_value], dtype=np.int64)
            chosen = rng.choice(
                pool,
                size=self.samples_per_group,
                replace=len(pool) < self.samples_per_group,
            )
            selected.extend(int(value) for value in chosen)
        rng.shuffle(selected)
        return iter(selected)


def _active_half_second_steps(path: Path, sample_rate: int, cfg: ExperimentConfig) -> List[int]:
    """Return complete non-silent 0.5-second blocks without loading a whole stem."""
    block_frames = int(round(cfg.isolated_pretrain_clip_seconds * sample_rate))
    threshold = 10.0 ** (cfg.activity_threshold_dbfs / 20.0)
    active: List[int] = []
    with sf.SoundFile(str(path), "r") as handle:
        if int(handle.samplerate) != int(sample_rate):
            raise RuntimeError(
                f"Pretraining stem sample rate mismatch for {path}: "
                f"{handle.samplerate} != {sample_rate}"
            )
        step = 0
        while True:
            block = handle.read(block_frames, dtype="float32", always_2d=True)
            if len(block) < block_frames:
                break
            mono = np.mean(block, axis=1, dtype=np.float64)
            rms = float(np.sqrt(np.mean(np.square(mono, dtype=np.float64))))
            if rms > threshold:
                active.append(step)
            step += 1
    return active


def _resolve_catalog_instruments(
    configured: Sequence[str], local_catalog: LocalCatalog
) -> Tuple[str, ...]:
    by_normalized = {
        normalize_name(instrument): instrument
        for instrument in local_catalog.stems
    }
    resolved: List[str] = []
    missing: List[str] = []
    for requested in configured:
        actual = by_normalized.get(normalize_name(requested))
        if actual is None:
            missing.append(requested)
        elif actual not in resolved:
            resolved.append(actual)
    if missing:
        raise RuntimeError(
            "Configured isolated-pretraining negative stems are missing: "
            + ", ".join(missing)
        )
    return tuple(resolved)


def _isolated_pretraining_descriptors(
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    train_song_ids: Sequence[SongId],
    cfg: ExperimentConfig,
) -> Tuple[List[IsolatedPretrainingDescriptor], pd.DataFrame, bool]:
    """Build strict active clips; one selected class uses balanced non-target negatives."""
    descriptors: List[IsolatedPretrainingDescriptor] = []
    counts: Counter = Counter()
    songs_by_name: Dict[str, set] = defaultdict(set)
    threshold = float(cfg.activity_threshold_dbfs)

    for song_id in train_song_ids:
        raw_rms = label_store.song_rms_dbfs[song_id]
        duration = float(label_store.song_durations[song_id])
        for instrument, class_index in label_store.class_to_index.items():
            if song_id not in local_catalog.stems[instrument]:
                continue
            active_steps = np.flatnonzero(
                np.isfinite(raw_rms[:, class_index])
                & (raw_rms[:, class_index] > threshold)
            )
            for start_step_value in active_steps:
                start_step = int(start_step_value)
                clip_end = start_step * cfg.label_hop_seconds + cfg.isolated_pretrain_clip_seconds
                if clip_end > duration + 1e-8:
                    continue
                descriptors.append(IsolatedPretrainingDescriptor(
                    song_id, start_step, instrument, int(class_index)
                ))
                counts[("positive", instrument)] += 1
                songs_by_name[f"positive:{instrument}"].add(song_id)

    binary_mode = len(label_store.instruments) == 1
    if binary_mode:
        negatives = _resolve_catalog_instruments(
            cfg.isolated_pretrain_negative_instruments, local_catalog
        )
        if not negatives:
            raise RuntimeError(
                "Single-instrument pretraining requires at least one non-target negative instrument."
            )
        for instrument in negatives:
            for song_id in train_song_ids:
                path = local_catalog.stems.get(instrument, {}).get(song_id)
                if path is None:
                    continue
                for start_step in _active_half_second_steps(
                    path, local_catalog.sample_rate, cfg
                ):
                    descriptors.append(IsolatedPretrainingDescriptor(
                        song_id, int(start_step), instrument, -1
                    ))
                    counts[("negative", instrument)] += 1
                    songs_by_name[f"negative:{instrument}"].add(song_id)

    rows: List[Dict[str, Any]] = []
    for (role, instrument), count in sorted(counts.items()):
        rows.append({
            "role": role,
            "instrument": instrument,
            "eligible_active_half_seconds": int(count),
            "eligible_seconds": int(count) * cfg.isolated_pretrain_clip_seconds,
            "training_song_count": len(songs_by_name[f"{role}:{instrument}"]),
            "strict_rms_threshold_dbfs": threshold,
        })
    summary = pd.DataFrame(rows)
    positive_count = sum(
        count for (role, _), count in counts.items() if role == "positive"
    )
    negative_count = sum(
        count for (role, _), count in counts.items() if role == "negative"
    )
    if positive_count == 0:
        raise RuntimeError("Isolated pretraining has no active positive clips.")
    if binary_mode and negative_count == 0:
        raise RuntimeError("Binary isolated pretraining has no active negative clips.")
    return descriptors, summary, binary_mode


def _isolated_pretraining_signature(
    descriptors: Sequence[IsolatedPretrainingDescriptor],
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    train_song_ids: Sequence[SongId],
    cfg: ExperimentConfig,
    binary_mode: bool,
) -> Tuple[str, Dict[str, Any]]:
    stem_files = []
    seen = set()
    for descriptor in descriptors:
        path = local_catalog.stems[descriptor.instrument][descriptor.song_id]
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        stat = path.stat()
        info = sf.info(str(path))
        stem_files.append({
            "instrument": descriptor.instrument,
            "song_id": descriptor.song_id,
            "path": key,
            "size": stat.st_size,
            "frames": int(info.frames),
            "sample_rate": int(info.samplerate),
        })
    payload = {
        "version": "cnn6_half_second_binary_or_multiclass_v3",
        "pretraining_mode": "binary_presence" if binary_mode else "multiclass_identity",
        "audio_mode": cfg.audio_mode,
        "sample_rate": local_catalog.sample_rate,
        "instruments": list(label_store.instruments),
        "negative_instruments": list(cfg.isolated_pretrain_negative_instruments),
        "train_song_ids": list(train_song_ids),
        "stem_files": sorted(stem_files, key=lambda row: (row["instrument"], float(row["song_id"]))),
        "model_architecture": cfg.model_architecture,
        "clip_seconds": cfg.isolated_pretrain_clip_seconds,
        "strict_raw_rms_threshold_dbfs": cfg.activity_threshold_dbfs,
        "descriptor_count": len(descriptors),
        "descriptor_sha256": hashlib.sha256(
            "|".join(
                f"{item.song_id}:{item.start_step}:{item.instrument}:{item.target_index}"
                for item in descriptors
            ).encode("utf-8")
        ).hexdigest(),
        "epochs": cfg.isolated_pretrain_epochs,
        "samples_per_group": cfg.isolated_pretrain_samples_per_instrument,
        "batch_size": cfg.isolated_pretrain_batch_size,
        "learning_rate": cfg.isolated_pretrain_learning_rate,
        "gain_db": [
            cfg.isolated_pretrain_random_gain_db_min,
            cfg.isolated_pretrain_random_gain_db_max,
        ],
        "seed": cfg.seed,
        "input_normalization": "per_frequency_batchnorm",
        "conv_normalization": "batchnorm",
        "temporal_pooling": "local_max_plus_mean",
        "embedding_size": cfg.embedding_size,
        "stft_window_ms": cfg.stft_window_ms,
        "stft_hop_ms": cfg.stft_hop_ms,
    }
    encoded = json.dumps(json_ready(payload), sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest(), payload


def _atomic_torch_save(payload: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(dict(payload), temporary)
    os.replace(temporary, path)


def _encoder_update_l2(
    parameters: Sequence[torch.nn.Parameter], reference: Sequence[torch.Tensor]
) -> float:
    squared = 0.0
    for parameter, initial in zip(parameters, reference):
        difference = parameter.detach().float().cpu() - initial
        squared += float(torch.sum(difference * difference).item())
    return math.sqrt(squared)


def train_or_load_isolated_stem_encoder(
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    train_song_ids: Sequence[SongId],
    device: torch.device,
    cfg: ExperimentConfig,
) -> Path:
    descriptors, pool_summary, binary_mode = _isolated_pretraining_descriptors(
        local_catalog, label_store, train_song_ids, cfg
    )
    signature, signature_payload = _isolated_pretraining_signature(
        descriptors, local_catalog, label_store, train_song_ids, cfg, binary_mode
    )
    classes_slug = "-".join(safe_slug(name) for name in label_store.instruments)
    mode_slug = "binary" if binary_mode else "multiclass"
    cache_name = f"{safe_slug(cfg.audio_mode)}_{classes_slug}_{mode_slug}_{signature[:16]}"
    if cfg.isolated_pretrain_force_rebuild:
        cache_name += "_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    cache_dir = Path(cfg.drive_output_root) / "_isolated_stem_pretraining" / cache_name
    cache_dir.mkdir(parents=True, exist_ok=True)
    complete_path = cache_dir / "isolated_stem_encoder.pth"
    recovery_path = cache_dir / "last_pretraining.pth"
    history_path = cache_dir / "pretraining_history.csv"
    pool_summary.to_csv(cache_dir / "pretraining_pool_summary.csv", index=False)
    save_json(cache_dir / "pretraining_signature.json", signature_payload)

    if cfg.isolated_pretrain_reuse_checkpoint and not cfg.isolated_pretrain_force_rebuild and complete_path.is_file():
        checkpoint = torch.load(complete_path, map_location="cpu", weights_only=False)
        if checkpoint.get("pretraining_signature") == signature:
            history = checkpoint.get("history", [])
            if not history or float(history[-1].get("encoder_update_l2", 0.0)) <= 0.0:
                raise RuntimeError("Cached pretraining checkpoint has no verified encoder update.")
            print(f"\nReusing verified isolated-stem CNN6 checkpoint: {complete_path}")
            return complete_path

    print("\nIsolated-stem CNN6 pretraining pool:")
    print(pool_summary.to_string(index=False))
    print(
        f"Pretraining mode={mode_slug}; epochs={cfg.isolated_pretrain_epochs}, "
        f"samples/group/epoch={cfg.isolated_pretrain_samples_per_instrument}, "
        f"clip={cfg.isolated_pretrain_clip_seconds:.1f}s, "
        f"strict raw RMS>{cfg.activity_threshold_dbfs:.1f} dBFS, "
        f"batch={cfg.isolated_pretrain_batch_size}. Validation family is excluded."
    )

    dataset = IsolatedStemPretrainingDataset(
        descriptors, local_catalog, cfg, binary_mode=binary_mode
    )
    group_values = (-1, 0) if binary_mode else tuple(range(len(label_store.instruments)))
    sampler = BalancedIsolatedStemSampler(
        descriptors,
        group_values,
        cfg.isolated_pretrain_samples_per_instrument,
        cfg.seed + 31_337,
    )
    loader = DataLoader(
        dataset,
        batch_size=cfg.isolated_pretrain_batch_size,
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory and torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        drop_last=False,
        **_data_loader_worker_kwargs(cfg),
    )

    model = make_model(
        cfg,
        local_catalog.sample_rate,
        len(label_store.instruments),
        output_steps=1,
        stage_name="isolated 0.5-second pretraining",
    ).to(device)
    classifier_outputs = 1 if binary_mode else len(label_store.instruments)
    classifier = nn.Linear(cfg.embedding_size, classifier_outputs).to(device)
    init_layer(classifier)
    encoder_parameters = [
        parameter
        for module in model.encoder_modules()
        for parameter in module.parameters()
    ]
    optimizer = torch.optim.AdamW(
        [{"params": encoder_parameters}, {"params": classifier.parameters()}],
        lr=cfg.isolated_pretrain_learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(1, cfg.isolated_pretrain_epochs),
        eta_min=cfg.isolated_pretrain_learning_rate * 0.10,
    )
    amp_enabled = cfg.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    criterion = nn.BCEWithLogitsLoss() if binary_mode else nn.CrossEntropyLoss()
    start_epoch = 0
    history: List[Dict[str, Any]] = []

    if cfg.isolated_pretrain_reuse_checkpoint and recovery_path.is_file():
        recovery = torch.load(recovery_path, map_location="cpu", weights_only=False)
        if recovery.get("pretraining_signature") == signature:
            model.load_state_dict(recovery["model_state_dict"])
            classifier.load_state_dict(recovery["classifier_state_dict"])
            optimizer.load_state_dict(recovery["optimizer_state_dict"])
            scheduler.load_state_dict(recovery["scheduler_state_dict"])
            scaler.load_state_dict(recovery["scaler_state_dict"])
            history = list(recovery.get("history", []))
            start_epoch = int(recovery.get("epoch", -1)) + 1
            print(f"Resuming isolated pretraining at epoch {start_epoch + 1}/{cfg.isolated_pretrain_epochs}.")

    encoder_reference = [parameter.detach().float().cpu().clone() for parameter in encoder_parameters]
    for epoch in range(start_epoch, cfg.isolated_pretrain_epochs):
        dataset.set_epoch(epoch)
        sampler.set_epoch(epoch)
        model.train()
        classifier.train()
        running_loss = 0.0
        group_count = 2 if binary_mode else len(label_store.instruments)
        correct = np.zeros(group_count, dtype=np.int64)
        total = np.zeros(group_count, dtype=np.int64)
        progress = tqdm(loader, desc=f"Isolated CNN6 pretrain {epoch + 1}/{cfg.isolated_pretrain_epochs}", leave=False)
        for batch_index, batch in enumerate(progress):
            audio = batch["audio"].to(device, non_blocking=True)
            target_index = batch["target_index"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                embedding = model._extract_framewise_hidden(audio)[:, 0]
                logits = classifier(embedding)
                loss = (
                    criterion(logits[:, 0], target_index.float())
                    if binary_mode
                    else criterion(logits, target_index)
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([*encoder_parameters, *classifier.parameters()], cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item())
            prediction = (
                (torch.sigmoid(logits[:, 0].detach()) >= 0.5).long()
                if binary_mode
                else torch.argmax(logits.detach(), dim=1)
            )
            truth_np = target_index.detach().cpu().numpy()
            prediction_np = prediction.cpu().numpy()
            for group_value in range(group_count):
                mask = truth_np == group_value
                total[group_value] += int(mask.sum())
                correct[group_value] += int(np.sum(prediction_np[mask] == group_value))
            progress.set_postfix(loss=f"{running_loss / (batch_index + 1):.4f}")

        scheduler.step()
        per_group_accuracy = correct / np.maximum(total, 1)
        update_l2 = _encoder_update_l2(encoder_parameters, encoder_reference)
        row: Dict[str, Any] = {
            "epoch": epoch + 1,
            "mode": mode_slug,
            "loss": running_loss / max(1, len(loader)),
            "accuracy": float(correct.sum() / max(total.sum(), 1)),
            "macro_accuracy": float(np.mean(per_group_accuracy)),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "encoder_update_l2": update_l2,
        }
        if binary_mode:
            row["negative_accuracy"] = float(per_group_accuracy[0])
            row["positive_accuracy"] = float(per_group_accuracy[1])
            row["balanced_accuracy"] = float(np.mean(per_group_accuracy))
        else:
            for class_index, instrument in enumerate(label_store.instruments):
                row[f"accuracy_{safe_slug(instrument)}"] = float(per_group_accuracy[class_index])
        if not math.isfinite(row["loss"]) or update_l2 <= 0.0:
            raise RuntimeError(
                "Isolated pretraining failed to update the encoder; refusing to transfer weights."
            )
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(
            f"Isolated pretrain {epoch + 1:02d}/{cfg.isolated_pretrain_epochs} | "
            f"loss={row['loss']:.4f}, macro-accuracy={row['macro_accuracy']:.4f}, "
            f"encoder-update-L2={update_l2:.6f}",
            flush=True,
        )
        _atomic_torch_save({
            "epoch": epoch,
            "pretraining_signature": signature,
            "model_state_dict": model.state_dict(),
            "classifier_state_dict": classifier.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history,
        }, recovery_path)

    if not history or float(history[-1].get("encoder_update_l2", 0.0)) <= 0.0:
        raise RuntimeError("Pretraining completed without a verified encoder update.")
    _atomic_torch_save({
        "pretraining_signature": signature,
        "pretraining_mode": mode_slug,
        "model_state_dict": model.state_dict(),
        "classifier_state_dict": classifier.state_dict(),
        "history": history,
        "config": json_ready(asdict(cfg)),
        "instruments": list(label_store.instruments),
        "negative_instruments": list(cfg.isolated_pretrain_negative_instruments),
        "train_song_ids": list(train_song_ids),
    }, complete_path)
    print(f"Completed verified isolated-stem CNN6 pretraining: {complete_path}")
    del model, classifier, optimizer, scheduler, scaler, loader, dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return complete_path


def load_pretrained_stem_encoder(
    model: Cnn6LinearSpecFramewise,
    checkpoint_path: Optional[str],
    transfer_fc1: bool,
) -> int:
    if not checkpoint_path:
        print("\nNo isolated-stem checkpoint selected; CNN encoder starts from random initialization.")
        return 0

    path = Path(checkpoint_path)
    if not path.exists():
        raise FileNotFoundError(f"Pretrained stem checkpoint does not exist: {path}")

    checkpoint = torch.load(str(path), map_location="cpu", weights_only=False)
    source_state = checkpoint.get("model_state_dict", checkpoint)
    cleaned: Dict[str, torch.Tensor] = {}
    for key, value in source_state.items():
        new_key = key.removeprefix("module.")
        cleaned[new_key] = value

    allowed_prefixes = [
        "bn0.",
        "conv_block1.",
        "conv_block2.",
        "conv_block3.",
        "conv_block4.",
    ]
    if transfer_fc1:
        allowed_prefixes.append("fc1.")

    target_state = model.state_dict()
    transferable = {
        key: value
        for key, value in cleaned.items()
        if any(key.startswith(prefix) for prefix in allowed_prefixes)
        and key in target_state
        and tuple(value.shape) == tuple(target_state[key].shape)
    }
    model.load_state_dict(transferable, strict=False)
    print(f"\nTransferred {len(transferable)} tensors from isolated-stem checkpoint: {path}")
    if not transferable:
        warnings.warn("No compatible encoder tensors were found in the supplied checkpoint.")
    return len(transferable)




In [ ]:
from __future__ import annotations

# =============================================================================
# TRAINING AND FULL-SONG VALIDATION
# =============================================================================


def _data_loader_worker_kwargs(cfg: ExperimentConfig) -> Dict[str, Any]:
    if cfg.num_workers <= 0:
        return {}
    return {
        "prefetch_factor": cfg.prefetch_factor,
        "persistent_workers": cfg.persistent_workers,
    }


def make_validation_loader(
    dataset: InstrumentSequenceDataset,
    cfg: ExperimentConfig,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory and torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        drop_last=False,
        **_data_loader_worker_kwargs(cfg),
    )


def make_selected_only_validation_loader(
    natural_val_descriptors: Sequence[SequenceDescriptor],
    local_catalog: LocalCatalog,
    label_store: LabelStore,
    cfg: ExperimentConfig,
) -> DataLoader:
    validation_song_ids = sorted(
        {canonical_song_id(item.song_id) for item in natural_val_descriptors},
        key=song_id_sort_key,
    )
    requested_backgrounds = _resolve_catalog_instruments(
        cfg.controlled_validation_background_instruments, local_catalog
    )
    required = tuple(dict.fromkeys((*label_store.instruments, *requested_backgrounds)))
    missing = [
        f"{instrument}@{song_id}"
        for instrument in required
        for song_id in validation_song_ids
        if song_id not in local_catalog.stems.get(instrument, {})
    ]
    if missing:
        raise RuntimeError(
            "Controlled validation requires every configured stem; missing: "
            + ", ".join(missing)
        )
    controlled_descriptors = [
        SequenceDescriptor(SYNTHETIC_REGULAR, item.song_id, item.start_step)
        for item in natural_val_descriptors
    ]
    dataset = InstrumentSequenceDataset(
        controlled_descriptors,
        local_catalog,
        label_store,
        cfg,
        dropout_plan=None,
        deterministic_synthetic=True,
        background_mode="all" if requested_backgrounds else "none",
        fallback_to_real_mix_when_no_target=False,
        fixed_background_instruments=requested_backgrounds,
    )
    print(
        f"Controlled validation sequences: {len(controlled_descriptors)}; "
        f"outputs={list(label_store.instruments)}; "
        f"fixed backgrounds={list(requested_backgrounds)}."
    )
    return make_validation_loader(dataset, cfg)


def make_data_loaders(
    train_dataset: InstrumentSequenceDataset,
    val_dataset: InstrumentSequenceDataset,
    train_descriptors: Sequence[SequenceDescriptor],
    cfg: ExperimentConfig,
) -> Tuple[DataLoader, DataLoader]:
    if not cfg.deterministic_epoch_sampling:
        raise ValueError("This notebook requires deterministic_epoch_sampling=True.")
    sampler = build_epoch_quota_sampler(train_dataset, cfg)
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory and torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        drop_last=False,
        **_data_loader_worker_kwargs(cfg),
    )
    val_loader = make_validation_loader(val_dataset, cfg)
    return train_loader, val_loader


def _binary_metrics(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    instruments: Sequence[str],
    threshold: float,
    valid_mask: Optional[np.ndarray] = None,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    y_pred = (y_probability >= threshold).astype(np.uint8)
    if valid_mask is None:
        valid_mask = np.ones_like(y_true, dtype=bool)
    else:
        valid_mask = np.asarray(valid_mask) > 0.5
        if valid_mask.shape != y_true.shape:
            raise ValueError("valid_mask must have the same shape as y_true.")
    if not np.any(valid_mask):
        raise ValueError("valid_mask contains no supervised elements.")
    rows: List[Dict[str, Any]] = []
    macro_f1_values: List[float] = []
    accuracy_values: List[float] = []
    balanced_accuracy_values: List[float] = []
    ap_values: List[float] = []
    auc_values: List[float] = []

    for class_index, instrument in enumerate(instruments):
        class_valid = valid_mask[:, class_index]
        truth = y_true[class_valid, class_index].astype(np.uint8)
        probability = y_probability[class_valid, class_index]
        prediction = y_pred[class_valid, class_index]
        if len(truth) == 0:
            raise ValueError(f"No supervised metric elements for {instrument}.")
        positives = int(truth.sum())
        negatives = int(len(truth) - positives)

        tp = int(np.sum((truth == 1) & (prediction == 1)))
        tn = int(np.sum((truth == 0) & (prediction == 0)))
        fp = int(np.sum((truth == 0) & (prediction == 1)))
        fn = int(np.sum((truth == 1) & (prediction == 0)))

        precision = precision_score(truth, prediction, zero_division=0)
        recall = recall_score(truth, prediction, zero_division=0)
        inactive_recall = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        balanced_accuracy = (
            0.5 * (recall + inactive_recall)
            if positives > 0 and negatives > 0 and np.isfinite(inactive_recall)
            else np.nan
        )
        accuracy = float(np.mean(truth == prediction))
        f1 = f1_score(truth, prediction, zero_division=0)
        ap = (
            average_precision_score(truth, probability)
            if positives > 0 and negatives > 0
            else np.nan
        )
        auc = roc_auc_score(truth, probability) if positives > 0 and negatives > 0 else np.nan

        if positives > 0:
            macro_f1_values.append(float(f1))
        accuracy_values.append(accuracy)
        if np.isfinite(balanced_accuracy):
            balanced_accuracy_values.append(float(balanced_accuracy))
        if np.isfinite(ap):
            ap_values.append(float(ap))
        if np.isfinite(auc):
            auc_values.append(float(auc))

        rows.append(
            {
                "instrument": instrument,
                "positive_steps": positives,
                "negative_steps": negatives,
                "true_positive": tp,
                "true_negative": tn,
                "false_positive": fp,
                "false_negative": fn,
                "precision": precision,
                "recall": recall,
                "inactive_recall": inactive_recall,
                "accuracy": accuracy,
                "balanced_accuracy": balanced_accuracy,
                "f1": f1,
                "average_precision": ap,
                "roc_auc": auc,
            }
        )

    supervised_rows = np.any(valid_mask, axis=1)
    row_correct = np.all((y_true == y_pred) | ~valid_mask, axis=1)
    summary = {
        "accuracy": float(np.mean(y_true[valid_mask] == y_pred[valid_mask])),
        "macro_accuracy": float(np.mean(accuracy_values)) if accuracy_values else float("nan"),
        "macro_balanced_accuracy": (
            float(np.mean(balanced_accuracy_values))
            if balanced_accuracy_values
            else float("nan")
        ),
        "worst_class_balanced_accuracy": (
            float(np.min(balanced_accuracy_values))
            if balanced_accuracy_values
            else float("nan")
        ),
        "balanced_accuracy_class_count": int(len(balanced_accuracy_values)),
        "subset_accuracy": float(np.mean(row_correct[supervised_rows])),
        "macro_f1": float(np.mean(macro_f1_values)) if macro_f1_values else float("nan"),
        "micro_f1": float(
            f1_score(y_true[valid_mask], y_pred[valid_mask], zero_division=0)
        ),
        "macro_average_precision": float(np.mean(ap_values)) if ap_values else float("nan"),
        "macro_roc_auc": float(np.mean(auc_values)) if auc_values else float("nan"),
    }
    return pd.DataFrame(rows), summary


@torch.no_grad()
def evaluate_full_songs(
    model: nn.Module,
    val_loader: DataLoader,
    label_store: LabelStore,
    criterion: nn.Module,
    device: torch.device,
    cfg: ExperimentConfig,
    description: str = "Validation",
) -> Dict[str, Any]:
    model.eval()
    amp_enabled = cfg.use_amp and device.type == "cuda"

    probability_sums: Dict[SongId, np.ndarray] = {}
    truth_sums: Dict[SongId, np.ndarray] = {}
    probability_counts: Dict[SongId, np.ndarray] = {}
    validation_song_ids = {canonical_song_id(d.song_id) for d in val_loader.dataset.descriptors}
    for song_id in sorted(validation_song_ids, key=song_id_sort_key):
        shape = label_store.song_labels[song_id].shape
        probability_sums[song_id] = np.zeros(shape, dtype=np.float64)
        truth_sums[song_id] = np.zeros(shape, dtype=np.float64)
        probability_counts[song_id] = np.zeros(shape[0], dtype=np.float64)

    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(val_loader, desc=description, leave=False):
        audio = batch["audio"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            output = model(audio)
            logits = output["segmentwise_logits"]
            loss = criterion(logits, target)

        probabilities = torch.sigmoid(logits).float().cpu().numpy()
        targets = target.detach().float().cpu().numpy()
        song_ids = batch["song_id"].cpu().numpy()
        start_steps = batch["start_step"].cpu().numpy()

        total_loss += float(loss.item())
        total_batches += 1

        for sample_index in range(len(song_ids)):
            song_id = canonical_song_id(song_ids[sample_index])
            start_step = int(start_steps[sample_index])
            end_step = min(
                start_step + probabilities.shape[1],
                probability_sums[song_id].shape[0],
            )
            available = end_step - start_step
            if available <= 0:
                continue
            probability_sums[song_id][start_step:end_step] += probabilities[sample_index, :available]
            truth_sums[song_id][start_step:end_step] += targets[sample_index, :available]
            probability_counts[song_id][start_step:end_step] += 1.0

    all_truth: List[np.ndarray] = []
    all_probability: List[np.ndarray] = []
    half_second_rows: List[Dict[str, Any]] = []

    for song_id in sorted(probability_sums):
        counts = probability_counts[song_id]
        covered = counts > 0
        if not np.all(covered):
            warnings.warn(
                f"Validation song {song_id} has {int((~covered).sum())} uncovered half-second steps."
            )
        probabilities = probability_sums[song_id] / np.maximum(counts[:, None], 1.0)
        truth = (truth_sums[song_id] / np.maximum(counts[:, None], 1.0) >= 0.5).astype(np.float32)

        all_truth.append(truth[covered])
        all_probability.append(probabilities[covered])

        prediction = probabilities >= cfg.probability_threshold
        for step in np.flatnonzero(covered):
            row: Dict[str, Any] = {
                "song_id": song_id,
                "step": int(step),
                "time_start_sec": float(step * cfg.label_hop_seconds),
                "time_end_sec": float((step + 1) * cfg.label_hop_seconds),
            }
            for class_index, instrument in enumerate(label_store.instruments):
                slug = safe_slug(instrument)
                row[f"label_{slug}"] = int(truth[step, class_index] >= 0.5)
                row[f"prob_{slug}"] = float(probabilities[step, class_index])
                row[f"pred_{slug}"] = int(prediction[step, class_index])
            half_second_rows.append(row)

    y_true = np.concatenate(all_truth, axis=0)
    y_probability = np.concatenate(all_probability, axis=0)
    per_class, summary = _binary_metrics(
        y_true,
        y_probability,
        label_store.instruments,
        cfg.probability_threshold,
    )
    summary["val_loss"] = total_loss / max(1, total_batches)

    half_second_frame = pd.DataFrame(half_second_rows)
    second_rows: List[Dict[str, Any]] = []
    for (song_id, second), group in half_second_frame.groupby(
        ["song_id", half_second_frame["time_start_sec"].astype(int)],
        sort=True,
    ):
        row = {"song_id": song_id, "second": int(second)}
        for instrument in label_store.instruments:
            slug = safe_slug(instrument)
            row[f"label_{slug}"] = int(group[f"label_{slug}"].max())
            row[f"prob_{slug}"] = float(group[f"prob_{slug}"].max())
            row[f"pred_{slug}"] = int(group[f"pred_{slug}"].max())
        second_rows.append(row)
    second_frame = pd.DataFrame(second_rows)

    return {
        "summary": summary,
        "per_class": per_class,
        "half_second_predictions": half_second_frame,
        "second_predictions": second_frame,
    }


def _encoder_parameter_ids(model: Cnn6LinearSpecFramewise) -> set[int]:
    return {id(parameter) for module in model.encoder_modules() for parameter in module.parameters()}


def build_optimizer(model: Cnn6LinearSpecFramewise, cfg: ExperimentConfig) -> torch.optim.Optimizer:
    encoder_ids = _encoder_parameter_ids(model)
    encoder_parameters = [p for p in model.parameters() if id(p) in encoder_ids]
    new_parameters = [p for p in model.parameters() if id(p) not in encoder_ids]
    return torch.optim.AdamW(
        [
            {"params": encoder_parameters, "lr": cfg.encoder_learning_rate},
            {"params": new_parameters, "lr": cfg.new_layers_learning_rate},
        ],
        weight_decay=cfg.weight_decay,
    )



def build_warmup_cosine_scheduler(
    optimizer: torch.optim.Optimizer,
    cfg: ExperimentConfig,
) -> Optional[Any]:
    if not cfg.use_lr_scheduler:
        return None
    base_lrs = [cfg.encoder_learning_rate, cfg.new_layers_learning_rate]
    min_lrs = [cfg.min_encoder_learning_rate, cfg.min_new_layers_learning_rate]
    lambdas = []
    for base_lr, min_lr in zip(base_lrs, min_lrs):
        minimum_ratio = min(1.0, float(min_lr) / max(float(base_lr), 1e-20))

        def schedule(epoch_index: int, minimum_ratio: float = minimum_ratio) -> float:
            epoch_number = int(epoch_index) + 1
            if cfg.warmup_epochs > 0 and epoch_number <= cfg.warmup_epochs:
                return epoch_number / cfg.warmup_epochs
            if epoch_number <= cfg.lr_hold_epochs:
                return 1.0
            cosine_epochs = max(1, cfg.epochs - cfg.lr_hold_epochs)
            progress = min(
                1.0,
                max(0.0, (epoch_number - cfg.lr_hold_epochs) / cosine_epochs),
            )
            cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
            return minimum_ratio + (1.0 - minimum_ratio) * cosine

        lambdas.append(schedule)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambdas)


def make_ema_model(model: nn.Module) -> nn.Module:
    ema_model = copy.deepcopy(model)
    ema_model.requires_grad_(False)
    ema_model.eval()
    return ema_model


@torch.no_grad()
def update_ema_model(ema_model: nn.Module, model: nn.Module, decay: float) -> None:
    ema_parameters = dict(ema_model.named_parameters())
    model_parameters = dict(model.named_parameters())
    if ema_parameters.keys() != model_parameters.keys():
        raise RuntimeError("EMA/model parameter names differ.")
    for name, ema_parameter in ema_parameters.items():
        source = model_parameters[name].detach()
        ema_parameter.mul_(decay).add_(source, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    model_buffers = dict(model.named_buffers())
    for name, ema_buffer in ema_buffers.items():
        if name in model_buffers:
            ema_buffer.copy_(model_buffers[name].detach())

def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[Any],
    scaler: torch.amp.GradScaler,
    epoch: int,
    best_metric: float,
    history: Sequence[Mapping[str, Any]],
    cfg: ExperimentConfig,
    label_store: LabelStore,
    train_song_ids: Sequence[SongId],
    val_song_ids: Sequence[SongId],
    ema_model: Optional[nn.Module] = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "ema_model_state_dict": ema_model.state_dict() if ema_model is not None else None,
        "validation_weights": "ema" if ema_model is not None else "raw",
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict(),
        "best_metric": best_metric,
        "history": list(history),
        "config": json_ready(asdict(cfg)),
        "class_to_index": label_store.class_to_index,
        "instruments": list(label_store.instruments),
        "train_song_ids": list(train_song_ids),
        "val_song_ids": list(val_song_ids),
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
    }
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temp)
    os.replace(temp, path)


_RESUME_SIGNATURE_IGNORED_FIELDS = frozenset(
    {
        "dry_run",
        "resume_checkpoint",
        "force_rebuild_audio_cache",
        "force_rebuild_labels",
        "num_workers",
        "pin_memory",
        "prefetch_factor",
        "persistent_workers",
        "use_amp",
    }
)


def _resume_config_signature(
    config: Union[ExperimentConfig, Mapping[str, Any]],
) -> Dict[str, Any]:
    """Comparable experiment settings, excluding runtime-only controls."""
    payload = asdict(config) if isinstance(config, ExperimentConfig) else dict(config)
    for field_name in _RESUME_SIGNATURE_IGNORED_FIELDS:
        payload.pop(field_name, None)
    return json_ready(payload)


def _latest_compatible_checkpoint(
    cfg: ExperimentConfig,
) -> Optional[Tuple[Path, int]]:
    """Find the newest checkpoint-bearing run with the same experiment setup."""
    output_root = Path(cfg.drive_output_root)
    if not output_root.exists():
        return None

    expected_signature = _resume_config_signature(cfg)
    mode_token = f"_{safe_slug(cfg.audio_mode)}_"
    candidates: List[Tuple[float, Path, int]] = []

    for run_dir in output_root.iterdir():
        if not run_dir.is_dir() or mode_token not in run_dir.name:
            continue

        config_path = run_dir / "config.json"
        history_path = run_dir / "training_history.csv"
        checkpoint_path = run_dir / "last_model.pth"
        if not (
            config_path.is_file()
            and history_path.is_file()
            and checkpoint_path.is_file()
        ):
            continue

        try:
            with config_path.open("r", encoding="utf-8") as handle:
                saved_config = json.load(handle)
            if _resume_config_signature(saved_config) != expected_signature:
                continue

            history = pd.read_csv(history_path, usecols=["epoch"])
            epoch_values = pd.to_numeric(history["epoch"], errors="coerce").dropna()
            if epoch_values.empty:
                continue
            last_epoch = int(epoch_values.max())
            candidates.append(
                (checkpoint_path.stat().st_mtime, checkpoint_path, last_epoch)
            )
        except (OSError, ValueError, TypeError, KeyError, json.JSONDecodeError):
            continue

    if not candidates:
        return None

    _, checkpoint_path, last_epoch = max(candidates, key=lambda item: item[0])
    return checkpoint_path, last_epoch


def configure_auto_resume(
    cfg: ExperimentConfig,
    enabled: bool,
) -> ExperimentConfig:
    """Select the latest compatible unfinished run, or force a fresh start."""
    if not isinstance(enabled, bool):
        raise ValueError("AUTO_RESUME must be True or False.")

    cfg.resume_checkpoint = None

    if not enabled:
        print("AUTO_RESUME=False: starting a new experiment at epoch 1.")
        return cfg

    candidate = _latest_compatible_checkpoint(cfg)
    if candidate is None:
        if cfg.use_isolated_stem_pretraining:
            print(
                "AUTO_RESUME=True, but no completed main-training epoch exists. "
                "The isolated-pretraining cache will resume/reuse automatically; "
                "main training will start at epoch 1."
            )
            return cfg
        raise RuntimeError(
            "AUTO_RESUME=True, but no compatible checkpoint was found. "
            "Set AUTO_RESUME=False to start this dataset/condition at epoch 1."
        )

    checkpoint_path, last_epoch = candidate
    if last_epoch >= cfg.epochs:
        raise RuntimeError(
            f"The latest compatible run already completed epoch {last_epoch}. "
            "Set AUTO_RESUME=False to start a new experiment."
        )

    cfg.resume_checkpoint = str(checkpoint_path)
    print(
        "AUTO_RESUME=True: found a compatible interrupted run.\n"
        f"  checkpoint: {checkpoint_path}\n"
        f"  completed epoch: {last_epoch}\n"
        f"  next epoch: {last_epoch + 1}"
    )
    return cfg


def load_resume_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Optional[Any],
    scaler: torch.amp.GradScaler,
    cfg: ExperimentConfig,
    ema_model: Optional[nn.Module] = None,
) -> Tuple[int, float, List[Dict[str, Any]]]:
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)

    # A resume is only valid for the same experiment condition. In particular,
    # never carry learned weights/history from 44.1 kHz into a 96 kHz or LP run.
    saved_config = checkpoint.get("config", {})
    if isinstance(saved_config, Mapping):
        saved_audio_mode = saved_config.get("audio_mode")
        if saved_audio_mode and saved_audio_mode != cfg.audio_mode:
            raise ValueError(
                "Refusing cross-condition resume: checkpoint audio_mode="
                f"{saved_audio_mode!r}, current audio_mode={cfg.audio_mode!r}. "
                "Set resume_checkpoint=None to start a new bandwidth experiment."
            )
        saved_instruments = tuple(saved_config.get("selected_instruments", ()))
        if saved_instruments and saved_instruments != tuple(cfg.selected_instruments):
            raise ValueError(
                "Refusing resume with different model classes: checkpoint="
                f"{saved_instruments}, current={tuple(cfg.selected_instruments)}."
            )

        saved_signature = _resume_config_signature(saved_config)
        current_signature = _resume_config_signature(cfg)
        if saved_signature != current_signature:
            changed_fields = sorted(
                field_name
                for field_name in set(saved_signature) | set(current_signature)
                if saved_signature.get(field_name) != current_signature.get(field_name)
            )
            raise ValueError(
                "Refusing resume because the experiment configuration changed: "
                + ", ".join(changed_fields)
                + ". Set AUTO_RESUME=False for a new experiment."
            )

    model.load_state_dict(checkpoint["model_state_dict"])
    if ema_model is not None:
        ema_state = checkpoint.get("ema_model_state_dict")
        if ema_state is None:
            warnings.warn("Checkpoint has no EMA state; initializing EMA from raw weights.")
            ema_model.load_state_dict(model.state_dict())
        else:
            ema_model.load_state_dict(ema_state)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler_state = checkpoint.get("scheduler_state_dict")
    if scheduler is not None and scheduler_state is not None:
        scheduler.load_state_dict(scheduler_state)
    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    if checkpoint.get("python_random_state") is not None:
        random.setstate(checkpoint["python_random_state"])
    if checkpoint.get("numpy_random_state") is not None:
        np.random.set_state(checkpoint["numpy_random_state"])
    if checkpoint.get("torch_rng_state") is not None:
        torch.set_rng_state(checkpoint["torch_rng_state"])
    if torch.cuda.is_available() and checkpoint.get("cuda_rng_state_all") is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])
    start_epoch = int(checkpoint.get("epoch", -1)) + 1
    best_metric = float(checkpoint.get("best_metric", -np.inf))
    history = list(checkpoint.get("history", []))
    print(f"Resumed from {path}; next epoch={start_epoch + 1}, best_metric={best_metric:.4f}")
    return start_epoch, best_metric, history


class BalancedBinaryCrossEntropy(nn.Module):
    """Binary cross-entropy with separate ON/OFF weights for every instrument."""

    def __init__(self, on_weight: torch.Tensor, off_weight: torch.Tensor) -> None:
        super().__init__()
        self.register_buffer("on_weight", on_weight.view(1, 1, -1))
        self.register_buffer("off_weight", off_weight.view(1, 1, -1))

    def forward(
        self,
        logits: torch.Tensor,
        target: torch.Tensor,
        loss_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        on_loss = -target * F.logsigmoid(logits) * self.on_weight
        off_loss = -(1.0 - target) * F.logsigmoid(-logits) * self.off_weight
        element_loss = on_loss + off_loss
        if loss_mask is None:
            return element_loss.mean()
        loss_mask = loss_mask.to(dtype=element_loss.dtype)
        denominator = loss_mask.sum().clamp_min(1.0)
        return (element_loss * loss_mask).sum() / denominator


def train_model(
    model: Cnn6LinearSpecFramewise,
    train_loader: DataLoader,
    val_loader: DataLoader,
    selected_only_val_loader: Optional[DataLoader],
    label_store: LabelStore,
    on_weight: torch.Tensor,
    off_weight: torch.Tensor,
    device: torch.device,
    run_dir: Path,
    transferred_tensors: int,
    train_song_ids: Sequence[SongId],
    val_song_ids: Sequence[SongId],
    cfg: ExperimentConfig,
) -> Dict[str, Any]:
    """Run every epoch; validate EMA weights while retaining raw weights for resume."""
    model.to(device)
    criterion = BalancedBinaryCrossEntropy(
        on_weight.to(device), off_weight.to(device)
    ).to(device)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_warmup_cosine_scheduler(optimizer, cfg)
    ema_model: Optional[nn.Module] = make_ema_model(model) if cfg.use_ema else None

    amp_enabled = cfg.use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    start_epoch = 0
    latest_validation_metric = float("nan")
    best_validation_metric = -np.inf
    best_epoch: Optional[int] = None
    history: List[Dict[str, Any]] = []
    if cfg.resume_checkpoint:
        start_epoch, best_validation_metric, history = load_resume_checkpoint(
            cfg.resume_checkpoint,
            model,
            optimizer,
            scheduler,
            scaler,
            cfg,
            ema_model=ema_model,
        )
        finite_history = [
            row for row in history
            if np.isfinite(float(row.get("val_macro_balanced_accuracy", np.nan)))
        ]
        if finite_history:
            best_row = max(
                finite_history,
                key=lambda row: float(row["val_macro_balanced_accuracy"]),
            )
            best_validation_metric = float(best_row["val_macro_balanced_accuracy"])
            best_epoch = int(best_row["epoch"])

    freeze_epochs = cfg.freeze_pretrained_encoder_epochs if transferred_tensors > 0 else 0
    last_path = run_dir / "last_model.pth"
    best_path = run_dir / "best_natural_bacc_model.pth"
    if cfg.resume_checkpoint:
        previous_best = Path(cfg.resume_checkpoint).parent / best_path.name
        if previous_best.is_file():
            shutil.copy2(previous_best, best_path)

    def save_recovery_checkpoint(epoch: int, score: float) -> None:
        save_checkpoint(
            last_path,
            model,
            optimizer,
            scheduler,
            scaler,
            epoch,
            score,
            history,
            cfg,
            label_store,
            train_song_ids,
            val_song_ids,
            ema_model=ema_model,
        )

    print(
        f"Training schedule: epochs={cfg.epochs}; early stopping disabled; "
        f"validation weights={'EMA' if ema_model is not None else 'raw'}. Every epoch will run.",
        flush=True,
    )
    print(
        f"Runtime: batch={cfg.batch_size}, accumulation={cfg.gradient_accumulation_steps}, "
        f"workers={cfg.num_workers}, prefetch={cfg.prefetch_factor}, AMP={amp_enabled}.",
        flush=True,
    )
    if cfg.save_last_checkpoint:
        print(
            "Recovery: raw model, EMA model, optimizer and scheduler are saved atomically each epoch.",
            flush=True,
        )

    for epoch in range(start_epoch, cfg.epochs):
        if hasattr(train_loader.sampler, "set_epoch"):
            train_loader.sampler.set_epoch(epoch)
        if hasattr(train_loader.dataset, "set_epoch"):
            train_loader.dataset.set_epoch(epoch)
        encoder_frozen = epoch < freeze_epochs
        model.set_encoder_trainable(not encoder_frozen)
        model.train()
        if encoder_frozen:
            model.keep_frozen_encoder_in_eval_mode()

        learning_rates_used = [float(group["lr"]) for group in optimizer.param_groups]
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        train_targets: List[np.ndarray] = []
        train_probabilities: List[np.ndarray] = []
        train_loss_masks: List[np.ndarray] = []

        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{cfg.epochs}", leave=False)
        for batch_index, batch in enumerate(progress):
            audio = batch["audio"].to(device, non_blocking=True)
            target = batch["target"].to(device, non_blocking=True)
            loss_mask = batch["loss_mask"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                output = model(audio)
                logits = output["segmentwise_logits"]
                loss = criterion(logits, target, loss_mask)
                scaled_loss = loss / cfg.gradient_accumulation_steps

            scaler.scale(scaled_loss).backward()
            should_step = (
                (batch_index + 1) % cfg.gradient_accumulation_steps == 0
                or (batch_index + 1) == len(train_loader)
            )
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema_model is not None:
                    update_ema_model(ema_model, model, cfg.ema_decay)

            running_loss += float(loss.item())
            train_targets.append(target.detach().float().cpu().numpy())
            train_probabilities.append(torch.sigmoid(logits.detach()).float().cpu().numpy())
            train_loss_masks.append(loss_mask.detach().float().cpu().numpy())
            progress.set_postfix(loss=f"{running_loss / (batch_index + 1):.4f}")

        train_true = np.concatenate(train_targets, axis=0).reshape(
            -1, len(label_store.instruments)
        )
        train_prob = np.concatenate(train_probabilities, axis=0).reshape(
            -1, len(label_store.instruments)
        )
        train_valid_mask = np.concatenate(train_loss_masks, axis=0).reshape(
            -1, len(label_store.instruments)
        )
        _, train_summary = _binary_metrics(
            train_true,
            train_prob,
            label_store.instruments,
            cfg.probability_threshold,
            valid_mask=train_valid_mask,
        )

        validation_model = ema_model if ema_model is not None else model
        validation = evaluate_full_songs(
            model=validation_model,
            val_loader=val_loader,
            label_store=label_store,
            criterion=criterion,
            device=device,
            cfg=cfg,
            description=f"Natural validation (song {cfg.val_song_id}, EMA)" if ema_model is not None else f"Natural validation (song {cfg.val_song_id})",
        )
        val_summary = validation["summary"]
        latest_validation_metric = float(val_summary["macro_balanced_accuracy"])
        selected_only_validation = None
        if selected_only_val_loader is not None:
            selected_only_validation = evaluate_full_songs(
                model=validation_model,
                val_loader=selected_only_val_loader,
                label_store=label_store,
                criterion=criterion,
                device=device,
                cfg=cfg,
                description=(
                    f"Selected-only validation (song {cfg.val_song_id}, EMA)"
                    if ema_model is not None
                    else f"Selected-only validation (song {cfg.val_song_id})"
                ),
            )
        selected_summary = (
            selected_only_validation["summary"]
            if selected_only_validation is not None
            else None
        )

        if scheduler is not None:
            scheduler.step()
        next_learning_rates = [float(group["lr"]) for group in optimizer.param_groups]

        epoch_row: Dict[str, Any] = {
            "epoch": epoch + 1,
            "encoder_frozen": encoder_frozen,
            "validation_weights": "ema" if ema_model is not None else "raw",
            "train_loss": running_loss / max(1, len(train_loader)),
            "train_macro_f1": train_summary["macro_f1"],
            "train_micro_f1": train_summary["micro_f1"],
            "val_loss": val_summary["val_loss"],
            "val_macro_f1": val_summary["macro_f1"],
            "val_micro_f1": val_summary["micro_f1"],
            "val_map": val_summary["macro_average_precision"],
            "val_macro_balanced_accuracy": latest_validation_metric,
            "val_worst_class_balanced_accuracy": val_summary[
                "worst_class_balanced_accuracy"
            ],
            "selected_only_val_loss": (
                selected_summary["val_loss"] if selected_summary is not None else np.nan
            ),
            "selected_only_val_macro_f1": (
                selected_summary["macro_f1"] if selected_summary is not None else np.nan
            ),
            "selected_only_val_macro_balanced_accuracy": (
                selected_summary["macro_balanced_accuracy"]
                if selected_summary is not None
                else np.nan
            ),
            "selected_only_val_worst_class_balanced_accuracy": (
                selected_summary["worst_class_balanced_accuracy"]
                if selected_summary is not None
                else np.nan
            ),
            "encoder_learning_rate": learning_rates_used[0],
            "new_layers_learning_rate": learning_rates_used[1],
            "next_encoder_learning_rate": next_learning_rates[0],
            "next_new_layers_learning_rate": next_learning_rates[1],
        }
        for _, class_row in validation["per_class"].iterrows():
            slug = safe_slug(str(class_row["instrument"]))
            epoch_row[f"val_bacc_{slug}"] = class_row["balanced_accuracy"]
            epoch_row[f"val_f1_{slug}"] = class_row["f1"]
            epoch_row[f"val_recall_{slug}"] = class_row["recall"]
            epoch_row[f"val_inactive_recall_{slug}"] = class_row["inactive_recall"]
        if selected_only_validation is not None:
            for _, class_row in selected_only_validation["per_class"].iterrows():
                slug = safe_slug(str(class_row["instrument"]))
                epoch_row[f"selected_only_val_bacc_{slug}"] = class_row["balanced_accuracy"]
                epoch_row[f"selected_only_val_f1_{slug}"] = class_row["f1"]
                epoch_row[f"selected_only_val_recall_{slug}"] = class_row["recall"]
                epoch_row[f"selected_only_val_inactive_recall_{slug}"] = class_row["inactive_recall"]
        history.append(epoch_row)

        print(
            f"Epoch {epoch + 1:03d} | "
            f"train loss={epoch_row['train_loss']:.4f}, "
            f"macro-F1={epoch_row['train_macro_f1']:.4f} | "
            f"natural BAcc={latest_validation_metric:.4f}, "
            f"F1={epoch_row['val_macro_f1']:.4f} | "
            + (
                f"selected-only BAcc={selected_summary['macro_balanced_accuracy']:.4f}, "
                f"F1={selected_summary['macro_f1']:.4f} | "
                if selected_summary is not None
                else ""
            )
            + f"weights={'EMA' if ema_model is not None else 'raw'} | "
            f"lr={learning_rates_used[0]:.2e}/{learning_rates_used[1]:.2e} | "
            f"encoder_frozen={encoder_frozen}",
            flush=True,
        )
        per_class_text = ", ".join(
            f"{row.instrument}={row.balanced_accuracy:.3f}"
            for row in validation["per_class"].itertuples(index=False)
        )
        print(f"  per-class natural BAcc: {per_class_text}", flush=True)
        if selected_only_validation is not None:
            selected_per_class_text = ", ".join(
                f"{row.instrument}={row.balanced_accuracy:.3f}"
                for row in selected_only_validation["per_class"].itertuples(index=False)
            )
            print(
                f"  per-class selected-only BAcc: {selected_per_class_text}",
                flush=True,
            )

        pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
        validation["per_class"].to_csv(
            run_dir / "latest_val_per_class.csv", index=False
        )
        if selected_only_validation is not None:
            selected_only_validation["per_class"].to_csv(
                run_dir / "latest_selected_only_val_per_class.csv", index=False
            )
        improved = (
            np.isfinite(latest_validation_metric)
            and latest_validation_metric > best_validation_metric + 1e-8
        )
        if improved:
            best_validation_metric = latest_validation_metric
            best_epoch = epoch + 1
            if cfg.save_best_checkpoint:
                save_checkpoint(
                    best_path,
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    epoch,
                    best_validation_metric,
                    history,
                    cfg,
                    label_store,
                    train_song_ids,
                    val_song_ids,
                    ema_model=ema_model,
                )
                validation["per_class"].to_csv(
                    run_dir / "best_natural_bacc_per_class.csv", index=False
                )
                print(
                    f"  Saved new best natural BAcc checkpoint: "
                    f"epoch={best_epoch}, BAcc={best_validation_metric:.4f}",
                    flush=True,
                )
        if cfg.save_last_checkpoint:
            save_recovery_checkpoint(epoch, best_validation_metric)

    completed_epochs = int(history[-1]["epoch"]) if history else start_epoch
    print(
        f"Training loop completed through epoch {completed_epochs}/{cfg.epochs}. "
        f"History rows saved: {len(history)}.",
        flush=True,
    )

    validation_model = ema_model if ema_model is not None else model
    final_validation = evaluate_full_songs(
        model=validation_model,
        val_loader=val_loader,
        label_store=label_store,
        criterion=criterion,
        device=device,
        cfg=cfg,
        description=f"Final natural validation (song {cfg.val_song_id}, EMA)" if ema_model is not None else f"Final natural validation (song {cfg.val_song_id})",
    )
    final_selected_only_validation = None
    if selected_only_val_loader is not None:
        final_selected_only_validation = evaluate_full_songs(
            model=validation_model,
            val_loader=selected_only_val_loader,
            label_store=label_store,
            criterion=criterion,
            device=device,
            cfg=cfg,
            description=(
                f"Final selected-only validation (song {cfg.val_song_id}, EMA)"
                if ema_model is not None
                else f"Final selected-only validation (song {cfg.val_song_id})"
            ),
        )
    return {
        "history": pd.DataFrame(history),
        "final_epoch": completed_epochs,
        "final_validation_metric": latest_validation_metric,
        "best_validation_metric": best_validation_metric,
        "best_epoch": best_epoch,
        "best_checkpoint": best_path if best_path.is_file() else None,
        "recovery_checkpoint": last_path if cfg.save_last_checkpoint else None,
        "validation": final_validation,
        "selected_only_validation": final_selected_only_validation,
        "ema_model": ema_model,
    }




In [ ]:
from __future__ import annotations

# =============================================================================
# EXPORTS AND PLOTS
# =============================================================================


def plot_training_history(history: pd.DataFrame, output_path: Path) -> None:
    if history.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(history["epoch"], history["train_macro_f1"], label="Train macro-F1")
    ax.plot(
        history["epoch"],
        history["val_macro_balanced_accuracy"],
        label="Song-5 validation balanced accuracy",
    )
    ax.plot(
        history["epoch"],
        history["val_macro_f1"],
        label="Song-5 natural validation F1",
    )
    if "selected_only_val_macro_balanced_accuracy" in history.columns:
        ax.plot(
            history["epoch"],
            history["selected_only_val_macro_balanced_accuracy"],
            label="Song-5 selected-only balanced accuracy",
        )
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def plot_validation_probabilities(
    half_second_frame: pd.DataFrame,
    instruments: Sequence[str],
    threshold: float,
    output_dir: Path,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for song_id, song_frame in half_second_frame.groupby("song_id"):
        for instrument in instruments:
            slug = safe_slug(instrument)
            fig, ax = plt.subplots(figsize=(16, 4.5))
            ax.plot(
                song_frame["time_start_sec"],
                song_frame[f"prob_{slug}"],
                linewidth=1.0,
                label="Predicted probability",
            )
            ax.step(
                song_frame["time_start_sec"],
                song_frame[f"label_{slug}"],
                where="post",
                linewidth=1.0,
                label="Automatic stem label",
            )
            ax.axhline(threshold, linestyle="--", linewidth=1.0, label=f"Threshold {threshold:.2f}")
            ax.set_title(f"Song {song_id} — {instrument}")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Activity / probability")
            ax.set_ylim(-0.05, 1.05)
            ax.grid(True, alpha=0.25)
            ax.legend(loc="upper right")
            fig.tight_layout()
            fig.savefig(output_dir / f"song_{song_id}_{slug}.png", dpi=180, bbox_inches="tight")
            plt.close(fig)


def export_final_results(
    result: Mapping[str, Any],
    run_dir: Path,
    label_store: LabelStore,
    cfg: ExperimentConfig,
) -> None:
    history: pd.DataFrame = result["history"]
    validation = result["validation"]
    history.to_csv(run_dir / "training_history.csv", index=False)
    validation["per_class"].to_csv(
        run_dir / "validation_metrics_per_class.csv", index=False
    )
    validation["half_second_predictions"].to_csv(
        run_dir / "validation_predictions_0p5s.csv", index=False
    )
    validation["second_predictions"].to_csv(
        run_dir / "validation_predictions_1s.csv", index=False
    )
    save_json(run_dir / "validation_summary.json", validation["summary"])
    plot_training_history(history, run_dir / "training_scores.png")
    plot_validation_probabilities(
        validation["half_second_predictions"],
        label_store.instruments,
        cfg.probability_threshold,
        run_dir / "validation_plots",
    )
    selected_only_validation = result.get("selected_only_validation")
    if selected_only_validation is not None:
        selected_only_validation["per_class"].to_csv(
            run_dir / "selected_only_validation_metrics_per_class.csv", index=False
        )
        selected_only_validation["half_second_predictions"].to_csv(
            run_dir / "selected_only_validation_predictions_0p5s.csv", index=False
        )
        selected_only_validation["second_predictions"].to_csv(
            run_dir / "selected_only_validation_predictions_1s.csv", index=False
        )
        save_json(
            run_dir / "selected_only_validation_summary.json",
            selected_only_validation["summary"],
        )
        plot_validation_probabilities(
            selected_only_validation["half_second_predictions"],
            label_store.instruments,
            cfg.probability_threshold,
            run_dir / "selected_only_validation_plots",
        )




In [ ]:
from __future__ import annotations

# =============================================================================
# COMPLETE EXPERIMENT
# =============================================================================


def _create_run_directory(cfg: ExperimentConfig) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    classes_slug = "-".join(safe_slug(x) for x in cfg.selected_instruments)
    synthetic_tag = "synthetic" if cfg.use_synthetic_mix else "no_synthetic"
    hard_negative_tag = (
        "hardneg" if cfg.use_hard_negative_backgrounds else "no_hardneg"
    )
    target_off_tag = (
        "targetoff" if cfg.use_target_off_hard_negatives else "no_targetoff"
    )
    sampling_tag = (
        "detquota" if cfg.deterministic_epoch_sampling else "randomsampling"
    )
    isolated_quota_tag = (
        f"stemq{cfg.isolated_positive_samples_per_instrument}"
        if cfg.isolated_positive_samples_per_instrument is not None
        else "stemqratio"
    )
    pretrain_tag = (
        "stempretrain" if cfg.use_isolated_stem_pretraining else "no_stempretrain"
    )
    rms_tag = (
        "rmsbalanced" if cfg.balance_synthetic_target_active_rms else "rawlevels"
    )
    run_name = (
        f"legacy_cnn6_framewise_{cfg.audio_mode}_val{safe_slug(str(cfg.val_song_id))}_family_"
        f"{classes_slug}_legacyfreqbn_maxmean_{sampling_tag}_"
        f"{isolated_quota_tag}_{pretrain_tag}_{rms_tag}_songbalanced_ema_"
        f"{synthetic_tag}_{hard_negative_tag}_{target_off_tag}_{timestamp}"
    )
    run_dir = Path(cfg.drive_output_root) / run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def run_experiment(cfg: ExperimentConfig) -> Dict[str, Any]:
    validate_config(cfg)
    set_global_seed(cfg.seed)
    mode = AUDIO_MODES[cfg.audio_mode]

    drive_44_root = Path(cfg.drive_44_root)
    drive_96_root = Path(cfg.drive_96_root)
    source_root = drive_44_root if mode.source_kind == "44" else drive_96_root

    print("=" * 80)
    print("POLYPHONIC INSTRUMENT ACTIVITY EXPERIMENT")
    print("=" * 80)
    print(f"Audio mode:          {cfg.audio_mode}")
    print(f"Selected instruments:{list(cfg.selected_instruments)}")
    print(f"Validation family:   {cfg.val_song_id} (all matching decimal derivatives)")
    print(f"Source root:         {source_root}")

    hard_negative_mapping = (
        active_hard_negative_mapping(cfg)
        if cfg.use_hard_negative_backgrounds
        else {}
    )
    hard_negative_instruments = (
        configured_hard_negative_instruments(cfg)
        if cfg.use_hard_negative_backgrounds
        else ()
    )
    need_background_stems = (
        (cfg.use_synthetic_mix and cfg.use_synthetic_background_instruments)
        or (
            cfg.use_controlled_dropout_validation
            and cfg.use_controlled_polyphonic_validation
        )
        or (cfg.use_hard_negative_backgrounds and bool(hard_negative_mapping))
        or bool(cfg.controlled_validation_background_instruments)
        or (len(cfg.selected_instruments) == 1 and bool(cfg.isolated_pretrain_negative_instruments))
    )
    source_catalog = scan_source_catalog(
        source_root,
        cfg.selected_instruments,
        include_background_instruments=need_background_stems,
    )
    # Use one common 96 kHz master-label source for every bandwidth condition.
    # This prevents the target labels themselves from changing between 44.1
    # and 96 kHz experiments.
    label_source_root = drive_96_root
    label_instruments = tuple(
        dict.fromkeys((*cfg.selected_instruments, *hard_negative_instruments))
    )
    label_source_catalog = scan_source_catalog(
        label_source_root,
        label_instruments,
    )

    detected_song_ids = sorted(source_catalog.mixes, key=song_id_sort_key)
    cfg.val_song_id = canonical_song_id(cfg.val_song_id)
    song_ids = (
        sorted((canonical_song_id(song_id) for song_id in cfg.song_ids), key=song_id_sort_key)
        if cfg.song_ids is not None
        else detected_song_ids
    )
    missing_mixes = [song_id for song_id in song_ids if song_id not in source_catalog.mixes]
    if missing_mixes:
        raise FileNotFoundError(f"Mix files are missing for song IDs: {missing_mixes}")
    if cfg.val_song_id not in song_ids:
        raise ValueError(f"val_song_id={cfg.val_song_id} is not in selected song_ids={song_ids}")

    # Keep every derivative of one recording session in the same split.
    # Thus val=2 holds out 2, 2.2, 2.3, ... together; choose another
    # validation family if the whole family 2 should be used for training.
    validation_family = song_family_id(cfg.val_song_id)
    val_song_ids = [
        song_id for song_id in song_ids if song_family_id(song_id) == validation_family
    ]
    train_song_ids = [
        song_id for song_id in song_ids if song_family_id(song_id) != validation_family
    ]
    if not train_song_ids:
        raise ValueError("At least one training song is required.")

    print(f"Detected/selected songs: {song_ids}")
    print(f"Train songs:             {train_song_ids}")
    print(f"Validation family songs: {val_song_ids}")
    print_source_availability(
        source_catalog,
        cfg.selected_instruments,
        song_ids,
        title=f"Stem availability in {cfg.audio_mode} source",
    )
    print_source_availability(
        label_source_catalog,
        cfg.selected_instruments,
        song_ids,
        title="Stem availability in 96 kHz label source",
    )
    if hard_negative_instruments:
        print_source_availability(
            label_source_catalog,
            hard_negative_instruments,
            song_ids,
            title="Hard-negative stem availability in 96 kHz label source",
        )

    local_catalog = prepare_local_catalog(
        source_catalog=source_catalog,
        song_ids=song_ids,
        mode=mode,
        cfg=cfg,
    )
    background_summary = validate_synthetic_background_availability(
        local_catalog,
        song_ids,
        cfg.selected_instruments,
        cfg,
    )

    run_dir = _create_run_directory(cfg)
    background_summary.to_csv(
        run_dir / "synthetic_background_availability.csv", index=False
    )
    save_json(run_dir / "config.json", asdict(cfg))
    save_json(
        run_dir / "dataset_manifest.json",
        {
            "audio_mode": cfg.audio_mode,
            "source_root": str(source_root),
            "label_source_root": str(label_source_root),
            "local_cache_root": str(local_catalog.root),
            "song_ids": song_ids,
            "train_song_ids": train_song_ids,
            "val_song_ids": val_song_ids,
            "selected_instruments": list(cfg.selected_instruments),
            "controlled_validation_background_instruments": list(
                cfg.controlled_validation_background_instruments
            ),
            "isolated_pretrain_negative_instruments": list(
                cfg.isolated_pretrain_negative_instruments
            ),
            "hard_negative_mapping": hard_negative_mapping,
            "hard_negative_instruments": list(hard_negative_instruments),
            "target_off_hard_negative_instruments": list(
                cfg.target_off_hard_negative_instruments
            ),
            "mixes": {song_id: str(path) for song_id, path in local_catalog.mixes.items()},
            "stems": {
                instrument: {song_id: str(path) for song_id, path in paths.items()}
                for instrument, paths in local_catalog.stems.items()
            },
        },
    )

    label_cache_root = Path(cfg.drive_output_root) / "_automatic_label_cache"
    label_store, label_summary = build_label_store(
        label_source_catalog=label_source_catalog,
        local_catalog=local_catalog,
        song_ids=song_ids,
        instruments=cfg.selected_instruments,
        label_cache_root=label_cache_root,
        cfg=cfg,
    )
    label_summary.to_csv(run_dir / "automatic_label_summary.csv", index=False)

    hard_negative_label_store: Optional[LabelStore] = None
    hard_negative_label_summary = pd.DataFrame()
    if cfg.use_hard_negative_backgrounds and hard_negative_instruments:
        hard_negative_label_store, hard_negative_label_summary = build_label_store(
            label_source_catalog=label_source_catalog,
            local_catalog=local_catalog,
            song_ids=song_ids,
            instruments=hard_negative_instruments,
            label_cache_root=label_cache_root,
            cfg=cfg,
        )
        hard_negative_label_summary.to_csv(
            run_dir / "hard_negative_label_summary.csv", index=False
        )

    # Hard preflight: every selected output must have both ON and OFF labels
    # in natural validation, otherwise balanced accuracy is not defined.
    invalid_validation_classes: List[str] = []
    for instrument, class_index in label_store.class_to_index.items():
        positive_train_songs = [
            song_id
            for song_id in train_song_ids
            if np.any(label_store.song_labels[song_id][:, class_index] > 0.5)
        ]
        val_class_labels = np.concatenate(
            [label_store.song_labels[song_id][:, class_index] for song_id in val_song_ids]
        )
        positive_val = bool(np.any(val_class_labels > 0.5))
        negative_val = bool(np.any(val_class_labels <= 0.5))
        if len(positive_train_songs) <= 1:
            warnings.warn(
                f"{instrument} has positive training activity in only {len(positive_train_songs)} "
                f"song(s): {positive_train_songs}"
            )
        if not (positive_val and negative_val):
            missing = []
            if not positive_val:
                missing.append("ON")
            if not negative_val:
                missing.append("OFF")
            invalid_validation_classes.append(
                f"{instrument} (missing {'/'.join(missing)})"
            )
    if invalid_validation_classes:
        raise RuntimeError(
            "Natural validation is invalid for balanced accuracy in family "
            f"{val_song_ids}: " + ", ".join(invalid_validation_classes)
        )
    print("Validation preflight passed: every selected instrument has ON and OFF labels.")

    train_descriptors, val_descriptors, descriptor_summary = build_descriptors(
        local_catalog=local_catalog,
        label_store=label_store,
        train_song_ids=train_song_ids,
        val_song_ids=val_song_ids,
        cfg=cfg,
    )
    descriptor_summary.to_csv(run_dir / "descriptor_summary.csv", index=False)
    hard_negative_eligibility_summary = summarize_hard_negative_eligibility(
        train_descriptors,
        local_catalog,
        label_store,
        hard_negative_label_store,
        cfg,
    )
    if len(hard_negative_eligibility_summary):
        hard_negative_eligibility_summary.to_csv(
            run_dir / "hard_negative_eligibility_summary.csv", index=False
        )

    train_dropout_plan = None
    if cfg.use_synthetic_mix and cfg.use_counterfactual_stem_dropout:
        train_dropout_plan = build_counterfactual_dropout_plan(
            label_store,
            local_catalog,
            train_song_ids,
            cfg,
            name="train",
            seed_offset=0,
        )
        train_dropout_plan.summary.to_json(
            run_dir / "train_counterfactual_dropout_plan.json",
            orient="records",
            indent=2,
        )

    train_descriptors = add_target_off_training_descriptors(
        train_descriptors,
        local_catalog,
        label_store,
        train_dropout_plan,
        cfg,
    )
    descriptor_counts = Counter(
        descriptor.source_type for descriptor in train_descriptors
    )
    descriptor_summary = pd.DataFrame(
        [
            {"source_type": source_type, "sequence_count": count}
            for source_type, count in descriptor_counts.items()
        ]
    )
    descriptor_summary.to_csv(run_dir / "descriptor_summary.csv", index=False)

    target_off_hard_negative_summary = (
        summarize_target_off_hard_negative_eligibility(
            train_descriptors,
            local_catalog,
            label_store,
            train_dropout_plan,
            cfg,
        )
    )
    if len(target_off_hard_negative_summary):
        target_off_hard_negative_summary.to_csv(
            run_dir / "target_off_hard_negative_summary.csv", index=False
        )

    train_dataset = InstrumentSequenceDataset(
        train_descriptors,
        local_catalog,
        label_store,
        cfg,
        dropout_plan=train_dropout_plan,
        hard_negative_label_store=hard_negative_label_store,
    )
    val_dataset = InstrumentSequenceDataset(
        val_descriptors, local_catalog, label_store, cfg
    )
    train_loader, val_loader = make_data_loaders(
        train_dataset, val_dataset, train_descriptors, cfg
    )
    selected_only_val_loader = None
    if cfg.use_selected_only_validation:
        selected_only_val_loader = make_selected_only_validation_loader(
            val_descriptors, local_catalog, label_store, cfg
        )
    if hasattr(train_loader.sampler, "quota_rows"):
        pd.DataFrame(train_loader.sampler.quota_rows()).to_csv(
            run_dir / "source_song_sampling_quotas.csv", index=False
        )




    on_weight, off_weight, balance_summary = calculate_class_balance_weights(
        train_descriptors,
        local_catalog,
        label_store,
        train_dropout_plan,
        cfg,
        sampler=train_loader.sampler,
    )
    balance_summary.to_csv(
        run_dir / "expected_training_class_balance.csv", index=False
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nDevice: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    automatic_pretraining_checkpoint: Optional[Path] = None
    if cfg.resume_checkpoint:
        print(
            "Main-training resume selected: isolated pretraining is not repeated; "
            "the recovery checkpoint already contains the encoder weights."
        )
    elif cfg.use_isolated_stem_pretraining and not cfg.dry_run:
        automatic_pretraining_checkpoint = train_or_load_isolated_stem_encoder(
            local_catalog,
            label_store,
            train_song_ids,
            device,
            cfg,
        )
        save_json(
            run_dir / "isolated_pretraining_reference.json",
            {"checkpoint": str(automatic_pretraining_checkpoint)},
        )
    elif cfg.use_isolated_stem_pretraining and cfg.dry_run:
        print("Dry run: isolated pretraining is configured but intentionally skipped.")

    model = make_model(cfg, local_catalog.sample_rate, len(cfg.selected_instruments))
    if cfg.resume_checkpoint:
        transferred_tensors = 0
    else:
        selected_pretraining_checkpoint = (
            str(automatic_pretraining_checkpoint)
            if automatic_pretraining_checkpoint is not None
            else cfg.pretrained_stem_checkpoint
        )
        transferred_tensors = load_pretrained_stem_encoder(
            model,
            selected_pretraining_checkpoint,
            cfg.transfer_fc1,
        )

    # One batch and one forward pass catch path/shape errors before a long run.
    diagnostic_batch = next(iter(train_loader))
    print("\nDiagnostic batch:")
    print(f"  audio shape:  {tuple(diagnostic_batch['audio'].shape)}")
    print(f"  target shape: {tuple(diagnostic_batch['target'].shape)}")
    model.to(device)
    model.eval()
    with torch.no_grad():
        diagnostic_output = model(diagnostic_batch["audio"].to(device))
    print(f"  output shape: {tuple(diagnostic_output['segmentwise_logits'].shape)}")
    expected_shape = tuple(diagnostic_batch["target"].shape)
    if tuple(diagnostic_output["segmentwise_logits"].shape) != expected_shape:
        raise RuntimeError(
            f"Model output {tuple(diagnostic_output['segmentwise_logits'].shape)} "
            f"does not match target {expected_shape}."
        )
    del diagnostic_batch, diagnostic_output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if cfg.dry_run:
        print(f"\nDry run completed successfully. Run directory: {run_dir}")
        return {
            "run_dir": run_dir,
            "label_store": label_store,
            "hard_negative_label_store": hard_negative_label_store,
            "hard_negative_label_summary": hard_negative_label_summary,
            "hard_negative_eligibility_summary": hard_negative_eligibility_summary,
            "target_off_hard_negative_summary": target_off_hard_negative_summary,
            "model": model,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "selected_only_val_loader": selected_only_val_loader,
            "train_dropout_plan": train_dropout_plan,
            "class_balance_summary": balance_summary,
            "on_weight": on_weight,
            "off_weight": off_weight,
            "isolated_pretraining_checkpoint": automatic_pretraining_checkpoint,
        }

    result = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        selected_only_val_loader=selected_only_val_loader,
        label_store=label_store,
        on_weight=on_weight,
        off_weight=off_weight,
        device=device,
        run_dir=run_dir,
        transferred_tensors=transferred_tensors,
        train_song_ids=train_song_ids,
        val_song_ids=val_song_ids,
        cfg=cfg,
    )
    export_final_results(result, run_dir, label_store, cfg)

    print("\nFinal validation summary:")
    print(json.dumps(json_ready(result["validation"]["summary"]), indent=2))
    print("\nPer-class natural validation metrics:")
    print(result["validation"]["per_class"].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    if result.get("selected_only_validation") is not None:
        print("\nFinal selected-only validation summary:")
        print(
            json.dumps(
                json_ready(result["selected_only_validation"]["summary"]),
                indent=2,
            )
        )
        print("\nPer-class selected-only validation metrics:")
        print(
            result["selected_only_validation"]["per_class"].to_string(
                index=False, float_format=lambda x: f"{x:.4f}"
            )
        )

    print(f"\nAll outputs saved to: {run_dir}")

    result = dict(result)
    result["run_dir"] = run_dir
    result["label_store"] = label_store
    return result


# In Colab, run the following in the final cell after editing CFG:
#
# results = run_experiment(CFG)


## 3. בניית הקונפיגורציה

התא הבא מחבר אוטומטית את הבחירות מראש המחברת לפייפליין. בדרך כלל אין צורך לערוך אותו.


In [ ]:
CFG = ExperimentConfig(
    # ---------- Drive paths ----------
    drive_44_root="/content/drive/MyDrive/train_44",
    drive_96_root="/content/drive/MyDrive/train_96",
    drive_output_root="/content/drive/MyDrive/instrument_mix_runs",
    local_cache_root="/content/instrument_mix_audio_cache",
    use_local_audio_cache=True,

    # ---------- Experiment ----------
    audio_mode=AUDIO_MODE,
    seed=RUN_SEED,

    model_architecture=MODEL_ARCHITECTURE,

    selected_instruments=SELECTED_INSTRUMENTS,
    val_song_id=VAL_SONG_ID,
    use_selected_only_validation=USE_SELECTED_ONLY_VALIDATION,
    controlled_validation_background_instruments=(
        CONTROLLED_VALIDATION_BACKGROUND_INSTRUMENTS
    ),
    song_ids=(1, 2, 3, 4, 5, 6),  # Exclude 2.2 completely from training/evaluation.

    # ---------- Source balance ----------
    use_real_mix=True,
    use_stems=True,
    use_synthetic_mix=True,
    real_mix_ratio_with_synthetic=REAL_MIX_RATIO,
    stem_ratio_with_synthetic=ISOLATED_STEM_RATIO,
    synthetic_mix_ratio=SYNTHETIC_MIX_RATIO,
    use_synthetic_background_instruments=True,
    synthetic_background_probability=SYNTHETIC_BACKGROUND_PROBABILITY,
    synthetic_background_min_instruments=1,
    synthetic_background_max_instruments=6,
    synthetic_min_total_instruments=SYNTHETIC_MIN_TOTAL_INSTRUMENTS,
    balance_synthetic_target_active_rms=BALANCE_SYNTHETIC_TARGET_ACTIVE_RMS,
    synthetic_target_reference_dbfs=SYNTHETIC_TARGET_REFERENCE_DBFS,
    synthetic_target_random_gain_db_min=SYNTHETIC_TARGET_RANDOM_GAIN_DB_MIN,
    synthetic_target_random_gain_db_max=SYNTHETIC_TARGET_RANDOM_GAIN_DB_MAX,
    synthetic_target_max_adjustment_db=SYNTHETIC_TARGET_MAX_ADJUSTMENT_DB,

    target_off_regular_synthetic_ratio=TARGET_OFF_REGULAR_RATIO,
    target_off_fully_off_ratio=TARGET_OFF_FULLY_OFF_RATIO,
    target_off_partially_off_ratio=TARGET_OFF_PARTIALLY_OFF_RATIO,
    use_target_off_hard_negatives=USE_TARGET_OFF_HARD_NEGATIVES,
    target_off_hard_negative_instruments=TARGET_OFF_INSTRUMENTS,
    target_off_hard_negative_probability=TARGET_OFF_BACKGROUND_PROBABILITY,
    target_off_hard_negative_min_seconds=TARGET_OFF_MIN_SECONDS,
    use_hard_negative_backgrounds=USE_CONFUSER_HARD_NEGATIVES,
    hard_negative_confusers=CONFUSER_HARD_NEGATIVE_MAPPING,

    use_counterfactual_stem_dropout=USE_COUNTERFACTUAL_STEM_DROPOUT,
    counterfactual_min_original_activity_fraction=0.70,
    counterfactual_target_activity_fraction=0.65,
    counterfactual_min_interval_seconds=5.0,
    counterfactual_max_interval_seconds=60.0,
    counterfactual_min_intervals=3,
    synthetic_background_relative_db_min=-6.0,
    synthetic_background_relative_db_max=0.0,

    # Only the untouched natural mix of the selected validation family is evaluated.
    use_controlled_dropout_validation=False,
    use_controlled_polyphonic_validation=False,
    use_class_balance_weights=USE_CLASS_BALANCE_WEIGHTS,
    max_class_balance_weight=MAX_CLASS_BALANCE_WEIGHT,

    # The same PANNs CNN6 is first trained on strict active half-second
    # isolated clips, then transferred into the CNN-only polyphonic model.
    use_isolated_stem_pretraining=USE_ISOLATED_STEM_PRETRAINING,
    activity_threshold_dbfs=ACTIVITY_THRESHOLD_DBFS,
    isolated_pretrain_clip_seconds=ISOLATED_PRETRAIN_CLIP_SECONDS,
    isolated_pretrain_epochs=ISOLATED_PRETRAIN_EPOCHS,
    isolated_pretrain_samples_per_instrument=ISOLATED_PRETRAIN_SAMPLES_PER_INSTRUMENT,
    isolated_pretrain_batch_size=ISOLATED_PRETRAIN_BATCH_SIZE,
    isolated_pretrain_learning_rate=ISOLATED_PRETRAIN_LEARNING_RATE,
    isolated_pretrain_random_gain_db_min=ISOLATED_PRETRAIN_RANDOM_GAIN_DB_MIN,
    isolated_pretrain_random_gain_db_max=ISOLATED_PRETRAIN_RANDOM_GAIN_DB_MAX,
    isolated_pretrain_reuse_checkpoint=ISOLATED_PRETRAIN_REUSE_CHECKPOINT,
    isolated_pretrain_force_rebuild=ISOLATED_PRETRAIN_FORCE_REBUILD,
    isolated_pretrain_negative_instruments=(
        ISOLATED_PRETRAIN_NEGATIVE_INSTRUMENTS
    ),
    pretrained_stem_checkpoint=None,
    transfer_fc1=True,
    freeze_pretrained_encoder_epochs=PRETRAINED_ENCODER_FREEZE_EPOCHS,

    # ---------- Runtime and stability ----------
    batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    train_samples_per_epoch=TRAIN_SAMPLES_PER_EPOCH,
    isolated_positive_samples_per_instrument=(
        ISOLATED_POSITIVE_SAMPLES_PER_INSTRUMENT
    ),
    isolated_stem_target_only_loss=ISOLATED_STEM_TARGET_ONLY_LOSS,
    deterministic_epoch_sampling=True,
    class_aware_sampling=False,
    encoder_learning_rate=ENCODER_LEARNING_RATE,
    new_layers_learning_rate=NEW_LAYERS_LEARNING_RATE,
    epochs=50,
    use_early_stopping=False,
    use_lr_scheduler=True,
    warmup_epochs=WARMUP_EPOCHS,
    lr_hold_epochs=LR_HOLD_EPOCHS,
    use_ema=USE_EMA,
    ema_decay=EMA_DECAY,
    num_workers=DATA_LOADER_WORKERS,
    prefetch_factor=DATA_LOADER_PREFETCH,
    persistent_workers=False,
    pin_memory=True,
    use_amp=True,
    save_last_checkpoint=True,
    save_best_checkpoint=False,  # Fixed final epoch only; never select on validation.
    resume_checkpoint=None,
    dry_run=RUN_DRY_RUN_BEFORE_TRAINING,
)

CFG = configure_auto_resume(CFG, AUTO_RESUME)
CFG


## אבחון תוויות וסנכרון לפני אימון

התא הבא משווה את תוויות הגיטרה הקלאסית ב־44.1 וב־96 קילוהרץ על אותה
רצועה, מחפש היסט של עד 10 שניות, ובודק חפיפה עם גיטרה אקוסטית וחשמלית.
הטבלאות, הגרף וקובצי ה־CSV נשמרים בתיקיית `_diagnostics` בדרייב.


In [ ]:
def _alignment_views(
    values_44: np.ndarray,
    values_96: np.ndarray,
    lag_steps_96_relative_to_44: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Align two step sequences. Positive lag means the 96k events occur later."""
    lag = int(lag_steps_96_relative_to_44)
    if lag >= 0:
        start_44, start_96 = 0, lag
    else:
        start_44, start_96 = -lag, 0
    count = min(len(values_44) - start_44, len(values_96) - start_96)
    if count <= 0:
        return values_44[:0], values_96[:0]
    return (
        np.asarray(values_44[start_44 : start_44 + count]),
        np.asarray(values_96[start_96 : start_96 + count]),
    )


def _binary_pair_metrics(reference: np.ndarray, comparison: np.ndarray) -> Dict[str, float]:
    reference = np.asarray(reference).astype(bool)
    comparison = np.asarray(comparison).astype(bool)
    tp = int(np.sum(reference & comparison))
    tn = int(np.sum(~reference & ~comparison))
    fp = int(np.sum(~reference & comparison))
    fn = int(np.sum(reference & ~comparison))
    positives = tp + fn
    negatives = tn + fp
    sensitivity = tp / positives if positives else np.nan
    specificity = tn / negatives if negatives else np.nan
    balanced_accuracy = (
        0.5 * (sensitivity + specificity)
        if np.isfinite(sensitivity) and np.isfinite(specificity)
        else np.nan
    )
    denominator = math.sqrt(
        float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    )
    mcc = ((tp * tn) - (fp * fn)) / denominator if denominator > 0 else np.nan
    total = tp + tn + fp + fn
    return {
        "compared_steps": int(total),
        "label_agreement": (tp + tn) / total if total else np.nan,
        "label_balanced_accuracy": balanced_accuracy,
        "label_mcc": mcc,
        "true_positive": tp,
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
    }


def _rms_correlation(left: np.ndarray, right: np.ndarray) -> float:
    left = np.clip(
        np.nan_to_num(np.asarray(left, dtype=np.float64), nan=-120.0, neginf=-120.0, posinf=0.0),
        -120.0,
        0.0,
    )
    right = np.clip(
        np.nan_to_num(np.asarray(right, dtype=np.float64), nan=-120.0, neginf=-120.0, posinf=0.0),
        -120.0,
        0.0,
    )
    if len(left) < 3 or np.std(left) <= 1e-12 or np.std(right) <= 1e-12:
        return np.nan
    return float(np.corrcoef(left, right)[0, 1])


def run_alignment_and_confuser_diagnostic(
    cfg: ExperimentConfig,
    song_id: SongId,
    target_instrument: str,
    confuser_instruments: Sequence[str],
    max_lag_seconds: float,
) -> Dict[str, Any]:
    song_id = canonical_song_id(song_id)
    diagnostic_instruments = tuple(dict.fromkeys((target_instrument, *confuser_instruments)))

    print("=" * 80)
    print("44.1/96 kHz LABEL-ALIGNMENT DIAGNOSTIC")
    print("=" * 80)
    print(f"Song:       {song_id}")
    print(f"Target:     {target_instrument}")
    print(f"Confusers:  {tuple(confuser_instruments)}")
    print(f"Threshold:  {cfg.activity_threshold_dbfs:.1f} dBFS")
    print(f"Label hop:  {cfg.label_hop_seconds:.3f} s")

    root_44 = Path(cfg.drive_44_root)
    root_96 = Path(cfg.drive_96_root)
    catalog_44 = scan_source_catalog(root_44, diagnostic_instruments)
    catalog_96 = scan_source_catalog(root_96, diagnostic_instruments)

    if song_id not in catalog_44.mixes or song_id not in catalog_96.mixes:
        raise FileNotFoundError(f"Song {song_id} mix must exist in both train_44 and train_96.")
    target_44_path = catalog_44.stems[target_instrument].get(song_id)
    target_96_path = catalog_96.stems[target_instrument].get(song_id)
    if target_44_path is None or target_96_path is None:
        raise FileNotFoundError(
            f"{target_instrument}, song {song_id}, must exist in both train_44 and train_96."
        )

    duration_44 = audio_duration(catalog_44.mixes[song_id])
    duration_96 = audio_duration(catalog_96.mixes[song_id])
    common_duration = min(duration_44, duration_96)
    print(f"44.1 mix duration: {duration_44:.3f} s")
    print(f"96k mix duration:  {duration_96:.3f} s")
    print(f"Compared duration: {common_duration:.3f} s")
    print(f"44.1 target stem:  {target_44_path}")
    print(f"96k target stem:   {target_96_path}")

    # Recompute directly from the two source trees; no cached 96k labels are reused.
    labels_44, rms_44 = compute_activity_labels(target_44_path, common_duration, cfg)
    labels_96, rms_96 = compute_activity_labels(target_96_path, common_duration, cfg)

    max_lag_steps = int(round(float(max_lag_seconds) / cfg.label_hop_seconds))
    lag_rows: List[Dict[str, Any]] = []
    for lag_steps in range(-max_lag_steps, max_lag_steps + 1):
        aligned_rms_44, aligned_rms_96 = _alignment_views(rms_44, rms_96, lag_steps)
        aligned_labels_44, aligned_labels_96 = _alignment_views(labels_44, labels_96, lag_steps)
        row: Dict[str, Any] = {
            "lag_steps_96_relative_to_44": lag_steps,
            "lag_seconds_96_relative_to_44": lag_steps * cfg.label_hop_seconds,
            "rms_correlation": _rms_correlation(aligned_rms_44, aligned_rms_96),
        }
        row.update(_binary_pair_metrics(aligned_labels_44, aligned_labels_96))
        lag_rows.append(row)
    lag_scan = pd.DataFrame(lag_rows)

    zero_lag = lag_scan.loc[
        lag_scan["lag_steps_96_relative_to_44"] == 0
    ].iloc[0]
    finite_corr = lag_scan[np.isfinite(lag_scan["rms_correlation"])]
    finite_mcc = lag_scan[np.isfinite(lag_scan["label_mcc"])]
    best_rms = (
        finite_corr.loc[finite_corr["rms_correlation"].idxmax()]
        if len(finite_corr)
        else zero_lag
    )
    best_mcc = (
        finite_mcc.loc[finite_mcc["label_mcc"].idxmax()]
        if len(finite_mcc)
        else zero_lag
    )

    comparison_table = pd.DataFrame(
        [
            {"candidate": "zero lag", **zero_lag.to_dict()},
            {"candidate": "best RMS correlation", **best_rms.to_dict()},
            {"candidate": "best label MCC", **best_mcc.to_dict()},
        ]
    )[
        [
            "candidate",
            "lag_seconds_96_relative_to_44",
            "rms_correlation",
            "label_agreement",
            "label_balanced_accuracy",
            "label_mcc",
            "compared_steps",
        ]
    ]
    print("\nAlignment candidates (positive lag = 96k occurs later):")
    print(comparison_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(
        f"\nTarget ON fraction: 44.1k={labels_44.mean():.3f}, "
        f"96k={labels_96.mean():.3f}"
    )

    common_steps = min(len(labels_44), len(labels_96), len(rms_44), len(rms_96))
    labels_44_common = labels_44[:common_steps].astype(bool)
    labels_96_common = labels_96[:common_steps].astype(bool)
    rms_44_common = rms_44[:common_steps]
    rms_96_common = rms_96[:common_steps]
    times = (np.arange(common_steps, dtype=np.float64) + 1.0) * cfg.label_hop_seconds

    confuser_labels: Dict[str, np.ndarray] = {}
    confuser_rms: Dict[str, np.ndarray] = {}
    for instrument in confuser_instruments:
        path = catalog_96.stems[instrument].get(song_id)
        if path is None:
            warnings.warn(f"No {instrument} stem for song {song_id}; treating it as always OFF.")
            confuser_labels[instrument] = np.zeros(common_steps, dtype=bool)
            confuser_rms[instrument] = np.full(common_steps, -np.inf, dtype=np.float32)
            continue
        labels, rms = compute_activity_labels(path, common_duration, cfg)
        padded_labels = np.zeros(common_steps, dtype=bool)
        padded_rms = np.full(common_steps, -np.inf, dtype=np.float32)
        count = min(common_steps, len(labels), len(rms))
        padded_labels[:count] = labels[:count].astype(bool)
        padded_rms[:count] = rms[:count]
        confuser_labels[instrument] = padded_labels
        confuser_rms[instrument] = padded_rms

    any_confuser_on = np.zeros(common_steps, dtype=bool)
    for values in confuser_labels.values():
        any_confuser_on |= values

    overlap_rows: List[Dict[str, Any]] = []

    def add_overlap(label: str, mask: np.ndarray) -> None:
        step_count = int(np.sum(mask))
        overlap_rows.append(
            {
                "condition": label,
                "steps": step_count,
                "seconds": step_count * cfg.label_hop_seconds,
                "percent_of_song": 100.0 * step_count / common_steps if common_steps else np.nan,
            }
        )

    add_overlap(f"{target_instrument} ON (96k label)", labels_96_common)
    add_overlap(f"{target_instrument} OFF (96k label)", ~labels_96_common)
    add_overlap("Any configured confuser ON", any_confuser_on)
    add_overlap(
        f"{target_instrument} OFF AND any confuser ON",
        (~labels_96_common) & any_confuser_on,
    )
    add_overlap(
        f"{target_instrument} ON AND any confuser ON",
        labels_96_common & any_confuser_on,
    )
    add_overlap(
        f"{target_instrument} OFF AND all confusers OFF",
        (~labels_96_common) & (~any_confuser_on),
    )
    for instrument, values in confuser_labels.items():
        add_overlap(f"{target_instrument} OFF AND {instrument} ON", (~labels_96_common) & values)

    overlap_table = pd.DataFrame(overlap_rows)
    print("\nConfuser overlap on the 96k master timeline:")
    print(overlap_table.to_string(index=False, float_format=lambda value: f"{value:.2f}"))

    diagnostic_root = Path(cfg.drive_output_root) / "_diagnostics"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = diagnostic_root / (
        f"alignment_44_vs_96_{safe_slug(target_instrument)}_"
        f"song_{safe_slug(str(song_id))}_{timestamp}"
    )
    output_dir.mkdir(parents=True, exist_ok=False)

    time_series_data: Dict[str, Any] = {
        "time_seconds": times,
        f"{safe_slug(target_instrument)}_44_rms_dbfs": rms_44_common,
        f"{safe_slug(target_instrument)}_44_active": labels_44_common.astype(np.uint8),
        f"{safe_slug(target_instrument)}_96_rms_dbfs": rms_96_common,
        f"{safe_slug(target_instrument)}_96_active": labels_96_common.astype(np.uint8),
    }
    for instrument in confuser_instruments:
        slug = safe_slug(instrument)
        time_series_data[f"{slug}_96_rms_dbfs"] = confuser_rms[instrument]
        time_series_data[f"{slug}_96_active"] = confuser_labels[instrument].astype(np.uint8)
    time_series = pd.DataFrame(time_series_data)

    lag_scan.to_csv(output_dir / "lag_scan.csv", index=False)
    comparison_table.to_csv(output_dir / "alignment_candidates.csv", index=False)
    overlap_table.to_csv(output_dir / "confuser_overlap.csv", index=False)
    time_series.to_csv(output_dir / "activity_timeline_0p5s.csv", index=False)

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, constrained_layout=True)
    axes[0].plot(times, rms_44_common, linewidth=0.8, label=f"{target_instrument} 44.1k")
    axes[0].plot(times, rms_96_common, linewidth=0.8, alpha=0.8, label=f"{target_instrument} 96k")
    axes[0].axhline(
        cfg.activity_threshold_dbfs,
        color="black",
        linestyle="--",
        linewidth=1.0,
        label=f"threshold {cfg.activity_threshold_dbfs:.1f} dBFS",
    )
    axes[0].set_ylabel("RMS (dBFS)")
    axes[0].set_title(f"Song {song_id}: 44.1/96 kHz label-source comparison")
    axes[0].legend(loc="upper right")
    axes[0].grid(alpha=0.2)

    activity_series: List[Tuple[str, np.ndarray]] = [
        (f"{target_instrument} 44.1k", labels_44_common),
        (f"{target_instrument} 96k", labels_96_common),
    ]
    activity_series.extend((f"{name} 96k", values) for name, values in confuser_labels.items())
    offsets = np.arange(len(activity_series), dtype=np.float64) * 1.4
    for offset, (name, values) in zip(offsets, activity_series):
        axes[1].step(times, values.astype(float) + offset, where="post", linewidth=0.9, label=name)
    axes[1].set_yticks(offsets + 0.5)
    axes[1].set_yticklabels([name for name, _ in activity_series])
    axes[1].set_ylim(-0.2, offsets[-1] + 1.2)
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Automatic activity")
    axes[1].grid(axis="x", alpha=0.2)
    figure_path = output_dir / "alignment_and_confuser_activity.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    zero_corr = float(zero_lag["rms_correlation"])
    best_corr = float(best_rms["rms_correlation"])
    best_lag_seconds = float(best_rms["lag_seconds_96_relative_to_44"])
    corr_gain = best_corr - zero_corr if np.isfinite(zero_corr) and np.isfinite(best_corr) else np.nan
    if abs(best_lag_seconds) >= cfg.label_hop_seconds and np.isfinite(corr_gain) and corr_gain >= 0.05:
        print(
            f"\nWARNING: a systematic offset is plausible: best RMS lag={best_lag_seconds:+.2f}s "
            f"and correlation improves by {corr_gain:.3f}."
        )
    elif float(zero_lag["label_mcc"]) < 0.80:
        print(
            "\nNo strong lag warning was triggered, but the zero-lag labels disagree. "
            "Inspect the RMS plot, gain difference, threshold, and stem leakage before training."
        )
    else:
        print("\n44.1k and 96k target labels are well aligned at zero lag.")

    summary = {
        "song_id": song_id,
        "target_instrument": target_instrument,
        "confuser_instruments": list(confuser_instruments),
        "threshold_dbfs": cfg.activity_threshold_dbfs,
        "label_hop_seconds": cfg.label_hop_seconds,
        "duration_44_seconds": duration_44,
        "duration_96_seconds": duration_96,
        "compared_duration_seconds": common_duration,
        "target_44_path": str(target_44_path),
        "target_96_path": str(target_96_path),
        "target_44_on_fraction": float(labels_44.mean()),
        "target_96_on_fraction": float(labels_96.mean()),
        "zero_lag": zero_lag.to_dict(),
        "best_rms_correlation": best_rms.to_dict(),
        "best_label_mcc": best_mcc.to_dict(),
        "confuser_overlap": overlap_rows,
    }
    save_json(output_dir / "diagnostic_summary.json", summary)
    print(f"\nDiagnostic outputs saved to: {output_dir}")
    return {
        "output_dir": output_dir,
        "summary": summary,
        "lag_scan": lag_scan,
        "alignment_candidates": comparison_table,
        "confuser_overlap": overlap_table,
        "time_series": time_series,
    }


if RUN_ALIGNMENT_DIAGNOSTIC:
    ALIGNMENT_DIAGNOSTIC_RESULTS = run_alignment_and_confuser_diagnostic(
        cfg=CFG,
        song_id=DIAGNOSTIC_SONG_ID,
        target_instrument=DIAGNOSTIC_TARGET_INSTRUMENT,
        confuser_instruments=DIAGNOSTIC_CONFUSER_INSTRUMENTS,
        max_lag_seconds=DIAGNOSTIC_MAX_LAG_SECONDS,
    )
else:
    ALIGNMENT_DIAGNOSTIC_RESULTS = None
    print("Alignment diagnostic is disabled.")


## 4. בדיקת מבנה אופציונלית

ברירת המחדל מדלגת על dry-run נפרד, משום שהאימון המלא כבר מבצע בדיקת batch
ו־forward לפני אפוק 1. לשם בדיקה בלבד ניתן להגדיר `RUN_DRY_RUN_BEFORE_TRAINING=True`.


In [ ]:
if RUN_DRY_RUN_BEFORE_TRAINING:
    dry_run_results = run_experiment(CFG)
else:
    dry_run_results = None
    print("Separate dry run skipped; the full run performs its own diagnostic batch.")


In [ ]:
# Accuracy, recall, inactive recall, F1, and balanced accuracy are computed
# directly by _binary_metrics for the natural song-5 validation set.
print("Natural-validation metrics are enabled.")


## אימון רב־כלי מלא — 50 אפוקים, CNN6 בלבד

אין LSTM ואין early stopping. תחילה CNN6 עובר קדם־אימון על חלונות stem של
0.5 שנייה שה־RMS הגולמי שלהם גבוה מ־−70 dBFS. לאחר מכן אותם משקלי CNN6
עוברים fine-tuning על מקטעי המיקס הפוליפוני: קלט של 8 שניות ופלט של 16
תחזיות מקומיות, אחת לכל חצי־שנייה.

המודל הגולמי ו־EMA נשמרים יחד לאחר כל אפוק לצורך התאוששות. הוולידציה
וה־checkpoint הטוב ביותר מבוססים על משקלי EMA, וההיסטוריה כוללת BAcc,
recall, inactive recall ו־F1 נפרדים לכל כלי.


In [ ]:
if RUN_FULL_TRAINING:
    CFG.dry_run = False
    results = run_experiment(CFG)
    print(results["run_dir"])
else:
    results = None
    print("Full training is disabled. Set RUN_FULL_TRAINING=True in the first configuration cell.")
